# Gradio Front-End for the Airbnb Price Serverless Endpoint

This notebook builds a **Gradio front-end UI** that invokes the SageMaker Serverless Endpoint
created in `03_production_pipeline.ipynb` — the *fair advertised nightly rate* regressor.

Default example configuration:

| Item | Value |
|---|---|
| Team | `team14` |
| Student/Profile | `s1402` |
| Region | `ap-southeast-1` |
| Endpoint name | `iti113-team14-airbnb-price-slsv2` |
| Task | Regression — continuous nightly `price`, in the listing's local currency |
| Contract | 27 raw listing fields (`SERVING_COLUMNS`), JSON |

This Gradio app does **not train a model**, and it does **not preprocess anything**. It collects
the 27 raw listing fields, formats them as a JSON object, sends them to the deployed SageMaker
Serverless Endpoint, and displays the prediction response.

Everything that turns a raw listing into features — distance from the city centre, amenity flags,
neighbourhood frequency, host tenure, scaling, one-hot encoding — happens **inside the endpoint**,
in the same fitted `Pipeline` object that training serialized. That is the train–serve consistency
guarantee proved in Notebook 03 Step 4, and this front-end is deliberately built so that it cannot
undermine it:

```text
Gradio UI  ──JSON──▶  inference.py  ──▶  ListingFeatureEngineer  ──▶  AirbnbPreprocessor
                                    ──▶  TransformedTargetRegressor(HistGradientBoosting)
                                    ──▶  suggested nightly price, local currency
```

## 1. Install Required Package

Run this once in SageMaker Studio. If Gradio is already installed, this cell will complete quickly.

In [1]:
# !pip install -q "starlette<1" "fastapi<1" --upgrade
# !pip install -q gradio

## 2. Configure Team, User, Region, and Check for Endpoint

The default endpoint is the Team14 regression endpoint:

```text
iti113-team14-airbnb-price-sls
```
The naming convention comes
straight from Notebook 03 Step 0:

```python
PRICE_ENDPOINT   = f"iti113-{TEAM_ID}-airbnb-price-sls"        # regression
BOOKING_ENDPOINT = f"iti113-{TEAM_ID}-airbnb-instant-booking"    # classification
ENDPOINTS = {"Airbnb price": PRICE_ENDPOINT, "Instant booking": BOOKING_ENDPOINT}
```

This code searches for serverless endpoints created by your own team.

In [2]:
import boto3
import json
import gradio as gr
from datetime import datetime

REGION = "ap-southeast-1"

# -------------------------------------------------------------------

TEAM_ID       = "team14"
STUDENT_ID    = "s1402"
COURSE        = "ITI113"
PROJECT_NAME  = "airbnb-listings"
PRICE_ENDPOINT   = f"iti113-{TEAM_ID}-airbnb-price-sls"        # regression
BOOKING_ENDPOINT = f"iti113-{TEAM_ID}-airbnb-instant-booking"    # classification
ENDPOINTS = {"Airbnb price": PRICE_ENDPOINT, "Instant booking": BOOKING_ENDPOINT}

# Where Notebook 01/03 published the fitted encoder state (used in Section 5)
BUCKET           = "nyp-26s1-iti113"
PREFIX           = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"
PROCESSED_PREFIX = f"{PREFIX}/processed"
ARTIFACTS_PREFIX = f"{PREFIX}/artifacts"

sts = boto3.client("sts", region_name=REGION)

print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("AWS identity:", sts.get_caller_identity()["Arn"])

sm      = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)
s3      = boto3.client("s3", region_name=REGION)

response = sm.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)

team_endpoints = []

for ep in response["Endpoints"]:
    endpoint_name = ep["EndpointName"]

    # Only show endpoints that follow the team naming convention
    if TEAM_ID in endpoint_name.lower():
        team_endpoints.append(ep)

print(f"\nEndpoints found for {TEAM_ID}: {len(team_endpoints)}\n")

for ep in team_endpoints:
    marker = ""
    for label, target in ENDPOINTS.items():
        if ep["EndpointName"] == target:
            marker = f"   <-- {label} tab"
    print("Endpoint name:", ep["EndpointName"], marker)
    print("Status:", ep["EndpointStatus"])
    print("Creation time:", ep["CreationTime"])
    print("Last modified:", ep["LastModifiedTime"])
    print("-" * 80)

found = {ep["EndpointName"] for ep in team_endpoints}
for label, target in ENDPOINTS.items():
    if target not in found:
        print(f"WARNING: [{label}] {target} is not in the list above.")
        print("         Deploy it from its Notebook 03, or correct the name.")

Region: ap-southeast-1
Team ID: team14
Student ID: s1402
AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team14/SageMaker



Endpoints found for team14: 2

Endpoint name: iti113-team14-airbnb-price-sls    <-- Airbnb price tab
Status: InService
Creation time: 2026-08-23 09:48:45.562000+00:00
Last modified: 2026-08-23 09:51:29.272000+00:00
--------------------------------------------------------------------------------
Endpoint name: iti113-team14-airbnb-instant-booking    <-- Instant booking tab
Status: InService
Creation time: 2026-08-21 17:05:48.831000+00:00
Last modified: 2026-08-21 17:08:27.436000+00:00
--------------------------------------------------------------------------------


## 3. Confirm That the Serverless Endpoint(s) Is Active

The endpoint must show `InService` before it can be invoked.

Note the loop variable below is `name`, not `ENDPOINT_NAME`. Reassigning `ENDPOINT_NAME` inside a
loop would leave it pointing at whichever endpoint happened to come last, and the Gradio UI further
down would then quietly talk to the wrong model.

In [3]:
try:
    for ep in team_endpoints:
        name = ep["EndpointName"]
        endpoint_desc = sm.describe_endpoint(EndpointName=name)
        print("\nEndpoint name:", endpoint_desc["EndpointName"])
        print("Status:", endpoint_desc["EndpointStatus"])
        print("Creation time:", endpoint_desc["CreationTime"])
        print("Last modified:", endpoint_desc["LastModifiedTime"])
except Exception as e:
    print("Unable to describe endpoint.")
    print("Check that ENDPOINT_NAME is correct and that your role has permission to access it.")
    print("Error:", e)

# The two this notebook will actually invoke
print("\n" + "=" * 80)
for label, target in ENDPOINTS.items():
    try:
        st = sm.describe_endpoint(EndpointName=target)["EndpointStatus"]
    except Exception:
        st = "NotFound"
    print(f"{label:<18s} {target:<42s} -> {st}")
    if st == "Creating":
        print("   Still provisioning. Deployment takes 3-5 minutes; re-run this cell.")
    elif st == "NotFound":
        print("   Not in this region. Deploy it from its Notebook 03 first.")
print("A tab whose endpoint is down degrades on its own; the other tab still works.")


Endpoint name: iti113-team14-airbnb-price-sls
Status: InService
Creation time: 2026-08-23 09:48:45.562000+00:00
Last modified: 2026-08-23 09:51:29.272000+00:00



Endpoint name: iti113-team14-airbnb-instant-booking
Status: InService
Creation time: 2026-08-21 17:05:48.831000+00:00
Last modified: 2026-08-21 17:08:27.436000+00:00

Airbnb price       iti113-team14-airbnb-price-sls             -> InService


Instant booking    iti113-team14-airbnb-instant-booking       -> InService
A tab whose endpoint is down degrades on its own; the other tab still works.


## 4. Inspect the Endpoint Model Artifact (Optional)

This cell shows which SageMaker model, endpoint configuration and model package are behind the
endpoint. This is how students connect the deployed endpoint back to the `model.tar.gz` produced by
Notebook 03 — the lineage check a reviewer performs before trusting a prediction.

In [4]:
for ep in team_endpoints:
    name = ep["EndpointName"]

    print("=" * 100)
    print("Endpoint name:", name)

    try:
        # 1. Describe endpoint
        endpoint_desc = sm.describe_endpoint(EndpointName=name)

        endpoint_config_name = endpoint_desc["EndpointConfigName"]

        print("Endpoint status:", endpoint_desc["EndpointStatus"])
        print("Endpoint config:", endpoint_config_name)

        # 2. Describe endpoint config
        endpoint_config = sm.describe_endpoint_config(
            EndpointConfigName=endpoint_config_name
        )

        production_variants = endpoint_config.get("ProductionVariants", [])

        if not production_variants:
            print("No production variants found.")
            continue

        # 3. Loop through each production variant
        for variant in production_variants:
            print("-" * 80)

            variant_name = variant.get("VariantName")
            model_name = variant.get("ModelName")

            print("Variant name:", variant_name)
            print("Model name:", model_name)

            # Check whether endpoint is serverless or real-time instance
            if "ServerlessConfig" in variant:
                serverless_config = variant["ServerlessConfig"]
                print("Endpoint type: Serverless")
                print("Memory size:", serverless_config.get("MemorySizeInMB"), "MB")
                print("Max concurrency:", serverless_config.get("MaxConcurrency"))
            else:
                print("Endpoint type: Real-time instance")
                print("Instance type:", variant.get("InstanceType"))
                print("Initial instance count:", variant.get("InitialInstanceCount"))

            if not model_name:
                print("No model name found for this variant.")
                continue

            # 4. Describe model
            model_desc = sm.describe_model(ModelName=model_name)

            # 5. Handle PrimaryContainer or Containers
            containers = model_desc.get("Containers")
            if containers is None and "PrimaryContainer" in model_desc:
                containers = [model_desc["PrimaryContainer"]]
                print("Container format: PrimaryContainer")
            else:
                print("Container format: Containers")

            for i, container in enumerate(containers or [], start=1):
                print(f"Container {i} image:", container.get("Image", "Not shown"))
                print(f"Container {i} model artifact S3 URI:", container.get("ModelDataUrl", "Not shown"))

                if "ModelPackageName" in container:
                    model_package_arn = container["ModelPackageName"]
                    print(f"Container {i} model package ARN:", model_package_arn)

                    try:
                        package_desc = sm.describe_model_package(
                            ModelPackageName=model_package_arn
                        )

                        inference_spec = package_desc.get("InferenceSpecification", {})
                        package_containers = inference_spec.get("Containers", [])

                        print("Model package approval status:",
                              package_desc.get("ModelApprovalStatus", "Not shown"))

                        for j, pkg_container in enumerate(package_containers, start=1):
                            print(f"Package container {j} image:", pkg_container.get("Image", "Not shown"))
                            print(f"Package container {j} model artifact S3 URI:",
                                  pkg_container.get("ModelDataUrl", "Not shown"))

                    except Exception as e:
                        print("Unable to inspect model package details.")
                        print("Error:", e)
    except Exception as e:
        print("Unable to inspect this endpoint's model details.")
        print("Error:", e)

print("=" * 100)
print("Endpoint inspection completed.")

Endpoint name: iti113-team14-airbnb-price-sls


Endpoint status: InService
Endpoint config: iti113-team14-airbnb-price-sls
--------------------------------------------------------------------------------
Variant name: AllTraffic
Model name: iti113-team14-airbnb-price-models-2026-08-23-09-48-44-125
Endpoint type: Serverless
Memory size: 3072 MB
Max concurrency: 5


Container format: Containers
Container 1 image: Not shown
Container 1 model artifact S3 URI: Not shown
Container 1 model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/iti113-team14-airbnb-price-models/18
Model package approval status: Approved
Package container 1 image: 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3
Package container 1 model artifact S3 URI: s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-08-23-09-36-05-368/pipelines-5lib4dgrbi1f-RegisterModel-Repack-pRp8VFqHyN/output/model.tar.gz
Endpoint name: iti113-team14-airbnb-instant-booking
Endpoint status: InService
Endpoint config: iti113-team14-airbnb-instant-booking
--------------------------------------------------------------------------------
Variant name: AllTraffic
Model name: team14-airbnb-instant-booking-2026-08-21-17-05-47-483
Endpoint type: Serverless
Memory size: 2048 MB
Max concurrency: 5


Container format: Containers
Container 1 image: Not shown
Container 1 model artifact S3 URI: Not shown
Container 1 model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team14-airbnb-instant-booking/29
Model package approval status: Approved
Package container 1 image: 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3
Package container 1 model artifact S3 URI: s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-08-21-16-52-59-754/pipelines-059euzj1rjtu-RegisterModel-Repack-EqaR9hFOqW/output/model.tar.gz
Endpoint inspection completed.


## Inspecting the Deployed Endpoint

The endpoint is connected to a SageMaker Model, and the model points to a Model Registry package
rather than directly exposing the container image and model artifact.

For this Team14 example:

- Endpoint: `iti113-team14-airbnb-price-sls`
- Endpoint type: Serverless — 3072 MB, `max_concurrency=5`, scale-to-zero
- Model package group: `iti113-team14-airbnb-price-models`
- Model package status: `Approved` (Notebook 03 Step 8, a deliberate human act)
- Container image: SageMaker Scikit-learn container
- Model artifact: `model.tar.gz` in S3, containing `model_pipeline.joblib`, `inference.py` and
  `pipeline_lib.py`

Alongside it in the MLflow registry:

- Registered model: `iti113-team14-airbnb-price-regressor` version 7, alias `champion`
- Data version: `sha256:097b0bbfea3c`
- Quality gate: `test_r2_log` 0.8708 ≥ 0.65 — **PASSED**

This confirms the endpoint is not merely created, but connected to an approved model package and a
saved model artifact whose lineage traces back to a specific dataset version and a gated run.

For the Gradio UI, these three values are needed:

- `REGION = "ap-southeast-1"`
- `TEAM_ID = "team14"`
- `ENDPOINT_NAME = "iti113-team14-airbnb-price-sls"`

### Check the endpoint invocation used previously from `03_production_pipeline.ipynb`

Notebook 03 Step 8.2 smoke-tests the same handler chain locally, then Step 8.3 invokes the live
endpoint with exactly the payload format used below.

## 5. Test Endpoint Invocation with boto3

Before building the Gradio UI, test the endpoint directly using `boto3`.

The endpoint expects input in **JSON format**, not CSV. Each input field is sent using its raw
feature name. A request is either one listing object or a list of them.

The deployed model bundle contains the trained pipeline *and* all preprocessing state, so the
endpoint accepts the **27 raw listing fields** of the `SERVING_COLUMNS` contract:

```text
city, latitude, longitude, neighbourhood, property_type, room_type,
accommodates, bedrooms, amenities, minimum_nights, maximum_nights,
instant_bookable, host_since, host_is_superhost, host_has_profile_pic,
host_identity_verified, host_total_listings_count, host_response_time,
host_response_rate, host_acceptance_rate,
review_scores_rating, review_scores_accuracy, review_scores_cleanliness,
review_scores_checkin, review_scores_communication, review_scores_location,
review_scores_value
```

The endpoint's `inference.py` performs all preprocessing internally:

```text
distance from city centre + coordinate coarsening
amenity parsing, amenity_count, rare_amenity_count
neighbourhood frequency lookup
host tenure, missing-value flags
property_type -> one of 7 governed property groups
numeric scaling, one-hot encoding, feature ordering (88 columns)
prediction, then expm1 back to price units
```

**Every one of the 27 keys must be present.** `input_fn` raises on a missing key. `null` is allowed
only for the fields the pipeline explicitly imputes:

| Field(s) | What `null` does |
|---|---|
| `bedrooms` | sets `bedrooms_missing`, fills from `ceil(accommodates / 2)` |
| `host_since` | `host_tenure_days` imputed to the fitted median |
| `host_response_rate`, `host_acceptance_rate` | sets the matching `_missing` flag, imputes the median |
| `host_response_time` | becomes the `"no history"` level |
| `neighbourhood` | becomes `"unknown"`, frequency `0.0` |
| the seven `review_scores_*` | sets `has_review_scores = 0`, imputes the medians |

Anything else sent as `null` produces a `ModelError`: the pipeline neither coerces nor imputes it,
and the endpoint fails its "no NaNs after preprocessing" assertion.

The response is a list, one record per listing:

```json
[{"suggested_nightly_price": 70.9, "currency": "EUR", "city": "Paris", "log_price": 4.2752}]
```

**Important:** the endpoint expects `ContentType="application/json"`. Using `text/csv` will fail,
because the deployed `inference.py` for this endpoint expects JSON input.

### 5.1 The serving contract, and self-configuring the UI

The cell below pulls the contract from `pipeline_lib` when `src/` is on the path — the same single
source of truth the endpoint imports — and mirrors it otherwise so this notebook still runs
standalone.

It also tries to read `preprocessing_artifacts.json`, the fitted encoder state that Notebook 03
Step 4 asserts the deployed pipeline against. When it is readable, the Gradio dropdowns, amenity
checkboxes and slider defaults are built from the **actual frozen state of the model** rather than
from assumptions. When it is not, the notebook falls back to sensible defaults and says so.

In [5]:
import os
import re
import sys
import math
import time
from datetime import date, datetime
from pathlib import Path

# --- The contract: pipeline_lib is authoritative ---------------------------------
for _p in ("src", ".", "../src"):
    if os.path.isdir(_p) and os.path.abspath(_p) not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))

try:
    from pipeline_lib import (CITY_CENTERS, CURRENCY, RESPONSE_TIME_LEVELS, REVIEW_COLS,
                              SERVING_COLUMNS, SNAPSHOT_DATE, property_group)
    from pipeline_lib import __version__ as PIPELINE_LIB_VERSION
    SNAPSHOT_DATE = date(SNAPSHOT_DATE.year, SNAPSHOT_DATE.month, SNAPSHOT_DATE.day)
    CONTRACT_SOURCE = "pipeline_lib (imported from src/)"
except Exception:
    PIPELINE_LIB_VERSION = "1.0.0 (mirrored)"
    CONTRACT_SOURCE = "mirrored constants in this notebook"
    SNAPSHOT_DATE = date(2021, 3, 1)

    CITY_CENTERS = {
        "Paris": (48.8530, 2.3499), "New York": (40.7580, -73.9855),
        "Sydney": (-33.8568, 151.2153), "Rome": (41.8986, 12.4769),
        "Rio de Janeiro": (-22.9711, -43.1822), "Istanbul": (41.0054, 28.9768),
        "Mexico City": (19.4326, -99.1332), "Bangkok": (13.7460, 100.5340),
        "Cape Town": (-33.9221, 18.4231), "Hong Kong": (22.2819, 114.1582),
    }
    CURRENCY = {"Paris": "EUR", "Rome": "EUR", "New York": "USD", "Sydney": "AUD",
                "Rio de Janeiro": "BRL", "Istanbul": "TRY", "Mexico City": "MXN",
                "Bangkok": "THB", "Cape Town": "ZAR", "Hong Kong": "HKD"}
    RESPONSE_TIME_LEVELS = ["within an hour", "within a few hours", "within a day",
                            "a few days or more", "no history"]
    REVIEW_COLS = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
                   "review_scores_checkin", "review_scores_communication",
                   "review_scores_location", "review_scores_value"]
    SERVING_COLUMNS = [
        "city", "latitude", "longitude", "neighbourhood", "property_type", "room_type",
        "accommodates", "bedrooms", "amenities", "minimum_nights", "maximum_nights",
        "instant_bookable", "host_since", "host_is_superhost", "host_has_profile_pic",
        "host_identity_verified", "host_total_listings_count", "host_response_time",
        "host_response_rate", "host_acceptance_rate",
    ] + REVIEW_COLS

    def property_group(pt):
        # Mirror of pipeline_lib.property_group -- display only.
        t = str(pt).lower()
        if any(k in t for k in ("hotel", "hostel", "resort")):                       return "hotel_hostel"
        if any(k in t for k in ("bed and breakfast", "guesthouse", "guest suite")):  return "bnb_guesthouse"
        if any(k in t for k in ("boat", "camper", "tent", "castle", "treehouse", "yurt",
                                "tiny house", "island", "cave", "dome", "windmill", "lighthouse",
                                "barn", "farm stay", "earth house", "hut", "tipi", "igloo",
                                "ryokan", "riad", "casa particular", "minsu", "pension",
                                "cycladic", "dammuso", "trullo", "shepherd")):       return "unique_stay"
        if any(k in t for k in ("apartment", "condominium", "loft", "serviced")):    return "apartment_condo"
        if any(k in t for k in ("house", "townhouse", "villa", "cottage", "bungalow",
                                "cabin", "chalet")):                                return "house_villa"
        if "private room" in t or "shared room" in t or "room in" in t:              return "room_other_dwelling"
        return "other"

CITIES = sorted(CITY_CENTERS)
R_EARTH_KM = 6371.0
MIN_NIGHTS_CAP, MAX_NIGHTS_CAP, BEDROOMS_CAP = 365, 1125, 16

# Fields that may legitimately be sent as null (see the table above)
NULLABLE = {"bedrooms", "host_since", "host_response_rate", "host_acceptance_rate",
            "host_response_time", "neighbourhood", *REVIEW_COLS}

# --- Fallbacks, used only when preprocessing_artifacts.json cannot be read --------
FALLBACK_ROOM_TYPES = ["Entire place", "Private room", "Shared room", "Hotel room"]
FALLBACK_PROPERTY_TYPES = [
    "Entire apartment", "Entire condominium", "Entire house", "Entire loft",
    "Entire serviced apartment", "Entire townhouse", "Entire villa",
    "Private room in apartment", "Private room in house", "Private room in condominium",
    "Private room in bed and breakfast", "Private room in guesthouse",
    "Private room in hostel", "Room in boutique hotel", "Room in hotel",
    "Shared room in apartment", "Shared room in hostel", "Boat", "Tiny house",
]
FALLBACK_AMENITIES = [
    "wifi", "kitchen", "air conditioning", "heating", "washer", "dryer", "tv",
    "essentials", "hangers", "iron", "shampoo", "hair dryer", "hot water",
    "refrigerator", "microwave", "coffee maker", "cooking basics", "oven", "stove",
    "dishes and silverware", "elevator", "free parking on premises", "pool", "gym",
    "smoke alarm", "carbon monoxide alarm", "fire extinguisher", "first aid kit",
    "long term stays allowed", "dedicated workspace",
]
FALLBACK_REVIEW_DEFAULTS = {
    "review_scores_rating": 95.0, "review_scores_accuracy": 10.0,
    "review_scores_cleanliness": 9.0, "review_scores_checkin": 10.0,
    "review_scores_communication": 10.0, "review_scores_location": 10.0,
    "review_scores_value": 9.0,
}
# Per-city test error from Notebook 03's evaluation_report.json, used to put an honest
# band around the point estimate. Replaced by the real report when it is readable.
FALLBACK_PER_CITY = {
    "Bangkok":        {"mape_pct": 37.4, "mae_ccy": 997.0, "n": 3863},
    "Cape Town":      {"mape_pct": 34.2, "mae_ccy": 957.3, "n": 3815},
    "Hong Kong":      {"mape_pct": 34.4, "mae_ccy": 324.2, "n": 1415},
    "Istanbul":       {"mape_pct": 39.0, "mae_ccy": 305.8, "n": 4893},
    "Mexico City":    {"mape_pct": 33.5, "mae_ccy": 427.3, "n": 4006},
    "New York":       {"mape_pct": 29.8, "mae_ccy":  51.9, "n": 7377},
    "Paris":          {"mape_pct": 27.6, "mae_ccy":  34.7, "n": 12916},
    "Rio de Janeiro": {"mape_pct": 44.6, "mae_ccy": 406.8, "n": 5321},
    "Rome":           {"mape_pct": 33.6, "mae_ccy":  47.6, "n": 5519},
    "Sydney":         {"mape_pct": 32.6, "mae_ccy":  88.4, "n": 6724},
}

# --- Read the published fitted state, if we can ----------------------------------
ARTIFACTS, ARTIFACTS_SOURCE, EVAL_REPORT = {}, "none - using built-in fallbacks", {}


def _load_json_local_or_s3(filename, prefixes):
    # Local folders first, then S3. Never raises.
    for folder in ("artifacts", "processed", "nb01_reference", ".", "../artifacts", "../processed"):
        p = Path(folder, filename)
        if p.is_file():
            try:
                return json.loads(p.read_text()), str(p)
            except Exception:
                pass
    for prefix in prefixes:
        key = f"{prefix}/{filename}"
        try:
            return json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()), f"s3://{BUCKET}/{key}"
        except Exception:
            continue
    return None, None


_arts, _src = _load_json_local_or_s3("preprocessing_artifacts.json",
                                     [PROCESSED_PREFIX, ARTIFACTS_PREFIX])
if _arts:
    ARTIFACTS, ARTIFACTS_SOURCE = _arts, _src

_report, _ = _load_json_local_or_s3("evaluation_report.json", [ARTIFACTS_PREFIX])
if _report:
    EVAL_REPORT = _report


def room_type_choices():
    return ARTIFACTS.get("onehot_levels", {}).get("room_type") or FALLBACK_ROOM_TYPES


def response_time_choices():
    return ARTIFACTS.get("onehot_levels", {}).get("host_response_time") or list(RESPONSE_TIME_LEVELS)


def amenity_choices():
    # The top-30 vocabulary the encoder actually one-hots.
    return ARTIFACTS.get("amenity_vocab") or FALLBACK_AMENITIES


def known_neighbourhoods(city):
    freq = ARTIFACTS.get("neighbourhood_freq") or {}
    return sorted({k.split("||", 1)[1] for k in freq if k.split("||", 1)[0] == city})


def neighbourhood_is_known(city, neighbourhood):
    # True / False, or None when we have no frequency table to check against.
    freq = ARTIFACTS.get("neighbourhood_freq")
    if not freq:
        return None
    return f"{city}||{neighbourhood or 'unknown'}" in freq


def review_defaults():
    # Slider defaults = the encoder's fitted imputation medians, when available.
    med = ARTIFACTS.get("imputation_values") or {}
    return {c: float(med.get(c, FALLBACK_REVIEW_DEFAULTS[c])) for c in REVIEW_COLS}


def review_slider_max(col, default_value):
    # review_scores_rating is 0-100 in this dataset; the six sub-scores are 0-10.
    # Infer from the fitted median so the UI follows the data, not an assumption.
    if col != "review_scores_rating":
        return 10.0
    return 100.0 if default_value > 10 else (10.0 if default_value > 5 else 5.0)


def per_city_error(city):
    for row in EVAL_REPORT.get("per_city", []) or []:
        if row.get("city") == city:
            return row
    return FALLBACK_PER_CITY.get(city)


print(f"Contract     : {len(SERVING_COLUMNS)} raw fields, from {CONTRACT_SOURCE}")
print(f"               pipeline_lib v{PIPELINE_LIB_VERSION}")
print(f"Fitted state : {ARTIFACTS_SOURCE}")
if ARTIFACTS:
    print(f"               {len(amenity_choices())} amenities in vocabulary")
    print(f"               {len(ARTIFACTS.get('neighbourhood_freq') or {})} (city, neighbourhood) pairs")
    print(f"               {len(ARTIFACTS.get('feature_columns') or [])} feature columns")
    print(f"               room types: {room_type_choices()}")
else:
    print("               Dropdowns will use built-in defaults, which may not match the")
    print("               frozen encoder levels. Run Notebook 03 first, or from the same")
    print("               directory, so artifacts/preprocessing_artifacts.json is readable.")

Contract     : 27 raw fields, from pipeline_lib (imported from src/)
               pipeline_lib v1.0.0
Fitted state : processed/preprocessing_artifacts.json
               30 amenities in vocabulary
               651 (city, neighbourhood) pairs
               88 feature columns
               room types: ['Entire place', 'Hotel room', 'Private room', 'Shared room']


### 5.2 Two example listings, sent straight to the endpoint

A Paris private room and a Bangkok condo — the same two markets Notebook 03 Step 8.2 used for its
local smoke test. The first call after an idle period is slow: the serverless endpoint scales to
zero, so it has to pull the container and deserialize the pipeline in `model_fn`. That shows up in
the `ModelSetupTime` CloudWatch metric and can take up to about a minute.

In [6]:
paris_listing = {
    "city": "Paris",
    "latitude": 48.8867,
    "longitude": 2.3431,
    "neighbourhood": "Buttes-Montmartre",
    "property_type": "Private room in apartment",
    "room_type": "Private room",
    "accommodates": 2,
    "bedrooms": 1,
    "amenities": json.dumps(["essentials", "hangers", "heating", "hot water", "iron",
                             "kitchen", "long term stays allowed", "shampoo", "washer", "wifi"]),
    "minimum_nights": 2,
    "maximum_nights": 90,
    "instant_bookable": "f",
    "host_since": "2015-06-12",
    "host_is_superhost": "f",
    "host_has_profile_pic": "t",
    "host_identity_verified": "t",
    "host_total_listings_count": 1,
    "host_response_time": "within a few hours",
    "host_response_rate": 90,
    "host_acceptance_rate": 80,
    "review_scores_rating": 95,
    "review_scores_accuracy": 10,
    "review_scores_cleanliness": 9,
    "review_scores_checkin": 10,
    "review_scores_communication": 10,
    "review_scores_location": 10,
    "review_scores_value": 9,
}

bangkok_listing = dict(paris_listing, **{
    "city": "Bangkok",
    "latitude": 13.7230,
    "longitude": 100.5680,
    "neighbourhood": "Khlong Toei",
    "property_type": "Entire condominium",
    "room_type": "Entire place",
    "accommodates": 4,
    "bedrooms": 2,
    "amenities": json.dumps(["air conditioning", "dryer", "elevator", "essentials", "gym",
                             "kitchen", "pool", "tv", "washer", "wifi"]),
    "minimum_nights": 1,
    "maximum_nights": 365,
    "instant_bookable": "t",
    "host_since": "2017-03-08",
    "host_is_superhost": "t",
    "host_total_listings_count": 6,
    "host_response_time": "within an hour",
    "host_response_rate": 100,
    "host_acceptance_rate": 95,
})


def invoke_airbnb_endpoint(payload, endpoint_name=PRICE_ENDPOINT):
    # POST the raw listing contract. Returns (records, latency_ms).
    t0 = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload)
    )
    records = json.loads(response["Body"].read())
    return records, (time.perf_counter() - t0) * 1000.0


# Contract check before we spend a round trip on it
for name, listing in [("paris_listing", paris_listing), ("bangkok_listing", bangkok_listing)]:
    missing = [c for c in SERVING_COLUMNS if c not in listing]
    assert not missing, f"{name} is missing required fields: {missing}"
print(f"Both example listings carry all {len(SERVING_COLUMNS)} contract fields.\n")

try:
    # One request, both listings -- input_fn accepts a list as well as a single object
    records, ms = invoke_airbnb_endpoint([paris_listing, bangkok_listing])

    for rec in records:
        print(f"{rec['city']:<10s} suggested fair rate {rec['suggested_nightly_price']:>10,.2f} "
              f"{rec['currency']}   (log_price {rec['log_price']})")
    print(f"\nRound trip: {ms:,.0f} ms")

except Exception as e:
    print("Endpoint invocation failed.")
    print("Error:", e)

Both example listings carry all 27 contract fields.



Paris      suggested fair rate      47.86 EUR   (log_price 3.889)
Bangkok    suggested fair rate   1,671.61 THB   (log_price 7.4221)

Round trip: 6,630 ms


### 5.3 The second model: instant-booking classifier

| | Airbnb price | Instant booking |
|---|---|---|
| Endpoint | `iti113-team14-airbnb-price-sls` | `iti113-team14-airbnb-instant-booking` |
| Task | regression - nightly rate | binary classification - will the host allow instant booking |
| Target | `price` | `instant_bookable` |
| Memory | 3072 MB | 2048 MB |
| Response | `suggested_nightly_price`, `currency`, `city`, `log_price` | `prediction`, `probability_class_1`, `class_1`, `class_0` |

> **Two hazards worth knowing before you trust this tab.**
>
> **1. The rate scale is not enforced anywhere.** The classifier's `preprocess.py` converts
> `host_response_rate` and `host_acceptance_rate` from 0-100 to 0-1 at training time
> (`mask = df[col] > 1` then `/ 100.0`). Its `inference.py` does not mention those columns at all.
> So the conversion exists only on the training side, and **the caller must send 0-1**. Send 95 and
> the model sees a value 100x larger than anything in training. Nothing errors.
>
> **2. Missing fields become zero, silently.** `create_features` ends with
> `df.reindex(columns=final_feature_columns, fill_value=0)`. Only four fields are hard-required -
> `city`, `amenities`, `host_total_listings_count`, `price`. Everything else you omit is filled with
> 0 and the endpoint answers anyway. This is the opposite of the price model, whose `input_fn` raises
> on any missing key. It means a typo in a field name here costs you accuracy instead of an error,
> so the UI below always sends the full 26-field set.

In [7]:
# ---- the classifier's contract, kept separate from the price model's ------------
BOOKING_FIELDS = [
    "city", "amenities", "host_total_listings_count", "price",
    "host_since", "accommodates", "bedrooms", "property_type", "room_type",
    "host_response_time", "host_response_rate", "host_acceptance_rate",
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "minimum_nights", "maximum_nights",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location",
    "review_scores_value", "latitude", "longitude",
]

# Enforced by create_features(). Everything else silently zero-fills.
BOOKING_HARD_REQUIRED = ["city", "amenities", "host_total_listings_count", "price"]

BOOKING_ROOM_TYPES = ["Entire home/apt", "Private room", "Shared room", "Hotel room"]
BOOKING_SNAPSHOT = date(2026, 8, 16)


def invoke_booking_endpoint(payload, endpoint_name=BOOKING_ENDPOINT):
    # Separate function, separate default endpoint. Do not fold this into the
    # price model's invoke helper -- one shared ENDPOINT_NAME is how a tab ends up
    # posting its payload to the wrong model.
    t0 = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(payload)
    )
    records = json.loads(response["Body"].read())
    return records, (time.perf_counter() - t0) * 1000.0


# The payload proven against the live endpoint in the classifier's Notebook 03.
booking_test_listing = {
    "city": "Paris",
    "amenities": '["Wifi", "Air conditioning", "Kitchen", "Pool", "Gym", "TV"]',
    "host_total_listings_count": 2,
    "price": 150.0,
    "host_since": "2018-05-15",
    "accommodates": 4,
    "bedrooms": 2,
    "property_type": "Entire apartment",
    "room_type": "Entire home/apt",
    "host_response_time": "within an hour",
    "host_response_rate": 1.0,          # 0-1, NOT 0-100
    "host_acceptance_rate": 0.95,       # 0-1, NOT 0-100
    "host_is_superhost": 1,
    "host_has_profile_pic": 1,
    "host_identity_verified": 1,
    "minimum_nights": 2,
    "maximum_nights": 1125,
    "review_scores_rating": 95,
    "review_scores_accuracy": 10,
    "review_scores_cleanliness": 9,
    "review_scores_checkin": 10,
    "review_scores_communication": 10,
    "review_scores_location": 10,
    "review_scores_value": 9,
    "latitude": 48.8867,
    "longitude": 2.3431,
}

missing = [f for f in BOOKING_FIELDS if f not in booking_test_listing]
assert not missing, f"payload is missing {missing}"
print(f"Booking payload carries all {len(BOOKING_FIELDS)} fields.\n")

try:
    records, ms = invoke_booking_endpoint(booking_test_listing)
    rec = records[0]
    label = "Instant bookable" if rec["prediction"] == 1 else "Not instant bookable"
    prob = rec.get("probability_class_1")
    print(f"Prediction : {rec['prediction']}  ({label})")
    print(f"P(class 1) : {prob:.1%}" if prob is not None else "P(class 1) : not returned")
    print(f"Round trip : {ms:,.0f} ms")
except Exception as e:
    print("Endpoint invocation failed.")
    print("Error:", e)

Booking payload carries all 26 fields.



Prediction : 0  (Not instant bookable)
P(class 1) : 43.9%
Round trip : 10,150 ms


> **On the review-score scale.** The classifier's `preprocess.py` never rescales the seven review
> columns, so the model saw whatever the raw CSV holds - the same 0-100 rating and 0-10 sub-scores
> the price model uses. The smoke-test payload in the original notebook sent `4.8` and `4.9`, which
> is roughly 20x too small for the rating. That payload returned `probability_class_1 = 0.567`, a
> near coin-flip, which is what you would expect from a mostly out-of-distribution request. The cell
> above sends 0-100 / 0-10 instead. Confirm against `train_imputations["review_scores_rating"]` in
> the classifier's own `preprocessing_artifacts.json`: if that median is around 95, this cell is
> right; if it is around 4.8, switch the sliders back.

## 6. Build the Gradio Prediction Function

This function will be connected to the Gradio interface. It collects values from UI fields, maps
them onto the 27-field contract, sends them to the endpoint, and returns the endpoint response.

The UI has more controls than the contract has fields of 33 versus 27 because some contract fields
need two controls. "Bedrooms not stated", "Host since: not known" and "This listing has reviews"
are toggles that decide whether a field is sent as a **value or as `null`**, which is how a student
exercises the missing-value flags inside the pipeline.

Rather than a 33-parameter function signature, the handler takes `*values` and zips them against
`UI_FIELDS`. The order of `UI_FIELDS` is the order of the `inputs=` list in Section 7, and it is the
only place the two have to agree.

In [8]:
import boto3
import json

REGION = "ap-southeast-1"

# change to your own team including ENDPOINT
TEAM_ID       = "team14"
STUDENT_ID    = "s1402"
PRICE_ENDPOINT   = "iti113-team14-airbnb-price-sls"        # regression
BOOKING_ENDPOINT = "iti113-team14-airbnb-instant-booking"    # classification

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

### 6.1 UI state → the serving contract

`build_payload` is the only place the contract is written down. The UI can be rearranged freely
without touching it, and `validate` catches the inputs the pipeline cannot recover from *before*
spending a round trip on a `ModelError`.

In [9]:
# The order here is the order of the inputs= list in Section 7.
UI_FIELDS = [
    "city", "neighbourhood", "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bedrooms", "bedrooms_unknown",
    "amenities_selected", "amenities_extra",
    "minimum_nights", "maximum_nights", "instant_bookable",
    "host_since", "host_since_unknown", "host_is_superhost", "host_has_profile_pic",
    "host_identity_verified", "host_total_listings_count", "host_response_time",
    "host_response_rate", "host_response_rate_known",
    "host_acceptance_rate", "host_acceptance_rate_known",
    "has_reviews", *REVIEW_COLS,
]


def _tf(flag):
    # The raw platform encoding for booleans: 't' / 'f', exactly as trained.
    return "t" if bool(flag) else "f"


def _amenities(selected, extra):
    items = list(selected or [])
    for token in re.split(r"[,;\n]", extra or ""):
        if token.strip():
            items.append(token.strip())
    return sorted({a.strip().lower() for a in items if a.strip()})


def build_payload(ui):
    # UI state -> one JSON object on the 27-field SERVING_COLUMNS contract.
    has_reviews = bool(ui.get("has_reviews", True))

    payload = {
        "city": ui["city"],
        "latitude": float(ui["latitude"]),
        "longitude": float(ui["longitude"]),
        "neighbourhood": (ui.get("neighbourhood") or "").strip() or None,
        "property_type": (ui.get("property_type") or "").strip() or "Other",
        "room_type": ui["room_type"],
        "accommodates": int(ui["accommodates"]),
        "bedrooms": None if ui.get("bedrooms_unknown") else int(ui["bedrooms"]),
        "amenities": json.dumps(_amenities(ui.get("amenities_selected"), ui.get("amenities_extra"))),
        "minimum_nights": int(ui["minimum_nights"]),
        "maximum_nights": int(ui["maximum_nights"]),
        "instant_bookable": _tf(ui.get("instant_bookable")),
        "host_since": None if ui.get("host_since_unknown") else (ui.get("host_since") or "").strip() or None,
        "host_is_superhost": _tf(ui.get("host_is_superhost")),
        "host_has_profile_pic": _tf(ui.get("host_has_profile_pic")),
        "host_identity_verified": _tf(ui.get("host_identity_verified")),
        "host_total_listings_count": int(ui.get("host_total_listings_count") or 0),
        "host_response_time": ui.get("host_response_time") or None,
        "host_response_rate": float(ui["host_response_rate"]) if ui.get("host_response_rate_known") else None,
        "host_acceptance_rate": float(ui["host_acceptance_rate"]) if ui.get("host_acceptance_rate_known") else None,
    }
    for col in REVIEW_COLS:
        payload[col] = float(ui[col]) if has_reviews else None

    # Order the keys as the contract declares them: cosmetic for the JSON view,
    # and it makes a missing field obvious at a glance.
    return {k: payload[k] for k in SERVING_COLUMNS}


def validate(payload):
    # Fail fast, in the UI, on the inputs the pipeline cannot recover from.
    problems = []
    if payload["city"] not in CITY_CENTERS:
        problems.append("City must be one of the 10 markets the model was trained on: "
                        + ", ".join(CITIES) + ".")
    for field in ("latitude", "longitude", "accommodates", "minimum_nights", "maximum_nights"):
        v = payload.get(field)
        if v is None or (isinstance(v, float) and math.isnan(v)):
            problems.append(f"{field} is required - the pipeline does not impute it.")

    # 0 is a value, not an absence: test against None, never truthiness.
    if payload["accommodates"] is not None and payload["accommodates"] < 1:
        problems.append("Accommodates must be at least 1; a listing for 0 guests is not rentable.")
    if payload["minimum_nights"] is not None and payload["minimum_nights"] < 1:
        problems.append("Minimum nights must be at least 1.")
    if (payload["minimum_nights"] is not None and payload["maximum_nights"] is not None
            and payload["minimum_nights"] > payload["maximum_nights"]):
        problems.append("Minimum nights is greater than maximum nights.")
    if payload["bedrooms"] is not None and payload["bedrooms"] < 0:
        problems.append("Bedrooms cannot be negative. Tick 'Bedrooms not stated' to send null.")

    for key in SERVING_COLUMNS:
        if key not in payload:
            problems.append(f"Missing contract field: {key}.")
        elif payload[key] is None and key not in NULLABLE:
            problems.append(f"{key} cannot be null on this contract.")
    return problems

### 6.2 Showing the invisible half

The 27 fields a student types are not the 88 columns the model scores. `derived_preview` mirrors
`pipeline_lib`'s stateless engineering step so the UI can display the features that never appear in
the payload — including whether the `(city, neighbourhood)` pair is one the encoder has ever seen,
which is exactly the unseen-neighbourhood signal the weekly drift job counts.

This mirror is **display only**. The endpoint recomputes all of it authoritatively, and the preview
is wrapped so a bug in it can never take down a real prediction.

In [10]:
def haversine_km(lat1, lon1, lat2, lon2):
    p = math.pi / 180.0
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 2 * R_EARTH_KM * math.asin(math.sqrt(a))


def derived_preview(payload):
    # Display only. Wrapped so it can never break a response.
    try:
        return _derived_preview(payload)
    except Exception as e:
        return f"Preview unavailable ({e.__class__.__name__}). The prediction above is unaffected."


def _derived_preview(payload):
    city = payload["city"]
    lines = []

    if city in CITY_CENTERS:
        clat, clon = CITY_CENTERS[city]
        dist = haversine_km(payload["latitude"], payload["longitude"], clat, clon)
        lines.append(f"| `dist_center_km` | {dist:,.3f} km from the {city} centre |")
        lines.append(f"| `lat_coarse` / `lon_coarse` | {round(payload['latitude'], 2)}, "
                     f"{round(payload['longitude'], 2)} - exact GPS is destroyed inside the "
                     f"pipeline and never reaches an artifact |")

    amen = json.loads(payload["amenities"])
    in_vocab = len([a for a in amen if a in set(amenity_choices())])
    lines.append(f"| `amenity_count` | {len(amen)} |")
    lines.append(f"| `rare_amenity_count` | {len(amen) - in_vocab} outside the top-30 vocabulary |")

    known = neighbourhood_is_known(city, payload["neighbourhood"])
    if known is True:
        lines.append(f"| `neighbourhood_freq` | ({city}, {payload['neighbourhood']}) is in the "
                     f"fitted frequency table |")
    elif known is False:
        lines.append(f"| `neighbourhood_freq` | **0.0 - unseen (city, neighbourhood) pair.** "
                     f"This is exactly what the weekly drift job counts |")
    else:
        lines.append("| `neighbourhood_freq` | unknown here - publish "
                     "`preprocessing_artifacts.json` to check the pair |")

    if payload["host_since"]:
        try:
            hs = datetime.strptime(payload["host_since"], "%Y-%m-%d").date()
            tenure = (SNAPSHOT_DATE - hs).days
            note = "" if tenure >= 0 else " **negative - after the snapshot date, out of distribution**"
            lines.append(f"| `host_tenure_days` | {tenure:,} days as of the "
                         f"{SNAPSHOT_DATE:%Y-%m-%d} snapshot{note} |")
        except ValueError:
            lines.append("| `host_tenure_days` | host_since is not YYYY-MM-DD; it will parse to "
                         "NaT and be imputed |")
    else:
        lines.append("| `host_tenure_days` | imputed to the fitted median (host_since is null) |")

    if payload["bedrooms"] is None:
        lines.append("| `bedrooms_missing` | 1 - bedrooms filled as ceil(accommodates / 2) |")
    else:
        ppb = payload["accommodates"] / max(payload["bedrooms"], 1)
        lines.append(f"| `persons_per_bedroom` | {ppb:.3f} |")

    lines.append(f"| `is_monthly_min` | {int(payload['minimum_nights'] >= 28)} - "
                 f"minimum_nights is {payload['minimum_nights']}, the threshold is 28 |")
    lines.append(f"| `property_group` | `{property_group(payload['property_type'])}` "
                 f"- one of 7 governed buckets |")
    lines.append(f"| `has_review_scores` | {int(payload['review_scores_rating'] is not None)} |")
    missing_host = [f"`{k}`" for k in ("host_response_rate", "host_acceptance_rate")
                    if payload[k] is None]
    if missing_host:
        lines.append(f"| missing-value flags | {', '.join(missing_host)} null -> flag set, "
                     f"value imputed |")

    return ("Derived inside the endpoint, from the fields above:\n\n"
            "| feature | value |\n|---|---|\n" + "\n".join(lines))


def explain_error(exc):
    # Say what went wrong and what to do about it.
    name, msg = exc.__class__.__name__, str(exc)
    hint = ""
    if "ValidationException" in name + msg and "endpoint" in msg.lower():
        hint = (f"No endpoint by that name in `{REGION}`. Re-run Section 2 and correct "
                f"PRICE_ENDPOINT / BOOKING_ENDPOINT.")
    elif "AccessDenied" in name + msg:
        hint = ("Your execution role cannot invoke this endpoint. Check that you are in your own "
                "Studio profile and that the endpoint belongs to your team.")
    elif "ModelError" in name + msg:
        hint = ("The endpoint received the request but `inference.py` rejected it - usually a "
                "contract mismatch. Compare the request JSON below against the 27 "
                "`SERVING_COLUMNS` fields.")
    elif "Throttling" in name + msg or "TooManyRequests" in msg:
        hint = "The serverless endpoint is at `max_concurrency=5`. Wait a moment and send again."
    elif "ExpiredToken" in msg or "InvalidClientTokenId" in msg or "NoCredentials" in name:
        hint = "AWS credentials are missing or expired. Restart the kernel or refresh your session."
    elif "ReadTimeout" in name or "timed out" in msg.lower():
        hint = ("This looks like a serverless cold start: the container pulls the image and "
                "deserializes the pipeline in `model_fn`, which can take up to ~60 s on the first "
                "call. Send the request again.")
    return f"### Request failed\n\n`{name}`\n\n```\n{msg}\n```\n\n{hint}".rstrip()

### 6.3 The Gradio handler

Returns four things: the result card, the derived-features table, the exact JSON that was posted,
and the raw endpoint response. Showing the request back to the student is the point — it is the
contract, and being able to read it is what makes a `ModelError` diagnosable.

The result carries a **±band from the per-city test MAPE**, not just a point estimate. A regression
model that answers "70.90 EUR" to three significant figures invites more confidence than 0.87 R²
on log-price deserves; Paris is ±27.6% and Rio is ±44.6%, and the UI says so.

In [11]:
def predict_airbnb_price(*values):
    # Gradio click handler. Returns (result card, derived table, request JSON, response JSON).
    ui = dict(zip(UI_FIELDS, values))

    try:
        payload = build_payload(ui)
    except (TypeError, ValueError) as e:
        return (f"### Cannot build the request\n\n{e}\n\nCheck the numeric fields.", "", "", "")

    request_json = json.dumps(payload, indent=2)

    problems = validate(payload)
    if problems:
        bullets = "\n".join(f"- {p}" for p in problems)
        return (f"### Fix these before sending\n\n{bullets}", "", request_json, "")

    try:
        records, ms = invoke_airbnb_endpoint(payload)
    except Exception as e:
        return (explain_error(e), derived_preview(payload), request_json, "")

    rec = records[0]
    price, ccy = rec["suggested_nightly_price"], rec["currency"]

    band = ""
    err = per_city_error(rec["city"])
    if err:
        mape = float(err["mape_pct"])
        lo, hi = price * (1 - mape / 100), price * (1 + mape / 100)
        band = (f"<div class='band'>Typical error for {rec['city']} is "
                f"<strong>&plusmn;{mape:.1f}%</strong> on {int(err.get('n', 0)):,} held-out test "
                f"listings, so read this as roughly <strong>{lo:,.0f}&ndash;{hi:,.0f} {ccy}</strong>, "
                f"not a precise figure.</div>")

    card = (
        "<div class='result'>"
        f"<div class='eyebrow'>Suggested fair advertised nightly rate &middot; {rec['city']}</div>"
        f"<div class='price'>{price:,.2f} <span class='ccy'>{ccy}</span></div>"
        f"{band}"
        f"<div class='meta'>log_price {rec['log_price']} &middot; {ms:,.0f} ms round trip "
        f"&middot; {PRICE_ENDPOINT}</div>"
        "</div>"
    )
    return card, derived_preview(payload), request_json, json.dumps(records, indent=2)


# --- Example listings for the UI buttons (see Section 8) --------------------------
def _base_preset(**overrides):
    d = review_defaults()
    preset = {
        "city": "Paris", "neighbourhood": "Buttes-Montmartre",
        "latitude": 48.8867, "longitude": 2.3431,
        "property_type": "Private room in apartment", "room_type": "Private room",
        "accommodates": 2, "bedrooms": 1, "bedrooms_unknown": False,
        "amenities_selected": [], "amenities_extra": "",
        "minimum_nights": 2, "maximum_nights": 90, "instant_bookable": False,
        "host_since": "2015-06-12", "host_since_unknown": False,
        "host_is_superhost": False, "host_has_profile_pic": True,
        "host_identity_verified": True, "host_total_listings_count": 1,
        "host_response_time": "within a few hours",
        "host_response_rate": 90.0, "host_response_rate_known": True,
        "host_acceptance_rate": 80.0, "host_acceptance_rate_known": True,
        "has_reviews": True, **{c: d[c] for c in REVIEW_COLS},
    }
    preset.update(overrides)
    return preset


def _amen(*names):
    # Split a listing's amenities across the two UI controls. Anything in the fitted
    # vocabulary becomes a ticked checkbox; the rest goes to the free-text box, where it
    # still raises amenity_count and rare_amenity_count. A real listing does not lose an
    # amenity just because the encoder has no flag for it.
    vocab = set(amenity_choices())
    return {"amenities_selected": [n for n in names if n in vocab],
            "amenities_extra": ", ".join(n for n in names if n not in vocab)}


PRESETS = {
    "Paris studio": _base_preset(
        **_amen("wifi", "heating", "kitchen", "essentials", "hangers", "iron", "shampoo",
                "washer", "hot water", "long term stays allowed"),
    ),
    "Bangkok condo": _base_preset(
        city="Bangkok", neighbourhood="Khlong Toei", latitude=13.7230, longitude=100.5680,
        property_type="Entire condominium", room_type="Entire place",
        accommodates=4, bedrooms=2,
        **_amen("wifi", "air conditioning", "pool", "gym", "kitchen", "washer", "dryer", "tv",
                "elevator", "free parking on premises", "essentials", "hot water",
                "dedicated workspace", "refrigerator", "microwave"),
        minimum_nights=1, maximum_nights=365, instant_bookable=True,
        host_since="2017-03-08", host_is_superhost=True, host_total_listings_count=6,
        host_response_time="within an hour", host_response_rate=100.0, host_acceptance_rate=95.0,
    ),
    "New York, brand new listing": _base_preset(
        city="New York", neighbourhood="Midtown", latitude=40.7549, longitude=-73.9840,
        property_type="Entire apartment", room_type="Entire place",
        accommodates=4, bedrooms=1,
        bedrooms_unknown=True,              # -> bedrooms_missing
        **_amen("wifi", "kitchen", "air conditioning", "heating", "elevator", "essentials", "tv"),
        minimum_nights=30,                  # -> is_monthly_min
        maximum_nights=365, instant_bookable=True,
        host_since="", host_since_unknown=True,   # -> host_tenure_days imputed
        host_has_profile_pic=True, host_identity_verified=False, host_total_listings_count=1,
        host_response_time="no history",
        host_response_rate_known=False,     # -> host_response_rate_missing
        host_acceptance_rate_known=False,
        has_reviews=False,                  # -> has_review_scores = 0
    ),
}

# Every preset must cover every UI control, or the loader buttons would leave stale values behind.
for _name, _p in PRESETS.items():
    assert set(_p) == set(UI_FIELDS), f"{_name}: {set(UI_FIELDS) ^ set(_p)}"

print(f"Prediction function ready. {len(UI_FIELDS)} UI controls -> "
      f"{len(SERVING_COLUMNS)} contract fields.")
print(f"Presets: {list(PRESETS)}")

Prediction function ready. 33 UI controls -> 27 contract fields.
Presets: ['Paris studio', 'Bangkok condo', 'New York, brand new listing']


### 6.4 The classifier's handler

Same shape as the price handler, different everything else: its own field list, its own payload
builder, its own renderer, and - critically - its own endpoint, bound as a default argument rather
than read from a shared global.

Classification needs a different honesty than regression. There is no MAPE band; what matters is the
probability and how close it sits to the 0.5 decision boundary. A prediction at 0.51 and one at 0.97
are the same label and completely different claims, so the card shows the probability bar and says
which side of the boundary it is on.

In [12]:
BOOKING_UI_FIELDS = [
    "city", "latitude", "longitude", "property_type", "room_type",
    "accommodates", "bedrooms", "price",
    "amenities_selected", "amenities_extra",
    "minimum_nights", "maximum_nights",
    "host_since", "host_total_listings_count", "host_response_time",
    "host_response_rate", "host_acceptance_rate",
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    *REVIEW_COLS,
]


def build_booking_payload(ui):
    # UI state -> the classifier's 26 fields. Note the unit conversions: the sliders
    # are 0-100 because that is what a person reads, but this endpoint needs 0-1.
    payload = {
        "city": ui["city"],
        "latitude": float(ui["latitude"]),
        "longitude": float(ui["longitude"]),
        "property_type": (ui.get("property_type") or "").strip() or "Other",
        "room_type": ui["room_type"],
        "accommodates": int(ui["accommodates"]),
        "bedrooms": int(ui["bedrooms"]),
        "price": float(ui["price"]),
        "amenities": json.dumps(_amenities(ui.get("amenities_selected"), ui.get("amenities_extra"))),
        "minimum_nights": int(ui["minimum_nights"]),
        "maximum_nights": int(ui["maximum_nights"]),
        "host_since": (ui.get("host_since") or "").strip() or None,
        "host_total_listings_count": int(ui.get("host_total_listings_count") or 0),
        "host_response_time": ui.get("host_response_time") or "no history",
        "host_response_rate": float(ui["host_response_rate"]) / 100.0,      # -> 0-1
        "host_acceptance_rate": float(ui["host_acceptance_rate"]) / 100.0,  # -> 0-1
        "host_is_superhost": int(bool(ui.get("host_is_superhost"))),
        "host_has_profile_pic": int(bool(ui.get("host_has_profile_pic"))),
        "host_identity_verified": int(bool(ui.get("host_identity_verified"))),
    }
    for col in REVIEW_COLS:
        payload[col] = float(ui[col])

    return {k: payload[k] for k in BOOKING_FIELDS}


def validate_booking(payload):
    problems = []
    for f in BOOKING_HARD_REQUIRED:
        if payload.get(f) is None:
            problems.append(f"{f} is hard-required by create_features(); the endpoint will raise.")
    if payload["price"] is not None and payload["price"] <= 0:
        problems.append("Price must be positive - it is log-transformed into a within-city z-score.")
    if payload["minimum_nights"] > payload["maximum_nights"]:
        problems.append("Minimum nights is greater than maximum nights.")
    # Silent zero-fill is the real hazard here, so check for it explicitly.
    zero_filled = [f for f in BOOKING_FIELDS if f not in payload]
    if zero_filled:
        problems.append(f"These would be silently zero-filled by the endpoint: {zero_filled}")
    for f in ("host_response_rate", "host_acceptance_rate"):
        if payload[f] is not None and payload[f] > 1.0:
            problems.append(f"{f} is {payload[f]}, but this model was trained on 0-1.")
    return problems


def booking_derived_preview(payload):
    # Display only, mirroring create_features().
    try:
        amen = json.loads(payload["amenities"])
        lines = [f"| `amenity_count` | {len(amen)} |"]
        lines.append(f"| `amenity_*` flags | matched against an 80-term vocabulary |")
        lines.append(f"| `log_price_city_zscore` | log1p({payload['price']:,.0f}) standardised "
                     f"within {payload['city']}; an unseen city gives exactly 0.0 |")
        if payload["host_since"]:
            try:
                hs = datetime.strptime(payload["host_since"], "%Y-%m-%d").date()
                lines.append(f"| `host_tenure_days` | {(BOOKING_SNAPSHOT - hs).days:,} days as of "
                             f"the {BOOKING_SNAPSHOT:%Y-%m-%d} snapshot |")
            except ValueError:
                lines.append("| `host_tenure_days` | host_since is not YYYY-MM-DD |")
        if payload["bedrooms"]:
            lines.append(f"| `accommodates_per_bedroom` | "
                         f"{payload['accommodates'] / payload['bedrooms']:.2f} |")
        lines.append(f"| `property_type_grouped` | `{payload['property_type']}` if it is in the "
                     f"top-K list, else `Other` |")
        lines.append(f"| rate scale | sent as {payload['host_response_rate']:.2f} / "
                     f"{payload['host_acceptance_rate']:.2f}, matching the 0-1 training scale |")
        lines.append("| anything not sent | **filled with 0, no error** |")
        return ("Derived inside the endpoint:\n\n| feature | value |\n|---|---|\n" + "\n".join(lines))
    except Exception as e:
        return f"Preview unavailable ({e.__class__.__name__})."


def predict_instant_booking(*values):
    ui = dict(zip(BOOKING_UI_FIELDS, values))
    try:
        payload = build_booking_payload(ui)
    except (TypeError, ValueError) as e:
        return (f"### Cannot build the request\n\n{e}", "", "", "")

    request_json = json.dumps(payload, indent=2)
    problems = validate_booking(payload)
    if problems:
        return ("### Fix these before sending\n\n" + "\n".join(f"- {p}" for p in problems),
                "", request_json, "")

    try:
        records, ms = invoke_booking_endpoint(payload)
    except Exception as e:
        return (explain_error(e), booking_derived_preview(payload), request_json, "")

    rec = records[0]
    pred = rec["prediction"]
    prob = rec.get("probability_class_1")
    label = "Instant bookable" if pred == 1 else "Not instant bookable"

    if prob is None:
        detail = "<div class='band'>The endpoint returned no probability for this request.</div>"
    else:
        margin = abs(prob - 0.5)
        confidence = ("barely past the 0.5 boundary - treat this label as a coin flip"
                      if margin < 0.10 else
                      "clear of the boundary" if margin > 0.25 else
                      "moderately clear of the boundary")
        detail = (
            f"<div class='band'>P(instant bookable) = <strong>{prob:.1%}</strong>, "
            f"{confidence}."
            f"<div class='bar'><div class='fill' style='width:{prob * 100:.1f}%'></div>"
            f"<div class='mid'></div></div>"
            f"<small>0% &middot; the marker is the 0.5 decision boundary &middot; 100%</small></div>"
        )

    card = (
        "<div class='result'>"
        f"<div class='eyebrow'>Instant-booking likelihood &middot; {payload['city']}</div>"
        f"<div class='price'>{label}</div>"
        f"{detail}"
        f"<div class='meta'>class {pred} &middot; {ms:,.0f} ms round trip "
        f"&middot; {BOOKING_ENDPOINT}</div>"
        "</div>"
    )
    return card, booking_derived_preview(payload), request_json, json.dumps(records, indent=2)


BOOKING_PRESETS = {
    "Professional host": dict(
        city="Paris", latitude=48.8867, longitude=2.3431, property_type="Entire apartment",
        room_type="Entire home/apt", accommodates=4, bedrooms=2, price=150.0,
        **_amen("wifi", "air conditioning", "kitchen", "pool", "gym", "tv"),
        minimum_nights=2, maximum_nights=1125, host_since="2018-05-15",
        host_total_listings_count=12, host_response_time="within an hour",
        host_response_rate=100.0, host_acceptance_rate=95.0,
        host_is_superhost=True, host_has_profile_pic=True, host_identity_verified=True,
        **review_defaults(),
    ),
    "Casual single-listing host": dict(
        city="Rome", latitude=41.9028, longitude=12.4964, property_type="Private room in apartment",
        room_type="Private room", accommodates=2, bedrooms=1, price=55.0,
        **_amen("wifi", "heating", "essentials"),
        minimum_nights=3, maximum_nights=90, host_since="2020-11-02",
        host_total_listings_count=1, host_response_time="a few days or more",
        host_response_rate=40.0, host_acceptance_rate=35.0,
        host_is_superhost=False, host_has_profile_pic=True, host_identity_verified=False,
        **review_defaults(),
    ),
}
for _n, _p in BOOKING_PRESETS.items():
    assert set(_p) == set(BOOKING_UI_FIELDS), f"{_n}: {set(BOOKING_UI_FIELDS) ^ set(_p)}"

print(f"Classifier handler ready. {len(BOOKING_UI_FIELDS)} UI controls -> "
      f"{len(BOOKING_FIELDS)} fields -> {BOOKING_ENDPOINT}")

Classifier handler ready. 27 UI controls -> 26 fields -> iti113-team14-airbnb-instant-booking


### 6.5 The third model: a recommender that consumes the other two

The price model answers *what should this listing cost*. The classifier answers *will this host let
me book instantly*. Neither answers the question a guest actually asks: **out of everything in this
city that fits my dates and my budget, which ten should I look at first?**

That is a **ranking** problem, and it is built on top of the other two rather than beside them:

| | Airbnb price | Instant booking | Recommender |
|---|---|---|---|
| Task | regression | binary classification | ranking / re-ranking |
| Serving | SageMaker serverless endpoint | SageMaker serverless endpoint | in-process over a scored dataset |
| Output | `suggested_nightly_price` | `probability_class_1` | an ordered shortlist |
| Input | one listing | one listing | a *search*, plus a corpus of listings |

The pipeline is the standard two-stage recommender shape:

1. **Candidate generation** — hard filters. City, party size, budget. Cheap, and it throws away
   99% of the corpus. No model is involved: a listing that sleeps 2 is not a *slightly worse* match
   for a party of 4, it is not a match at all.
2. **Re-ranking** — a score over the survivors, from the two models' outputs:

   ```text
   deal_value          = pred_price - price          # positive = cheaper than the model says it should be
   scaled_value_score  = min-max(deal_value)         # 0-1 within the comparison set
   ml_score = w_value * scaled_value_score + w_convenience * instant_book_prob
   ```

   The two weights are the guest's preference, not a fitted parameter. Sliding `w_value` to 1.0
   asks for the biggest bargains; sliding `w_convenience` to 1.0 asks for the listings most likely
   to be bookable without waiting on a host to reply.

#### Where `pred_price` and `instant_book_prob` come from

This matters, and it is easy to state wrongly in a report.

`master_recsys_dataset.csv` already **carries** those two columns. They were produced by a **batch
scoring job** — every listing run through both models once, offline, and the results written back to
the file. The FastAPI service in `Test API Gradio.ipynb` reads them; it never calls a SageMaker
endpoint at request time. That is the correct production design: re-scoring 280,000 listings on
every keystroke would be absurd, and a serverless endpoint at `max_concurrency=5` would fall over.

So the honest description of the default path is:

> The recommender **consumes the outputs of** the regression and classification models. It does not
> **invoke** them per request.

The **Live re-scoring** switch on the tab makes the stronger claim true. Turn it on and the tab
takes the shortlist after filtering, POSTs those rows to `PRICE_ENDPOINT` and `BOOKING_ENDPOINT`,
and re-ranks on the fresh values. That is a real round trip to both endpoints, and it is slow and
capped for exactly the reason batch scoring exists — which is the point worth demonstrating.

#### On the min-max scaling

The FastAPI version fits the min-max over the **entire dataset, all ten cities at once**. Bangkok
prices are in THB and Paris in EUR, so a single `deal_value` range is dominated by the largest
currency: nearly every Paris listing lands in a narrow band near the bottom, and `w_value` barely
moves the ordering. Scaling **within the filtered candidates** — one city, one currency — fixes it.
The tab offers both so the difference is visible rather than argued about.

In [13]:
import io
import pandas as pd

# --- Where the batch-scored corpus lives -----------------------------------------
# Its own project prefix: this dataset is an output of the recommender's own notebook,
# not of the price or booking pipelines.
RECSYS_PROJECT = "airbnb-recommendation"
RECSYS_PREFIX  = f"iti113/{TEAM_ID}/data/{RECSYS_PROJECT}"
RECSYS_KEY     = f"{RECSYS_PREFIX}/processed/master_recsys_dataset.csv"

# Without these six columns there is nothing to rank.
RECSYS_REQUIRED = ["listing_id", "city", "accommodates", "price",
                   "pred_price", "instant_book_prob"]

_RECSYS_LOCAL = ["master_recsys_dataset.csv",
                 os.path.join("data", "master_recsys_dataset.csv"),
                 os.path.join("processed", "master_recsys_dataset.csv")]


def _read_recsys_csv():
    # Local file first, then S3 through the boto3 client Section 2 already built.
    # Deliberately not pd.read_csv("s3://...") -- that needs s3fs, which is one more
    # thing to be installed on a demo machine at the worst possible moment.
    for p in _RECSYS_LOCAL:
        if os.path.isfile(p):
            return pd.read_csv(p, encoding="utf-8", encoding_errors="replace",
                               low_memory=False), p
    body = s3.get_object(Bucket=BUCKET, Key=RECSYS_KEY)["Body"].read()
    return (pd.read_csv(io.BytesIO(body), encoding="utf-8", encoding_errors="replace",
                        low_memory=False), f"s3://{BUCKET}/{RECSYS_KEY}")


# Never raise here. A failure to load must leave the other two tabs working and the
# recommender tab able to say why it is empty.
RECSYS_DF = None
RECSYS_SOURCE = None
RECSYS_ERROR = None
RECSYS_CITIES = list(CITIES)
RECSYS_LIVE_OK = False
RECSYS_LIVE_MISSING = []
RATE_IS_UNIT = {}

try:
    RECSYS_DF, RECSYS_SOURCE = _read_recsys_csv()

    absent = [c for c in RECSYS_REQUIRED if c not in RECSYS_DF.columns]
    if absent:
        raise KeyError(f"the scored dataset is missing {absent}. It must carry the batch-scored "
                       "pred_price and instant_book_prob columns, not just the raw listing fields.")

    # Drop rows the ranker cannot score. Silently ranking a NaN is worse than a smaller corpus.
    _before = len(RECSYS_DF)
    RECSYS_DF = RECSYS_DF.dropna(subset=["pred_price", "instant_book_prob", "price",
                                         "accommodates", "city"]).copy()
    RECSYS_DROPPED = _before - len(RECSYS_DF)

    # Positive residual = the model thinks it is worth more than it is asking = a deal.
    RECSYS_DF["deal_value"] = RECSYS_DF["pred_price"] - RECSYS_DF["price"]

    # The global min-max the FastAPI service uses, precomputed once so the "whole dataset"
    # scope is free at request time.
    _lo, _hi = RECSYS_DF["deal_value"].min(), RECSYS_DF["deal_value"].max()
    RECSYS_DF["scaled_value_global"] = (0.5 if _hi <= _lo
                                        else (RECSYS_DF["deal_value"] - _lo) / (_hi - _lo))

    RECSYS_CITIES = sorted(RECSYS_DF["city"].dropna().astype(str).unique().tolist())

    # Which scale do the rate columns hold? Guessing here is how a 100x error gets shipped.
    for _col in ("host_response_rate", "host_acceptance_rate"):
        if _col in RECSYS_DF.columns:
            _s = pd.to_numeric(RECSYS_DF[_col], errors="coerce").dropna()
            RATE_IS_UNIT[_col] = bool(len(_s)) and float(_s.max()) <= 1.0

    # Live re-scoring needs the raw fields both endpoints were trained on.
    RECSYS_LIVE_MISSING = sorted({c for c in list(SERVING_COLUMNS) + list(BOOKING_FIELDS)
                                  if c not in RECSYS_DF.columns})
    RECSYS_LIVE_OK = not RECSYS_LIVE_MISSING

except Exception as e:
    # A half-loaded frame is worse than none: it would reach the tab missing the columns
    # the ranker assumes, and fail later with something much harder to read.
    RECSYS_DF = None
    RECSYS_ERROR = f"{e.__class__.__name__}: {e}"


# --- verification output ----------------------------------------------------------
if RECSYS_DF is None:
    print("Recommender corpus : NOT LOADED")
    print(f"                     {RECSYS_ERROR}")
    print(f"                     Looked for {_RECSYS_LOCAL} locally, then")
    print(f"                     s3://{BUCKET}/{RECSYS_KEY}")
    print("                     The tab will render and explain this; the other two tabs are unaffected.")
else:
    print(f"Recommender corpus : {len(RECSYS_DF):,} scored listings")
    print(f"Source             : {RECSYS_SOURCE}")
    print(f"Dropped            : {RECSYS_DROPPED:,} rows with a null price / score")
    print(f"Cities             : {len(RECSYS_CITIES)} -> {', '.join(RECSYS_CITIES)}")
    print(f"deal_value range   : {RECSYS_DF['deal_value'].min():,.0f} .. "
          f"{RECSYS_DF['deal_value'].max():,.0f} (mixed currencies - see the note above)")
    print(f"instant_book_prob  : {RECSYS_DF['instant_book_prob'].min():.3f} .. "
          f"{RECSYS_DF['instant_book_prob'].max():.3f}")
    for _col, _unit in RATE_IS_UNIT.items():
        print(f"{_col:<19}: stored as {'0-1' if _unit else '0-100'} in this CSV")
    if RECSYS_LIVE_OK:
        print("Live re-scoring    : available - the corpus carries every field both endpoints need")
    else:
        print(f"Live re-scoring    : unavailable - corpus is missing {RECSYS_LIVE_MISSING}")
        print("                     The tab falls back to the batch-scored columns.")

Recommender corpus : 279,712 scored listings
Source             : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-recommendation/processed/master_recsys_dataset.csv
Dropped            : 0 rows with a null price / score
Cities             : 10 -> Bangkok, Cape Town, Hong Kong, Istanbul, Mexico City, New York, Paris, Rio de Janeiro, Rome, Sydney
deal_value range   : -624,677 .. 17,163 (mixed currencies - see the note above)
instant_book_prob  : 0.000 .. 1.000
host_response_rate : stored as 0-1 in this CSV
host_acceptance_rate: stored as 0-1 in this CSV
Live re-scoring    : available - the corpus carries every field both endpoints need


### 6.6 A row in the corpus is not a payload for either endpoint

Live re-scoring means taking a row out of `master_recsys_dataset.csv` and posting it to two
endpoints that disagree with each other about how to write the same fact down. Section 5.3 listed
those disagreements for hand-typed input; here they have to be handled row by row, in code:

| Field | In the CSV | Price endpoint wants | Booking endpoint wants |
|---|---|---|---|
| `host_is_superhost` and the other flags | `t` / `f` | `t` / `f` | `1` / `0` |
| `host_response_rate`, `host_acceptance_rate` | whatever the raw file holds — detected at load | `0-100` | `0-1` |
| `room_type` | `Entire home/apt` | `Entire place` (renamed in Notebook 01) | `Entire home/apt` |
| missing values | `NaN` | `null` | `null`, then zero-filled internally |

Two of these fail *quietly*, which is what makes them worth writing down:

- **The rate scale.** Send `95` to a model trained on `0-1` and it does not error, it just reads a
  host with a 95% acceptance rate as one sitting 95 standard deviations out. The prediction comes
  back looking like a prediction. `_rate()` is the only place the conversion happens, and it reads
  the direction from the data at load time rather than from an assumption.
- **`room_type`.** `Entire home/apt` is not a level the price model's one-hot encoder has ever seen,
  so it encodes as all-zeros — indistinguishable from a missing room type, and the single largest
  category in the dataset silently becomes a null. `_price_room_type()` maps it back.

`NaN` is the third: `json.dumps(float('nan'))` emits the bare token `NaN`, which is not valid JSON.
Python's parser accepts it, so it survives the round trip and lands in the model as a float NaN
rather than a handled null. `_clean()` converts it to `None` before serialisation.

The batching is a capacity decision, not a style one. Both endpoints are serverless at
`max_concurrency=5`, so the shortlist is capped and posted in chunks — and if an endpoint turns out
to reject a list, the code falls back to one row per request and the cap is what keeps that from
becoming a five-minute hang.

In [14]:
# --- Row -> payload adapters ------------------------------------------------------

def _clean(v):
    # NaN / NaT / pd.NA -> None. json.dumps(nan) emits the bare token NaN, which is not
    # valid JSON; Python's parser accepts it, so it survives the round trip and reaches
    # the model as a float NaN instead of a handled null.
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v


def _num(v, cast=float, default=None):
    v = _clean(v)
    if v is None:
        return default
    try:
        return cast(v)   # also strips numpy types: json cannot serialise np.int64
    except (TypeError, ValueError):
        return default


def _as_tf(v):
    # The price model's boolean encoding: raw 't' / 'f', exactly as trained.
    v = _clean(v)
    if v is None:
        return "f"
    if isinstance(v, (bool, int, float)):
        return "t" if float(v) >= 0.5 else "f"
    return "t" if str(v).strip().lower() in {"t", "true", "yes", "1"} else "f"


def _as01(v):
    # The classifier's boolean encoding: 1 / 0.
    return 1 if _as_tf(v) == "t" else 0


def _rate(v, col, want_unit):
    # Response / acceptance rates. Reads whatever scale the CSV holds (detected at load)
    # and emits the scale the target endpoint was trained on. One function, because a
    # 100x error here does not raise -- it just returns a confident wrong number.
    v = _clean(v)
    if isinstance(v, str):
        v = v.replace("%", "").replace(",", "").strip()
    v = _num(v)
    if v is None:
        return None
    stored_as_unit = RATE_IS_UNIT.get(col, False)
    if want_unit:                                    # target trained on 0-1
        return v if stored_as_unit else v / 100.0
    return v * 100.0 if stored_as_unit else v        # target trained on 0-100


def _price_room_type(v):
    # Notebook 01 renamed 'Entire home/apt' to 'Entire place' for the price model. The
    # recommender's corpus holds the raw platform value, so an unmapped pass-through
    # would one-hot encode the largest category in the dataset as all-zeros.
    levels = room_type_choices()
    v = str(_clean(v) or "").strip()
    if v in levels:
        return v
    for alias in {"entire home/apt": ["Entire place", "Entire home/apt"],
                  "entire place": ["Entire home/apt", "Entire place"]}.get(v.lower(), []):
        if alias in levels:
            return alias
    return v or (levels[0] if levels else "Entire place")


def _date10(v):
    v = _clean(v)
    return (str(v)[:10] or None) if v is not None else None


def price_payload_from_row(row):
    # One corpus row -> the price model's 27-field SERVING_COLUMNS contract.
    p = {
        "city": _clean(row.get("city")),
        "latitude": _num(row.get("latitude")),
        "longitude": _num(row.get("longitude")),
        "neighbourhood": _clean(row.get("neighbourhood")),
        "property_type": _clean(row.get("property_type")) or "Other",
        "room_type": _price_room_type(row.get("room_type")),
        "accommodates": _num(row.get("accommodates"), int, 1),
        "bedrooms": _num(row.get("bedrooms"), int),
        "amenities": str(_clean(row.get("amenities")) or "[]"),
        "minimum_nights": _num(row.get("minimum_nights"), int, 1),
        "maximum_nights": _num(row.get("maximum_nights"), int, MAX_NIGHTS_CAP),
        "instant_bookable": _as_tf(row.get("instant_bookable")),
        "host_since": _date10(row.get("host_since")),
        "host_is_superhost": _as_tf(row.get("host_is_superhost")),
        "host_has_profile_pic": _as_tf(row.get("host_has_profile_pic")),
        "host_identity_verified": _as_tf(row.get("host_identity_verified")),
        "host_total_listings_count": _num(row.get("host_total_listings_count"), int, 0),
        "host_response_time": _clean(row.get("host_response_time")),
        "host_response_rate": _rate(row.get("host_response_rate"),
                                    "host_response_rate", want_unit=False),
        "host_acceptance_rate": _rate(row.get("host_acceptance_rate"),
                                      "host_acceptance_rate", want_unit=False),
    }
    for col in REVIEW_COLS:
        p[col] = _num(row.get(col))
    return {k: p.get(k) for k in SERVING_COLUMNS}


def booking_payload_from_row(row):
    # The same corpus row -> the classifier's 26 fields, on the classifier's conventions.
    p = {
        "city": _clean(row.get("city")),
        "amenities": str(_clean(row.get("amenities")) or "[]"),
        "host_total_listings_count": _num(row.get("host_total_listings_count"), int, 0),
        "price": _num(row.get("price"), float, 0.0),
        "host_since": _date10(row.get("host_since")),
        "accommodates": _num(row.get("accommodates"), int, 1),
        "bedrooms": _num(row.get("bedrooms"), int, 1),
        "property_type": _clean(row.get("property_type")) or "Other",
        "room_type": str(_clean(row.get("room_type")) or "Entire home/apt"),   # raw, not renamed
        "host_response_time": _clean(row.get("host_response_time")) or "no history",
        "host_response_rate": _rate(row.get("host_response_rate"),
                                    "host_response_rate", want_unit=True),
        "host_acceptance_rate": _rate(row.get("host_acceptance_rate"),
                                      "host_acceptance_rate", want_unit=True),
        "host_is_superhost": _as01(row.get("host_is_superhost")),
        "host_has_profile_pic": _as01(row.get("host_has_profile_pic")),
        "host_identity_verified": _as01(row.get("host_identity_verified")),
        "minimum_nights": _num(row.get("minimum_nights"), int, 1),
        "maximum_nights": _num(row.get("maximum_nights"), int, MAX_NIGHTS_CAP),
        "latitude": _num(row.get("latitude")),
        "longitude": _num(row.get("longitude")),
    }
    for col in REVIEW_COLS:
        p[col] = _num(row.get(col))
    return {k: p.get(k) for k in BOOKING_FIELDS}


# --- Live re-scoring through both endpoints ---------------------------------------

RECSYS_CHUNK = 25   # rows per request; both endpoints are serverless at max_concurrency=5


def _post_chunks(payloads, invoke, extract):
    # Try batched first: the price endpoint's input_fn takes a list, which turns 25 cold
    # starts into 1. Fall back to one row per request if an endpoint refuses a list.
    values, mode = [], "batched"
    try:
        for i in range(0, len(payloads), RECSYS_CHUNK):
            part = payloads[i:i + RECSYS_CHUNK]
            recs, _ = invoke(part)
            if isinstance(recs, dict):
                recs = [recs]
            if not isinstance(recs, list) or len(recs) != len(part):
                raise ValueError(f"endpoint returned {len(recs)} records for {len(part)} rows")
            values.extend(extract(r) for r in recs)
        return values, mode
    except Exception:
        values, mode = [], "one row per request"
        for p in payloads:
            recs, _ = invoke(p)
            values.append(extract(recs[0] if isinstance(recs, list) else recs))
        return values, mode


def rescore_live(frame):
    # POST the shortlist to PRICE_ENDPOINT and BOOKING_ENDPOINT and return the frame with
    # fresh pred_price / instant_book_prob. Raises: the caller decides what to say about it.
    rows = frame.to_dict(orient="records")
    t0 = time.perf_counter()

    preds, price_mode = _post_chunks(
        [price_payload_from_row(r) for r in rows], invoke_airbnb_endpoint,
        lambda rec: float(rec["suggested_nightly_price"]))
    probs, booking_mode = _post_chunks(
        [booking_payload_from_row(r) for r in rows], invoke_booking_endpoint,
        lambda rec: float(rec.get("probability_class_1") or 0.0))

    ms = (time.perf_counter() - t0) * 1000.0
    out = frame.copy()
    out["pred_price"] = preds
    out["instant_book_prob"] = probs
    out["deal_value"] = out["pred_price"] - out["price"]
    return out, {"n": len(rows), "ms": ms, "price_mode": price_mode,
                 "booking_mode": booking_mode,
                 "price_sample": price_payload_from_row(rows[0]) if rows else {},
                 "booking_sample": booking_payload_from_row(rows[0]) if rows else {}}


# --- Candidate generation and re-ranking ------------------------------------------

SCOPE_CANDIDATES = "These candidates (per city, recommended)"
SCOPE_GLOBAL = "The whole dataset (matches the FastAPI service)"

RECSYS_UI_FIELDS = ["city", "min_guests", "max_price", "w_value", "w_convenience",
                    "top_k", "scope", "live", "live_n", "use_api", "api_url"]


def generate_candidates(city, min_guests, max_price):
    # Stage 1. Hard filters only -- a listing that sleeps 2 is not a slightly worse match
    # for a party of 4, it is not a match at all, and no score should be able to rescue it.
    df = RECSYS_DF
    return df[(df["city"].astype(str).str.lower() == str(city).strip().lower())
              & (df["accommodates"] >= int(min_guests))
              & (df["price"] <= float(max_price))].copy()


def rerank(cand, w_value, w_convenience, scope):
    # Stage 2. Normalising the weights by their sum cannot change the ordering -- it is the
    # same positive constant on both terms -- but it puts ml_score on 0-1 so the number
    # printed next to a listing means something.
    total = float(w_value) + float(w_convenience)
    wv, wc = (0.5, 0.5) if total <= 0 else (float(w_value) / total, float(w_convenience) / total)

    if scope == SCOPE_GLOBAL and "scaled_value_global" in cand.columns:
        cand["scaled_value_score"] = cand["scaled_value_global"]
    else:
        lo, hi = cand["deal_value"].min(), cand["deal_value"].max()
        cand["scaled_value_score"] = 0.5 if hi <= lo else (cand["deal_value"] - lo) / (hi - lo)

    cand["ml_score"] = wv * cand["scaled_value_score"] + wc * cand["instant_book_prob"]
    return cand.sort_values("ml_score", ascending=False), (wv, wc)


RECSYS_RETURN_COLS = ["listing_id", "city", "accommodates", "price", "pred_price",
                      "instant_book_prob", "ml_score"]


def _recsys_rows_html(results, ccy):
    body = []
    for i, rec in enumerate(results, start=1):
        price, pred = float(rec["price"]), float(rec["pred_price"])
        deal = pred - price
        cls = "good" if deal > 0 else "bad"
        sign = "+" if deal > 0 else ""
        body.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td class='id'>{int(rec['listing_id'])}</td>"
            f"<td>{price:,.0f}</td>"
            f"<td>{pred:,.0f}</td>"
            f"<td class='deal {cls}'>{sign}{deal:,.0f}</td>"
            f"<td>{float(rec['instant_book_prob']):.1%}</td>"
            f"<td>{float(rec['ml_score']):.4f}</td>"
            "</tr>")
    return (
        "<table class='rank'><thead><tr>"
        "<th>#</th><th>Listing</th>"
        f"<th>Asking ({ccy})</th><th>Model value</th><th>Deal</th>"
        "<th>P(instant)</th><th>Score</th>"
        "</tr></thead><tbody>" + "".join(body) + "</tbody></table>")


def _recsys_empty(message):
    return (f"<div class='result'><div class='eyebrow'>No shortlist</div>"
            f"<div class='band'>{message}</div></div>")


def recommend_listings(*values):
    # Gradio click handler. Returns (shortlist card, explanation, request JSON, response JSON)
    # -- the same four outputs as the other two tabs.
    ui = dict(zip(RECSYS_UI_FIELDS, values))

    # The POST body of the FastAPI service, field for field. Shown whether or not the
    # service is the thing actually being called, because it is the contract either way.
    query = {"city": str(ui["city"]),
             "min_guests": int(ui["min_guests"]),
             "max_price": float(ui["max_price"]),
             "w_value": float(ui["w_value"]),
             "w_convenience": float(ui["w_convenience"])}
    request_json = json.dumps(query, indent=2)
    ccy = CURRENCY.get(ui["city"], "")

    # --- Path A: delegate to the running FastAPI service ---------------------------
    if ui.get("use_api"):
        try:
            import requests
            resp = requests.post(str(ui["api_url"]).strip(), json=query, timeout=90)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            return (_recsys_empty(f"Could not reach the service at "
                                  f"<code>{ui['api_url']}</code>.<br><code>"
                                  f"{e.__class__.__name__}: {e}</code><br><br>"
                                  "Start the FastAPI cell in <code>Test API Gradio.ipynb</code>, or "
                                  "untick the box to rank in this process instead."),
                    "", request_json, "")
        results = data.get("results", [])
        if not results:
            return (_recsys_empty(data.get("message", "The service returned no listings.")),
                    "", request_json, json.dumps(data, indent=2))
        card = ("<div class='result'>"
                f"<div class='eyebrow'>Top {len(results)} &middot; {ui['city']} "
                f"&middot; via FastAPI</div>"
                f"{_recsys_rows_html(results, ccy)}"
                f"<div class='meta'>{ui['api_url']} &middot; scored offline, "
                f"served from the precomputed columns</div></div>")
        note = ("Ranked by the external service. The **Scope** and **Live re-scoring** controls are "
                "ignored on this path — the service has its own copy of the logic, with the "
                "min-max fitted across all ten cities.")
        return card, note, request_json, json.dumps(data, indent=2)

    # --- Path B: rank in this process ----------------------------------------------
    if RECSYS_DF is None:
        return (_recsys_empty(f"The scored corpus did not load.<br><code>{RECSYS_ERROR}</code>"),
                "", request_json, "")

    cand = generate_candidates(query["city"], query["min_guests"], query["max_price"])
    n_candidates = len(cand)
    if n_candidates == 0:
        return (_recsys_empty(f"Nothing in {query['city']} sleeps {query['min_guests']} at or under "
                              f"{query['max_price']:,.0f} {ccy}. Raise the budget or drop the party size."),
                "", request_json, json.dumps({"results": [], "count": 0}, indent=2))

    # Optional stage 3: re-score the shortlist against the live endpoints.
    live_note, live_meta = "", None
    if ui.get("live"):
        if not RECSYS_LIVE_OK:
            live_note = (f"\n\n**Live re-scoring skipped.** The corpus is missing "
                         f"`{'`, `'.join(RECSYS_LIVE_MISSING)}`, so a payload cannot be built "
                         "from it. Ranked on the batch-scored columns instead.")
        else:
            shortlist_n = int(ui["live_n"])
            head, _ = rerank(cand, query["w_value"], query["w_convenience"], ui["scope"])
            head = head.head(shortlist_n)
            try:
                cand, live_meta = rescore_live(head)
            except Exception as e:
                return (explain_error(e),
                        f"Live re-scoring failed after filtering {n_candidates:,} candidates down "
                        f"to {shortlist_n}. Untick **Live re-scoring** to rank on the batch-scored "
                        "columns, which do not need either endpoint.",
                        request_json, "")

    ranked, (wv, wc) = rerank(cand, query["w_value"], query["w_convenience"], ui["scope"])
    top = ranked.head(int(ui["top_k"]))
    results = top[RECSYS_RETURN_COLS].to_dict(orient="records")
    response = {"results": results, "count": len(results)}

    if live_meta:
        eyebrow = (f"Top {len(results)} &middot; {query['city']} &middot; "
                   f"re-scored live on both endpoints")
        meta = (f"{live_meta['n']} listings &times; 2 endpoints in {live_meta['ms']:,.0f} ms "
                f"&middot; price {live_meta['price_mode']} &middot; "
                f"booking {live_meta['booking_mode']}")
    else:
        eyebrow = f"Top {len(results)} &middot; {query['city']} &middot; from batch scores"
        meta = (f"{n_candidates:,} candidates survived the filters &middot; "
                f"no endpoint was called")

    card = ("<div class='result'>"
            f"<div class='eyebrow'>{eyebrow}</div>"
            f"<div class='band'>Weights normalised to "
            f"<strong>{wv:.2f} value</strong> / <strong>{wc:.2f} convenience</strong>. "
            f"<em>Deal</em> is model value minus asking price: positive means the model reads the "
            f"listing as underpriced.</div>"
            f"{_recsys_rows_html(results, ccy)}"
            f"<div class='meta'>{meta}</div></div>")

    # The explanation panel: what happened, in the order it happened.
    lines = [
        "| stage | outcome |", "|---|---|",
        f"| corpus | {len(RECSYS_DF):,} scored listings from `{RECSYS_SOURCE}` |",
        f"| 1. candidate generation | {n_candidates:,} survived `city == {query['city']}`, "
        f"`accommodates >= {query['min_guests']}`, `price <= {query['max_price']:,.0f}` |",
    ]
    if live_meta:
        lines.append(f"| 2. live re-scoring | {live_meta['n']} rows POSTed to `{PRICE_ENDPOINT}` "
                     f"and `{BOOKING_ENDPOINT}`, {live_meta['ms']:,.0f} ms |")
    else:
        lines.append("| 2. live re-scoring | off - `pred_price` and `instant_book_prob` read "
                     "from the batch-scored columns |")
    lines.append(f"| 3. value scaling | min-max over **{ui['scope'].split('(')[0].strip().lower()}** |")
    lines.append(f"| 4. re-ranking | `{wv:.2f} x scaled_value + {wc:.2f} x instant_book_prob` |")
    lines.append(f"| 5. cut | top {len(results)} of {n_candidates:,} |")
    note = "How this shortlist was built:\n\n" + "\n".join(lines) + live_note

    if live_meta:
        note += ("\n\n<details><summary>The first row's two payloads, as posted</summary>\n\n"
                 "```json\n" + json.dumps(
                     {"to_price_endpoint": live_meta["price_sample"],
                      "to_booking_endpoint": live_meta["booking_sample"]}, indent=2, default=str)
                 + "\n```\n</details>")

    return card, note, request_json, json.dumps(response, indent=2, default=str)


# --- Example searches for the UI buttons ------------------------------------------
def _search_preset(**overrides):
    p = {"city": "Paris", "min_guests": 2, "max_price": 150.0,
         "w_value": 0.5, "w_convenience": 0.5, "top_k": 10,
         "scope": SCOPE_CANDIDATES, "live": False, "live_n": 25,
         "use_api": False, "api_url": "http://localhost:8000/recommend"}
    p.update(overrides)
    return p


RECSYS_PRESETS = {
    "Paris, balanced (the API Test Client query)": _search_preset(scope=SCOPE_GLOBAL),
    "Paris, bargain hunter": _search_preset(w_value=1.0, w_convenience=0.0),
    "Paris, book it now": _search_preset(w_value=0.0, w_convenience=1.0),
    "Bangkok family of 4": _search_preset(city="Bangkok", min_guests=4, max_price=3000.0),
}
for _n, _p in RECSYS_PRESETS.items():
    assert set(_p) == set(RECSYS_UI_FIELDS), f"{_n}: {set(RECSYS_UI_FIELDS) ^ set(_p)}"

print(f"Recommender handler ready. {len(RECSYS_UI_FIELDS)} UI controls -> "
      f"filter -> {'live re-score -> ' if RECSYS_LIVE_OK else ''}rank")
print(f"Presets: {list(RECSYS_PRESETS)}")

Recommender handler ready. 11 UI controls -> filter -> live re-score -> rank
Presets: ['Paris, balanced (the API Test Client query)', 'Paris, bargain hunter', 'Paris, book it now', 'Bangkok family of 4']


### 6.7 The fourth model: the Ridge baseline, scored in-process

The three tabs above all end in a network call. This one does not, and the reason is worth
stating plainly before any code runs.

**The baseline was never deployed, and it was never written to S3.** Notebook 02 section 4
fits `RidgeCV(alphas=np.logspace(-2, 3, 12), cv=3)` on `X_fit` and persists it with a single
line — `log_sklearn_model(ridge, X_val)` — inside the run `team14_s1402_ridge_baseline`
(`60fdc021…`). That writes it to the SageMaker MLflow App and nowhere else. Notebook 02's
own publish step, 8A, mirrors **only the champion** to
`s3://nyp-26s1-iti113/…/artifacts/champion/443da800…/`. So MLflow is the single durable copy,
and section 6.7a below tries a local file and an S3 key first — both are placeholders you can
point at your own mirror — before falling back to the run URI that actually exists today.

**The baseline is a bare estimator, not a pipeline.** This is the part that decides the whole
design of the tab. The champion tab can post 27 raw fields because `model.tar.gz` contains
`model_pipeline.joblib`, preprocessing and estimator fused into one object. Notebook 02 never
saw a raw field: it read `train_features.csv`, the 88-column matrix Notebook 01 had already
produced, and fitted Ridge directly on that. What came out of `log_sklearn_model` therefore
expects 88 engineered floats in a fixed order and has no idea what a city name is.

Something has to bridge that gap, and the choice matters more than it looks. Re-implementing
Notebook 01's engineering here would work right up until it drifted — and then the price
difference between the two tabs would be measuring *my transform code*, not the two models.
So 6.7b borrows the champion's own fitted preprocessor out of the deployed pipeline
(`Pipeline[:-1]`). Identical 88 columns into both models, one difference left to explain.

| | Champion tab | Baseline tab |
|---|---|---|
| Model | HistGradientBoostingRegressor | RidgeCV, alpha 0.01 |
| Where it runs | serverless endpoint | this kernel |
| Preprocessing | inside the container | champion's preprocessor, borrowed |
| Latency | cold start up to ~60 s | sub-millisecond, no network |
| Reported error | MAE(log) 0.3207, MAPE 33.4% (test) | MAE(log) 0.4255, MAPE 46.8% (validation) |

One caveat carried into the UI rather than buried here: those two error columns are **not**
measured on the same split. Notebook 02's protocol only lets the champion read the test set,
so Ridge is quoted on validation and the champion on test. The ranking is safe; the decimal
places are not a like-for-like subtraction.

#### 6.7a Loading the estimator

Three sources, cheapest first, none of them fatal. A baseline that cannot be loaded has to
leave the other three tabs working and say why in its own tab — the same degradation rule the
endpoint tabs follow in Section 2.

In [15]:
# =====================================================================================
# 6.7a  Load the baseline RidgeCV estimator
# -------------------------------------------------------------------------------------
# WHERE IT ACTUALLY LIVES. Notebook 02 section 4 fits
#     RidgeCV(alphas=np.logspace(-2, 3, 12), cv=3).fit(X_fit, y_fit)
# and persists it with one line -- `log_sklearn_model(ridge, X_val)` -- inside the run
# named `team14_s1402_ridge_baseline`. That writes it to the SageMaker MLflow App, and
# nowhere else. Notebook 02's 8A publish step mirrors ONLY the champion to
# s3://<bucket>/.../artifacts/champion/<run-id>/, so there is no S3 copy of the baseline
# unless you make one (see nb02_publish_ridge_baseline.py, which writes both the joblib
# and a baseline_model.json hand-off).
#
# Three sources are tried, cheapest first. Nothing here raises: a baseline that cannot
# be loaded must leave the other three tabs working, and must say why in the UI.
# =====================================================================================
import tempfile

import joblib
import numpy as np
import pandas as pd

# --- EDIT THESE IF YOUR LINEAGE DIFFERS ----------------------------------------------
BASELINE_RUN_ID     = "60fdc0210519493aa5e23d1c77703819"   # team14_s1402_ridge_baseline
MLFLOW_TRACKING_URI = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS"
MLFLOW_EXPERIMENT   = f"{COURSE}/{TEAM_ID}/Experiment1"

# Source 2 placeholder: set this to wherever you mirrored the baseline, if you did.
# The default is the key nb02_publish_ridge_baseline.py writes to.
BASELINE_S3_BUCKET = BUCKET
BASELINE_S3_KEY    = f"{ARTIFACTS_PREFIX}/baseline/ridge_baseline.joblib"

# Source 1 placeholder: local files, checked in this order.
BASELINE_LOCAL_CANDIDATES = [
    "ridge_baseline.joblib",
    os.path.join("artifacts", "ridge_baseline.joblib"),
    os.path.join("downloaded_model", "ridge_baseline.joblib"),
    "ridge_baseline.pkl",
]
# --------------------------------------------------------------------------------------

BASELINE_MODEL, BASELINE_SOURCE, BASELINE_ERROR = None, None, None
BASELINE_ATTEMPTS = []


def _baseline_try(fn):
    """Run one loading strategy. Record what happened; never propagate."""
    global BASELINE_MODEL, BASELINE_SOURCE
    if BASELINE_MODEL is not None:
        return
    try:
        result = fn()
    except Exception as e:
        BASELINE_ATTEMPTS.append((fn.__name__, f"{type(e).__name__}: {e}"))
        return
    if result is None:
        BASELINE_ATTEMPTS.append((fn.__name__, "not found"))
        return
    BASELINE_MODEL, BASELINE_SOURCE = result
    BASELINE_ATTEMPTS.append((fn.__name__, f"loaded from {BASELINE_SOURCE}"))


def _from_local_file():
    for p in BASELINE_LOCAL_CANDIDATES:
        if os.path.isfile(p):
            return joblib.load(p), os.path.abspath(p)
    return None


def _from_s3():
    # head_object first: a 404 here is the expected case, not an error worth a traceback.
    try:
        s3.head_object(Bucket=BASELINE_S3_BUCKET, Key=BASELINE_S3_KEY)
    except Exception:
        return None
    dst = Path(tempfile.mkdtemp()) / Path(BASELINE_S3_KEY).name
    s3.download_file(BASELINE_S3_BUCKET, BASELINE_S3_KEY, str(dst))
    return joblib.load(dst), f"s3://{BASELINE_S3_BUCKET}/{BASELINE_S3_KEY}"


def _from_mlflow():
    # The one that actually exists today. MLflow 3 stores this run's model as
    # model.skops, so the sklearn flavour is the right loader; pyfunc is the fallback
    # for older/newer serialisations.
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    uri = f"runs:/{BASELINE_RUN_ID}/model"
    try:
        import mlflow.sklearn
        return mlflow.sklearn.load_model(uri), uri
    except Exception:
        return mlflow.pyfunc.load_model(uri), f"{uri} (pyfunc wrapper)"


for _strategy in (_from_local_file, _from_s3, _from_mlflow):
    _baseline_try(_strategy)

if BASELINE_MODEL is None:
    BASELINE_ERROR = ("no source yielded a model — see the attempts below, then either set "
                      "BASELINE_S3_KEY / BASELINE_LOCAL_CANDIDATES, or run "
                      "nb02_publish_ridge_baseline.py in Notebook 02 to mirror it")

print("Baseline RidgeCV")
for _name, _outcome in BASELINE_ATTEMPTS:
    print(f"  {_name:<18} {_outcome}")

if BASELINE_MODEL is not None:
    BASELINE_TYPE = type(BASELINE_MODEL).__name__
    BASELINE_ALPHA = getattr(BASELINE_MODEL, "alpha_", None)
    BASELINE_N_FEATURES = getattr(BASELINE_MODEL, "n_features_in_", None)
    print(f"\n  type        : {BASELINE_TYPE}")
    if BASELINE_ALPHA is not None:
        print(f"  alpha_      : {BASELINE_ALPHA:g}   (Notebook 02 selected 0.01 by 3-fold CV)")
    if BASELINE_N_FEATURES is not None:
        print(f"  n_features_ : {BASELINE_N_FEATURES}   "
              f"(the 88-column matrix, NOT the 27 raw serving fields)")
else:
    BASELINE_TYPE = BASELINE_ALPHA = BASELINE_N_FEATURES = None
    print(f"\n  UNAVAILABLE — {BASELINE_ERROR}")

Baseline RidgeCV
  _from_local_file   not found
  _from_s3           not found
  _from_mlflow       loaded from runs:/60fdc0210519493aa5e23d1c77703819/model

  type        : RidgeCV
  alpha_      : 0.01   (Notebook 02 selected 0.01 by 3-fold CV)
  n_features_ : 88   (the 88-column matrix, NOT the 27 raw serving fields)


#### 6.7b Borrowing the champion's preprocessor

The tarball is resolved from the live endpoint by the same lineage walk as Section 4
(endpoint → config → model → model package → `ModelDataUrl`), so it survives a redeploy.
Override `CHAMPION_TARBALL_S3` or `CHAMPION_PIPELINE_LOCAL` to skip the lookup.

**On the scikit-learn version gap.** The champion pipeline was pickled inside
`sagemaker-scikit-learn:1.2-1-cpu-py3` — scikit-learn 1.2.1 — and this kernel runs a newer
one. A plain `joblib.load` therefore dies with
`AttributeError: Can't get attribute '__pyx_unpickle_CyHalfSquaredError'`. That symbol is
part of HistGradientBoosting's **loss object**, which belongs to the estimator this cell
discards two lines later; the preprocessing steps in front of it are ordinary transformers
that deserialize fine.

So the cell tries a normal load first and only falls back to a tolerant one, which lets an
unavailable class become a recorded stub instead of aborting the file. Loading scikit-learn
objects across versions is unsupported and the dangerous failure is silent — an attribute
that deserializes into something plausible and wrong — so 6.7b-verify checks the arithmetic
before anything is allowed to quote a price.

In [16]:
# =====================================================================================
# 6.7b  Borrowing the champion's preprocessor  (version-tolerant)
# -------------------------------------------------------------------------------------
# The baseline is a bare estimator: Notebook 02 fitted RidgeCV on the 88-column matrix,
# so it needs a transform to get there from 27 raw fields. The right one to use is the
# champion's, because then the two regression tabs differ in exactly one component.
#
# WHY THIS CELL IS MORE THAN A joblib.load. The champion pipeline was pickled inside
# the SageMaker container `sagemaker-scikit-learn:1.2-1-cpu-py3` -- scikit-learn 1.2.1.
# This kernel runs a newer scikit-learn, and the two disagree about a Cython symbol:
#
#     AttributeError: Can't get attribute '__pyx_unpickle_CyHalfSquaredError'
#
# That symbol belongs to HistGradientBoosting's LOSS OBJECT -- part of the estimator,
# which this cell throws away one line later. The preprocessing steps in front of it
# are ordinary scikit-learn transformers and unpickle fine. So the fix is to let the
# unavailable class deserialize as a stub rather than abort the whole file, and then
# drop the step that contained it.
#
# That is a real liberty, so this cell does not take it on trust: 6.7b-verify below
# un-scales a transformed row using the published scaler statistics and checks that
# known inputs come back. A cross-version unpickle that silently mangles a fitted
# attribute fails that check.
# =====================================================================================
import inspect
import tarfile
import types

import sklearn
from joblib.numpy_pickle import NumpyUnpickler

# --- OPTIONAL OVERRIDES ---------------------------------------------------------------
CHAMPION_TARBALL_S3     = None   # e.g. "s3://sagemaker-ap-southeast-1-0445.../model.tar.gz"
CHAMPION_PIPELINE_LOCAL = None   # e.g. "model_pipeline.joblib" if you already have it
# --------------------------------------------------------------------------------------

PREPROCESSOR, PREPROCESSOR_SOURCE, PREPROCESSOR_ERROR = None, None, None
PREPROCESSOR_STUBBED = []
FEATURE_COLUMNS = list(ARTIFACTS.get("feature_columns") or [])


def _tolerant_joblib_load(path):
    """joblib.load, except a class this environment no longer exposes deserializes as a
    stub. Every substitution is recorded in PREPROCESSOR_STUBBED so nothing is silent."""

    class _Tolerant(NumpyUnpickler):
        def find_class(self, module, name):
            try:
                return super().find_class(module, name)
            except (AttributeError, ImportError):
                PREPROCESSOR_STUBBED.append(f"{module}.{name}")

                def _placeholder(*args, **kwargs):
                    obj = types.SimpleNamespace()
                    obj.__setstate__ = lambda state: None
                    return obj

                _placeholder.__name__ = name
                return _placeholder

    # joblib >= 1.5 added a required ensure_native_byte_order argument.
    extra = {}
    if "ensure_native_byte_order" in inspect.signature(NumpyUnpickler.__init__).parameters:
        extra["ensure_native_byte_order"] = True

    with open(path, "rb") as raw:
        # joblib's own opener handles zlib/gzip/bz2/lzma. It was renamed between
        # releases, and the newer one yields a (fobj, mmap_mode) pair.
        utils = joblib.numpy_pickle_utils
        for attr, takes_mmap in (("_validate_fileobject_and_memmap", True),
                                 ("_read_fileobject", False)):
            opener = getattr(utils, attr, None)
            if opener is None:
                continue
            args = (raw, str(path), None) if takes_mmap else (raw, str(path))
            with opener(*args) as opened:
                fobj = opened[0] if isinstance(opened, tuple) else opened
                if isinstance(fobj, str):      # very old joblib on-disk format
                    return joblib.load(path)
                return _Tolerant(str(path), fobj, **extra).load()
        return _Tolerant(str(path), raw, **extra).load()


def _resolve_champion_tarball(endpoint_name=PRICE_ENDPOINT):
    """endpoint -> ... -> the s3 URI of model.tar.gz. Mirrors Section 4's lineage walk."""
    cfg = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointConfigName"]
    variants = sm.describe_endpoint_config(EndpointConfigName=cfg)["ProductionVariants"]
    model_desc = sm.describe_model(ModelName=variants[0]["ModelName"])
    containers = model_desc.get("Containers") or (
        [model_desc["PrimaryContainer"]] if "PrimaryContainer" in model_desc else [])
    for container in containers:
        if container.get("ModelDataUrl"):
            return container["ModelDataUrl"]
        # The Team14 endpoint takes this branch: the container names a model package
        # rather than an artifact, so the URI is one hop further in.
        if container.get("ModelPackageName"):
            pkg = sm.describe_model_package(ModelPackageName=container["ModelPackageName"])
            for pkg_container in pkg.get("InferenceSpecification", {}).get("Containers", []):
                if pkg_container.get("ModelDataUrl"):
                    return pkg_container["ModelDataUrl"]
    raise RuntimeError(f"no ModelDataUrl behind {endpoint_name}")


def _download_and_extract(s3_uri):
    """s3://bucket/key/model.tar.gz -> the directory it unpacked into."""
    bucket, key = s3_uri.replace("s3://", "", 1).split("/", 1)
    workdir = Path(tempfile.mkdtemp())
    tarball = workdir / "model.tar.gz"
    s3.download_file(bucket, key, str(tarball))
    with tarfile.open(tarball) as tf:
        # Refuse absolute paths and ../ escapes rather than trusting the archive.
        safe = [m for m in tf.getmembers()
                if not m.name.startswith(("/", "..")) and ".." not in Path(m.name).parts]
        tf.extractall(workdir / "unpacked", members=safe)
    unpacked = workdir / "unpacked"
    # The tarball ships pipeline_lib.py alongside the model. If the pickle references it
    # by module name, it has to be importable BEFORE the load, not after. src/ is already
    # first on the path from Section 5.1, so this is a fallback, not an override.
    if str(unpacked) not in sys.path:
        sys.path.append(str(unpacked))
    return unpacked


def _find_pipeline_file(root):
    exact = sorted(Path(root).rglob("model_pipeline.joblib"))
    return exact[0] if exact else next(iter(sorted(Path(root).rglob("*.joblib"))), None)


def _split_off_estimator(bundle):
    """Return (fitted transformer, description). The estimator is discarded."""
    from sklearn.pipeline import Pipeline

    if isinstance(bundle, dict):
        for k in ("pipeline", "model_pipeline", "model", "estimator"):
            if k in bundle:
                bundle = bundle[k]
                break

    if isinstance(bundle, Pipeline) and len(bundle.steps) >= 2:
        return bundle[:-1], f"Pipeline[:-1] — dropped final step '{bundle.steps[-1][0]}'"

    for attr in ("preprocessor", "preprocess", "prep", "features", "transformer",
                 "feature_pipeline", "column_transformer"):
        candidate = getattr(bundle, attr, None)
        if candidate is not None and hasattr(candidate, "transform"):
            return candidate, f"{type(bundle).__name__}.{attr}"

    if isinstance(bundle, Pipeline):
        for name, step in bundle.steps:
            if hasattr(step, "transform") and not hasattr(step, "predict"):
                return step, f"Pipeline step '{name}'"

    raise TypeError(
        f"cannot separate preprocessing from prediction in a {type(bundle).__name__}. "
        "Set CHAMPION_PIPELINE_LOCAL to a bundle whose preprocessing is reachable, or "
        "expose a transform-only object from src/pipeline_lib.py.")


try:
    if CHAMPION_PIPELINE_LOCAL and os.path.isfile(CHAMPION_PIPELINE_LOCAL):
        _path = Path(CHAMPION_PIPELINE_LOCAL)
        _origin = os.path.abspath(CHAMPION_PIPELINE_LOCAL)
    else:
        _uri = CHAMPION_TARBALL_S3 or _resolve_champion_tarball()
        _path = _find_pipeline_file(_download_and_extract(_uri))
        if _path is None:
            raise FileNotFoundError(f"no .joblib inside {_uri}")
        _origin = f"{_uri} :: {_path.name}"

    # Straight load first. The tolerant path is a fallback, not the default -- if the two
    # environments happen to agree, nothing is stubbed and there is nothing to justify.
    try:
        _bundle = joblib.load(_path)
        _load_note = "joblib.load"
    except (AttributeError, ImportError, ModuleNotFoundError):
        _bundle = _tolerant_joblib_load(_path)
        _load_note = f"tolerant load, stubbed {len(set(PREPROCESSOR_STUBBED))} unavailable class(es)"

    PREPROCESSOR, _how = _split_off_estimator(_bundle)
    PREPROCESSOR_SOURCE = f"{_origin}  ({_how}; {_load_note})"
except Exception as _e:
    PREPROCESSOR_ERROR = f"{type(_e).__name__}: {_e}"


def raw_to_features(payloads):
    """One or more raw SERVING_COLUMNS dicts -> the 88-column float32 matrix.

    The same transform the endpoint runs, so the only thing that differs between this
    tab and the champion tab is the estimator on the end of it.
    """
    if PREPROCESSOR is None:
        raise RuntimeError(f"no preprocessor available ({PREPROCESSOR_ERROR})")

    rows = payloads if isinstance(payloads, list) else [payloads]
    raw = pd.DataFrame(rows).reindex(columns=list(SERVING_COLUMNS))
    out = PREPROCESSOR.transform(raw)

    if not isinstance(out, pd.DataFrame):
        names = FEATURE_COLUMNS or list(getattr(BASELINE_MODEL, "feature_names_in_", []))
        if len(names) != out.shape[1]:
            raise ValueError(
                f"transform produced {out.shape[1]} columns but the feature contract "
                f"names {len(names)}. preprocessing_artifacts.json and the deployed "
                "pipeline disagree — do not trust a prediction built on this.")
        out = pd.DataFrame(out, columns=names)

    # Column ORDER is part of the contract for a linear model: a permuted matrix still
    # predicts, it just predicts nonsense. Reindex against what the estimator was fitted
    # on rather than assuming the two orders happen to coincide.
    expected = list(getattr(BASELINE_MODEL, "feature_names_in_", [])) or FEATURE_COLUMNS
    if expected:
        missing = [c for c in expected if c not in out.columns]
        if missing:
            raise ValueError(f"transform is missing {len(missing)} fitted columns, "
                             f"first few: {missing[:5]}")
        out = out[expected]

    return out.astype(np.float32)   # Notebook 02 trained on float32; match it exactly


print("Baseline preprocessor")
print(f"  local scikit-learn : {sklearn.__version__}   "
      f"(champion pickled under 1.2.1, in sagemaker-scikit-learn:1.2-1)")
if PREPROCESSOR is not None:
    print(f"  source : {PREPROCESSOR_SOURCE}")
    print(f"  target : {len(FEATURE_COLUMNS)} feature columns from {ARTIFACTS_SOURCE}")
    if PREPROCESSOR_STUBBED:
        print("\n  Stubbed during unpickling — every one of these should belong to the")
        print("  DISCARDED estimator, not to the preprocessing:")
        for _s in dict.fromkeys(PREPROCESSOR_STUBBED):
            print(f"    {_s}")
        print("  The next cell checks the transform arithmetic, which is what decides")
        print("  whether that assumption held.")
else:
    print(f"  UNAVAILABLE — {PREPROCESSOR_ERROR}")
    print("  The Baseline tab will load and explain this instead of predicting.")

Baseline preprocessor
  local scikit-learn : 1.7.2   (champion pickled under 1.2.1, in sagemaker-scikit-learn:1.2-1)
  source : s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-08-23-09-36-05-368/pipelines-5lib4dgrbi1f-RegisterModel-Repack-pRp8VFqHyN/output/model.tar.gz :: model_pipeline.joblib  (Pipeline[:-1] — dropped final step 'model'; tolerant load, stubbed 1 unavailable class(es))
  target : 88 feature columns from processed/preprocessing_artifacts.json

  Stubbed during unpickling — every one of these should belong to the
  DISCARDED estimator, not to the preprocessing:
    sklearn._loss._loss.__pyx_unpickle_CyHalfSquaredError
  The next cell checks the transform arithmetic, which is what decides
  whether that assumption held.


/tmp/ipykernel_1164/1709613743.py:114: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(workdir / "unpacked", members=safe)
/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator HistGradientBoostingRegressor from version 1.2.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persis

#### 6.7b-verify Proving the borrowed transform, rather than hoping

Ground truth comes from `preprocessing_artifacts.json` — the fitted state Notebook 03
asserts the deployed pipeline against, and nothing to do with the pickle. A listing with
known values goes through the transform, the scaling is inverted with the published
mean/scale, and the original values have to come back.

Only the exact-arithmetic checks can veto. If they fail the tab is **disabled rather than
left quoting a number nobody can defend**, and the cell prints the three ways to fix it,
starting with matching the container's scikit-learn.

In [17]:
# =====================================================================================
# 6.7b-verify  Is the borrowed transform actually correct?
# -------------------------------------------------------------------------------------
# Loading a scikit-learn object pickled by a different version is not supported, and the
# failure mode that matters is not a crash -- it is a fitted attribute that deserializes
# into something plausible but wrong. A scaler with the right shape and the wrong mean_
# produces predictions all day and every one of them is nonsense.
#
# So this cell does not ask "did it load". It asks "does it compute the right numbers",
# using a source of truth that has nothing to do with the pickle:
# preprocessing_artifacts.json, the fitted state Notebook 03 asserts the deployed
# pipeline against. Notebook 02 section 9.6 un-scales features the same way.
#
# The check: push a listing with known values through the transform, invert the scaling
# with the published mean/scale, and see whether the original values come back.
#
# Only checks that are EXACT ARITHMETIC against published ground truth may disable the
# tab. The last one is a heuristic smoke test, so it warns and does not veto -- a
# heuristic that switches off a working transform is worse than no heuristic.
# =====================================================================================
PREPROC_VERIFIED, PREPROC_VERIFY_REPORT = False, []

_SC = (ARTIFACTS.get("scaler") or {})
_probe = dict(paris_listing, accommodates=3, bedrooms=2, minimum_nights=5,
              latitude=48.8867, longitude=2.3431)

if PREPROCESSOR is None:
    PREPROC_VERIFY_REPORT.append(("preprocessor", "SKIP", "not loaded"))
elif not _SC.get("columns"):
    PREPROC_VERIFY_REPORT.append(
        ("scaler stats", "SKIP",
         "preprocessing_artifacts.json has no scaler block — cannot verify arithmetic"))
else:
    _s_idx = {c: i for i, c in enumerate(_SC["columns"])}

    try:
        _X = raw_to_features(_probe)

        # 1. Shape and the feature contract.
        _PREPROC_OK_SHAPE = _X.shape == (1, len(FEATURE_COLUMNS))
        PREPROC_VERIFY_REPORT.append(
            ("shape", "PASS" if _PREPROC_OK_SHAPE else "FAIL",
             f"{_X.shape} vs expected (1, {len(FEATURE_COLUMNS)})"))

        # 2. No NaNs. The endpoint asserts this too; a stubbed imputer would show here.
        _n_nan = int(_X.isna().sum().sum())
        PREPROC_VERIFY_REPORT.append(
            ("no NaNs", "PASS" if _n_nan == 0 else "FAIL", f"{_n_nan} NaN cells"))

        # 3. THE ONE THAT MATTERS. Un-scale and compare against what went in.
        #    accommodates and minimum_nights pass through the pipeline unchanged except
        #    for standardisation, so their round trip is exact arithmetic, not a guess.
        for _col, _expected in (("accommodates", 3.0), ("minimum_nights", 5.0)):
            if _col not in _X.columns or _col not in _s_idx:
                PREPROC_VERIFY_REPORT.append((f"round-trip {_col}", "SKIP", "column absent"))
                continue
            _got = (float(_X[_col].iloc[0]) * _SC["scale"][_s_idx[_col]]
                    + _SC["mean"][_s_idx[_col]])
            _ok = abs(_got - _expected) < 0.01
            PREPROC_VERIFY_REPORT.append(
                (f"round-trip {_col}", "PASS" if _ok else "FAIL",
                 f"un-scaled to {_got:.3f}, sent {_expected:.1f}"))

        # 4. The city one-hot fires on the right column and nowhere else.
        _city_cols = [c for c in _X.columns if c.startswith("city_")]
        if _city_cols:
            _hot = [c for c in _city_cols if float(_X[c].iloc[0]) > 0.5]
            _ok = _hot == [f"city_{_probe['city'].replace(' ', '_')}"] or (
                len(_hot) == 1 and _probe["city"].replace(" ", "_") in _hot[0])
            PREPROC_VERIFY_REPORT.append(
                ("city one-hot", "PASS" if _ok else "FAIL",
                 f"{_hot or 'nothing'} set for {_probe['city']}"))

        # 5. The transform responds to its input. A stub that returns constants would
        #    sail through everything above; two different listings must not agree.
        _X2 = raw_to_features(dict(_probe, city="Bangkok", accommodates=8,
                                   latitude=13.7230, longitude=100.5680))
        _differs = int((_X.values != _X2.values).sum())
        PREPROC_VERIFY_REPORT.append(
            ("responds to input", "PASS" if _differs >= 2 else "WARN",
             f"{_differs} of {_X.shape[1]} columns differ between two listings"))

    except Exception as _e:
        PREPROC_VERIFY_REPORT.append(("transform", "FAIL", f"{type(_e).__name__}: {_e}"))

_results = [s for _, s, _ in PREPROC_VERIFY_REPORT]
PREPROC_VERIFIED = bool(_results) and "FAIL" not in _results and "PASS" in _results

print("Transform verification")
for _name, _status, _detail in PREPROC_VERIFY_REPORT:
    print(f"  [{_status:<4}] {_name:<20} {_detail}")

if PREPROC_VERIFIED:
    print("\n  The borrowed transform reproduces the published scaler statistics, so the "
          "\n  cross-version unpickle recovered the preprocessing intact. Baseline "
          "\n  predictions built on it are trustworthy.")
elif "FAIL" in _results:
    # Better a dead tab than a tab quoting a number nobody can defend.
    PREPROCESSOR = None
    PREPROCESSOR_ERROR = ("the borrowed transform failed verification — see 6.7b-verify. "
                          "The cross-version unpickle did not recover the preprocessing "
                          "correctly, so the Baseline tab has been disabled rather than "
                          "left quoting an indefensible price.")
    print("\n  DISABLED. The transform loaded but does not compute the right numbers,")
    print("  which is the dangerous case: it would have predicted happily and been wrong.")
    print("  Options, in order of preference:")
    print("    1. Match the container's scikit-learn:  %pip install scikit-learn==1.2.1")
    print("       then restart the kernel. This is the only fix with no caveats.")
    print("    2. Re-export the preprocessor from Notebook 03 under this kernel's")
    print("       scikit-learn and point CHAMPION_PIPELINE_LOCAL at it.")
    print("    3. Expose a transform function from src/pipeline_lib.py and call it in")
    print("       raw_to_features instead of PREPROCESSOR.transform.")
else:
    print("\n  Not verified (checks were skipped). The tab will still run, but treat its")
    print("  numbers as unconfirmed until preprocessing_artifacts.json is readable.")

Transform verification
  [PASS] shape                (1, 88) vs expected (1, 88)
  [PASS] no NaNs              0 NaN cells
  [PASS] round-trip accommodates un-scaled to 3.000, sent 3.0
  [SKIP] round-trip minimum_nights column absent
  [PASS] city one-hot         ['city_Paris'] set for Paris
  [PASS] responds to input    8 of 88 columns differ between two listings

  The borrowed transform reproduces the published scaler statistics, so the 
  cross-version unpickle recovered the preprocessing intact. Baseline 
  predictions built on it are trustworthy.


#### 6.7c The handler

Same four returns as `predict_airbnb_price`, same `build_payload`, same `validate`, same
derived-features table. With the comparison box ticked it also posts the identical payload to
the champion endpoint, so the two answers appear on one card.

In [18]:
# =====================================================================================
# 6.7c  The baseline handler
# -------------------------------------------------------------------------------------
# Same four returns as `predict_airbnb_price`, same payload builder, same validation,
# same derived-features table. Two things differ, and both are the point of the tab:
# there is no round trip (the estimator is in this kernel), and the result card can
# put the champion's answer next to its own.
# =====================================================================================

# Notebook 02's own numbers, quoted as it reported them. The splits are NOT the same --
# see the note in the card. Ridge: section 4, validation. Champion: section 8B, test.
BASELINE_SCORES = {"label": "RidgeCV (alpha=0.01)", "mae_log": 0.4255, "rmse_log": 0.5768,
                   "r2_log": 0.7877, "mape": 46.84, "split": "validation, 44,679 rows",
                   "trained_on": "178,715 rows (the fit half)"}
CHAMPION_SCORES = {"label": "HistGradientBoosting", "mae_log": 0.3207, "rmse_log": 0.4510,
                   "r2_log": 0.8708, "mape": 33.41, "split": "test, 55,849 rows",
                   "trained_on": "223,394 rows (full train)"}

BASELINE_UI_FIELDS = [*UI_FIELDS, "compare_with_champion"]


def baseline_predict_raw(payload):
    """One raw listing -> the record shape the price endpoint returns. (record, ms)."""
    t0 = time.perf_counter()
    X = raw_to_features(payload)
    log_price = float(np.asarray(BASELINE_MODEL.predict(X)).ravel()[0])
    ms = (time.perf_counter() - t0) * 1000.0
    city = payload["city"]
    return {
        "suggested_nightly_price": round(float(np.expm1(log_price)), 2),
        "currency": CURRENCY.get(city, ""),
        "city": city,
        "log_price": round(log_price, 4),
        "model": BASELINE_TYPE,
        "served_by": "in-process (no endpoint)",
    }, ms


def _unavailable_card():
    reasons = []
    if BASELINE_MODEL is None:
        reasons.append(f"- **Estimator**: {BASELINE_ERROR}")
    if PREPROCESSOR is None:
        reasons.append(f"- **Preprocessor**: {PREPROCESSOR_ERROR}")
    attempts = "\n".join(f"  - `{n}` -> {o}" for n, o in BASELINE_ATTEMPTS)
    return ("### Baseline unavailable\n\n" + "\n".join(reasons) +
            "\n\nLoad attempts:\n" + attempts +
            "\n\nFix it in Section 6.7a/6.7b: point `BASELINE_S3_KEY` at your mirror, add a "
            "path to `BASELINE_LOCAL_CANDIDATES`, or check that `BASELINE_RUN_ID` and "
            "`MLFLOW_TRACKING_URI` match the run that logged the baseline. The other three "
            "tabs are unaffected.")


def _comparison_block(baseline_rec, champion_rec, champion_error):
    if champion_error:
        return (f"<div class='cmp'><div class='cmp-head'>Champion comparison unavailable</div>"
                f"<div class='cmp-note'>{champion_error}</div></div>")

    b, ch = baseline_rec["suggested_nightly_price"], champion_rec["suggested_nightly_price"]
    ccy = champion_rec["currency"]
    delta = b - ch
    pct = (delta / ch * 100.0) if ch else float("nan")
    arrow = "higher than" if delta > 0 else ("lower than" if delta < 0 else "identical to")

    rows = "".join(
        f"<tr><td class='id'>{s['label']}</td><td>{p:,.2f}</td><td>{s['mae_log']:.4f}</td>"
        f"<td>{s['r2_log']:.4f}</td><td>{s['mape']:.1f}%</td></tr>"
        for s, p in ((BASELINE_SCORES, b), (CHAMPION_SCORES, ch)))

    return (
        "<div class='cmp'>"
        "<div class='cmp-head'>Same 88 columns, two estimators</div>"
        "<table class='rank'><thead><tr><th>model</th><th>rate</th><th>MAE (log)</th>"
        "<th>R&sup2; (log)</th><th>MAPE</th></tr></thead>"
        f"<tbody>{rows}</tbody></table>"
        f"<div class='cmp-note'>The baseline asks <strong>{abs(delta):,.2f} {ccy} "
        f"({abs(pct):.1f}%)</strong> {arrow} the champion for this listing. "
        "The metric columns are quoted from Notebook 02 and are <em>not</em> measured on "
        "the same split — Ridge on validation, champion on test — so read the ranking, "
        "not the decimal places.</div>"
        "</div>")


def predict_airbnb_price_baseline(*values):
    # Gradio click handler. Returns (result card, derived table, request JSON, response JSON).
    ui = dict(zip(BASELINE_UI_FIELDS, values))
    compare = bool(ui.pop("compare_with_champion", False))

    if BASELINE_MODEL is None or PREPROCESSOR is None:
        return _unavailable_card(), "", "", ""

    try:
        payload = build_payload(ui)
    except (TypeError, ValueError) as e:
        return (f"### Cannot build the request\n\n{e}\n\nCheck the numeric fields.", "", "", "")

    request_json = json.dumps(payload, indent=2)

    problems = validate(payload)
    if problems:
        bullets = "\n".join(f"- {p}" for p in problems)
        return (f"### Fix these before sending\n\n{bullets}", "", request_json, "")

    try:
        record, ms = baseline_predict_raw(payload)
    except Exception as e:
        return (f"### Local inference failed\n\n`{type(e).__name__}`\n\n```\n{e}\n```\n\n"
                "This is a transform problem, not an endpoint problem — the estimator never "
                "saw a payload. Check that `preprocessing_artifacts.json` and the deployed "
                "pipeline agree on the 88 columns.",
                derived_preview(payload), request_json, "")

    price, ccy = record["suggested_nightly_price"], record["currency"]
    response_records = [record]

    band = (f"<div class='band'>Notebook 02 measured this model at "
            f"<strong>&plusmn;{BASELINE_SCORES['mape']:.1f}%</strong> MAPE across all ten "
            f"markets ({BASELINE_SCORES['split']}), so read this as roughly "
            f"<strong>{price * (1 - BASELINE_SCORES['mape'] / 100):,.0f}&ndash;"
            f"{price * (1 + BASELINE_SCORES['mape'] / 100):,.0f} {ccy}</strong>. "
            f"That band is wider than the champion's for the same listing, which is the "
            f"whole reason the champion was promoted.</div>")

    cmp_html = ""
    if compare:
        champion_error = None
        try:
            champion_records, champion_ms = invoke_airbnb_endpoint(payload)
            champion_rec = champion_records[0]
            champion_rec["served_by"] = f"{PRICE_ENDPOINT} ({champion_ms:,.0f} ms)"
            response_records.append(champion_rec)
        except Exception as e:
            champion_rec, champion_error = None, f"{type(e).__name__}: {e}"
        cmp_html = _comparison_block(record, champion_rec, champion_error)

    card = (
        "<div class='result'>"
        f"<div class='eyebrow'>Baseline suggested nightly rate &middot; {record['city']}</div>"
        f"<div class='price'>{price:,.2f} <span class='ccy'>{ccy}</span></div>"
        f"{band}{cmp_html}"
        f"<div class='meta'>log_price {record['log_price']} &middot; {ms:,.1f} ms local "
        f"inference, no network &middot; {BASELINE_TYPE} from {BASELINE_SOURCE}</div>"
        "</div>")

    derived = derived_preview(payload).replace(
        "Derived inside the endpoint",
        "Derived in this notebook by the champion's fitted preprocessor")

    return card, derived, request_json, json.dumps(response_records, indent=2)


# --- Smoke test: the two Section 5.2 listings, scored by both models ------------------
if BASELINE_MODEL is not None and PREPROCESSOR is not None:
    try:
        for _name, _listing in [("paris_listing", paris_listing),
                                ("bangkok_listing", bangkok_listing)]:
            _rec, _ms = baseline_predict_raw(_listing)
            print(f"{_rec['city']:<10s} baseline {_rec['suggested_nightly_price']:>10,.2f} "
                  f"{_rec['currency']}   (log_price {_rec['log_price']})   {_ms:.1f} ms")
        print("\nBaseline tab ready. Tick 'Compare against the champion endpoint' in the UI "
              "to put the two side by side on the same payload.")
    except Exception as _e:
        print(f"Smoke test failed: {type(_e).__name__}: {_e}")
else:
    print("Baseline tab will render its diagnostic instead of a prediction — see above.")

Paris      baseline     166.28 EUR   (log_price 5.1197)   39.7 ms


Bangkok    baseline   6,896.75 THB   (log_price 8.839)   35.2 ms

Baseline tab ready. Tick 'Compare against the champion endpoint' in the UI to put the two side by side on the same payload.


### 6.8 The fifth model: the LogisticRegression baseline, scored in-process

Section 6.7 did this for the *regression* side. This section does it for the
*classification* side, and the two are not the same job.

**Where the baseline actually lives.** `02_baseline_experiments_sagemaker_mlflow_app`
fits `LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000)` on
`X_train`, and persists it with a single line — `mlflow.sklearn.log_model(model_lr,
artifact_path="model", ...)` — inside the run `team14_s1401_logistic_regression_baseline`
(`5449338e…`). That writes it to the SageMaker MLflow App and nowhere else. Notebook 02's
only export step writes `best_model.json`, which points at the **tuned Random Forest**
(`6f91c874…`, test ROC-AUC 0.8383) — the champion behind
`iti113-team14-airbnb-instant-booking`. There is no S3 copy of the baseline and no local
`.joblib`. So 6.8a tries a local file and an S3 key first — both are placeholders you can
point at your own mirror — before falling back to the run URI that actually exists today.

**Notebook 02 contains no preprocessing to borrow.** This is the part that decides the
whole design of the tab, and it is worth being blunt about. Notebook 02's data-loading
section is four `load_s3_csv` calls: it reads `train_features.csv`, the **100-column**
matrix Notebook 01 had already produced, and fits LogisticRegression directly on it. The
estimator has never seen a city name, an amenities JSON string or a `host_since` date.
Asking it to score a raw listing is a `ValueError` at best, and a confident wrong number
at worst.

So something has to bridge 26 raw fields → 100 engineered columns, and the choice
determines what this tab actually measures. Reimplementing the engineering here would
mean the probability gap between this tab and the champion tab is measuring *my transform
code*, not the two models. Borrowing the champion classifier's own transform means the
two tabs differ in exactly one component — the estimator — which is the comparison the tab
claims to be showing. 6.8b does the latter.

**Why this pair is more informative than the Ridge pair.** The two regression tabs quote
metrics from different splits, so the card there has to caveat the comparison. Here both
models were evaluated on the same 55,629-row test split from the same Notebook 01
partition:

| | Baseline LogReg | Champion RF (tuned) | Δ |
|---|---:|---:|---:|
| ROC-AUC | 0.7612 | 0.8383 | **+0.0771** |
| PR-AUC | 0.6895 | 0.8111 | +0.1216 |
| Accuracy | 0.7049 | 0.7657 | +0.0608 |
| F1 | 0.6069 | 0.6841 | +0.0772 |
| Precision | 0.6726 | 0.7698 | +0.0972 |
| Recall | 0.5529 | 0.6155 | +0.0626 |
| Log-loss | 0.5708 | 0.4873 | −0.0835 |
| Brier | 0.1936 | 0.1605 | −0.0331 |

That last row is the one worth arguing about in a review. The champion does not just
label better, it is better *calibrated* — which is what makes its probability, and not
merely its label, safe to act on.

**One caveat the tab is honest about.** Notebook 02's logistic regression emitted a
`ConvergenceWarning`: lbfgs stopped at the 1,000-iteration limit rather than at a
solution, and suggested scaling the data. The coefficients are still exactly the ones
that scored 0.7612, so the comparison is not rigged — but this is a baseline that was
never given its best shot, and that flatters the champion slightly. The result card says
so rather than leaving it in a warning nobody reads.

#### 6.8a Loading the estimator

Three sources, cheapest first, none of them fatal. A baseline that cannot be loaded has to leave the other four tabs working and say why in its own tab — the same degradation rule the endpoint tabs follow in Section 2.

In [19]:
# =====================================================================================
# 6.8a  Load the baseline LogisticRegression classifier
# -------------------------------------------------------------------------------------
# WHERE IT ACTUALLY LIVES. Notebook 02 (the classification notebook, target
# `instant_bookable`) fits
#
#     LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000,
#                        random_state=42).fit(X_train, y_train)
#
# in the cell headed "RUN 1: LOGISTIC REGRESSION BASELINE", and persists it with one
# line -- `mlflow.sklearn.log_model(sk_model=model_lr, artifact_path="model", ...)` --
# inside the run named `team14_s1401_logistic_regression_baseline`.
#
# That writes it to the SageMaker MLflow App and NOWHERE ELSE. There is no S3 copy and
# no local .joblib: Notebook 02's only export step writes `best_model.json`, which
# points at the TUNED RANDOM FOREST (run 6f91c874..., test ROC-AUC 0.8383), not at this
# baseline. So sources 1 and 2 below are placeholders for a mirror you can create with
# Section 12's publish cell; source 3 is the one that exists today.
#
# Three sources are tried, cheapest first. Nothing here raises: a baseline that cannot
# be loaded must leave the other four tabs working, and must say why in its own tab.
# =====================================================================================
import importlib
import tempfile

import joblib
import numpy as np
import pandas as pd

# --- EDIT THESE IF YOUR LINEAGE DIFFERS ----------------------------------------------
LOGREG_RUN_ID    = "5449338e75254622a4d50c70beec773b"   # team14_s1401_logistic_regression_baseline
LOGREG_RUN_NAME  = "team14_s1401_logistic_regression_baseline"
LOGREG_STUDENT   = "s1401"          # Notebook 02 ran as s1401; this notebook's default is s1402
BOOKING_PROJECT  = "airbnb-instant-booking"
BOOKING_PREFIX   = f"iti113/{TEAM_ID}/data/{BOOKING_PROJECT}"

BOOKING_MLFLOW_TRACKING_URI = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS")
BOOKING_MLFLOW_EXPERIMENT   = f"{COURSE}/{TEAM_ID}/InstantBooking-Classification"

# Source 2 placeholder: the key Section 12's publish cell writes to.
LOGREG_S3_BUCKET = BUCKET
LOGREG_S3_KEY    = f"{BOOKING_PREFIX}/artifacts/baseline/logreg_baseline.joblib"

# Source 1 placeholder: local files, checked in this order.
LOGREG_LOCAL_CANDIDATES = [
    "logreg_baseline.joblib",
    os.path.join("artifacts", "logreg_baseline.joblib"),
    os.path.join("downloaded_model", "logreg_baseline.joblib"),
    "logreg_baseline.pkl",
]

# The 100-column contract, copied verbatim from Notebook 02's own printed output
# ("Number of features: 100" / the numbered feature list). This is INDEPENDENT ground
# truth: it did not come out of the pickle, so comparing it against the loaded
# estimator's feature_names_in_ catches loading a different model by mistake.
LOGREG_FEATURE_CONTRACT = [
    "host_response_rate", "host_acceptance_rate", "host_is_superhost",
    "host_total_listings_count", "host_has_profile_pic", "host_identity_verified",
    "latitude", "longitude", "accommodates", "bedrooms", "minimum_nights",
    "maximum_nights", "review_scores_rating", "review_scores_accuracy",
    "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication",
    "review_scores_location", "review_scores_value",
    "host_response_rate_isna", "host_acceptance_rate_isna", "host_response_time_isna",
    "review_scores_rating_isna", "review_scores_accuracy_isna",
    "review_scores_cleanliness_isna", "review_scores_checkin_isna",
    "review_scores_communication_isna", "review_scores_location_isna",
    "review_scores_value_isna", "bedrooms_isna",
    "amenity_count", "has_self_checkin", "has_keypad", "has_smart_lock", "has_lockbox",
    "has_building_staff", "has_keyless_entry_composite", "host_tenure_days",
    "is_superhost_verified", "accommodates_per_bedroom", "is_single_night",
    "is_extended_stay", "superhost_x_single_night", "dist_to_center_km",
    "amenity_wifi", "amenity_essentials", "amenity_kitchen",
    "amenity_long_term_stays_allowed", "amenity_hangers", "amenity_hair_dryer",
    "amenity_iron", "amenity_washer", "amenity_heating", "amenity_dedicated_workspace",
    "amenity_shampoo", "amenity_hot_water", "amenity_smoke_alarm",
    "amenity_air_conditioning", "amenity_dishes_and_silverware", "amenity_refrigerator",
    "amenity_cooking_basics", "amenity_elevator", "amenity_microwave",
    "amenity_bed_linens", "amenity_fire_extinguisher", "amenity_dryer", "amenity_stove",
    "amenity_coffee_maker", "amenity_oven", "amenity_first_aid_kit",
    "amenity_carbon_monoxide_alarm", "amenity_free_parking_on_premises",
    "amenity_private_entrance", "amenity_extra_pillows_and_blankets",
    "log_price_city_zscore",
    "city_Cape Town", "city_Hong Kong", "city_Istanbul", "city_Mexico City",
    "city_New York", "city_Paris", "city_Rio de Janeiro", "city_Rome", "city_Sydney",
    "room_type_Hotel room", "room_type_Private room", "room_type_Shared room",
    "property_type_grouped_Entire condominium", "property_type_grouped_Entire house",
    "property_type_grouped_Entire loft",
    "property_type_grouped_Entire serviced apartment", "property_type_grouped_Other",
    "property_type_grouped_Private room in apartment",
    "property_type_grouped_Private room in bed and breakfast",
    "property_type_grouped_Private room in condominium",
    "property_type_grouped_Private room in house",
    "property_type_grouped_Room in boutique hotel",
    "host_response_time_within a day", "host_response_time_within a few hours",
    "host_response_time_within an hour",
]
# --------------------------------------------------------------------------------------

LOGREG_MODEL, LOGREG_SOURCE, LOGREG_ERROR = None, None, None
LOGREG_ATTEMPTS = []


def _logreg_try(fn):
    """Run one loading strategy. Record what happened; never propagate."""
    global LOGREG_MODEL, LOGREG_SOURCE
    if LOGREG_MODEL is not None:
        return
    try:
        result = fn()
    except Exception as e:
        LOGREG_ATTEMPTS.append((fn.__name__, f"{type(e).__name__}: {e}"))
        return
    if result is None:
        LOGREG_ATTEMPTS.append((fn.__name__, "not found"))
        return
    LOGREG_MODEL, LOGREG_SOURCE = result
    LOGREG_ATTEMPTS.append((fn.__name__, f"loaded from {LOGREG_SOURCE}"))


def _logreg_from_local_file():
    for p in LOGREG_LOCAL_CANDIDATES:
        if os.path.isfile(p):
            return joblib.load(p), os.path.abspath(p)
    return None


def _logreg_from_s3():
    # head_object first: a 404 here is the expected case today, not an error worth a
    # traceback in the middle of a demo.
    try:
        s3.head_object(Bucket=LOGREG_S3_BUCKET, Key=LOGREG_S3_KEY)
    except Exception:
        return None
    dst = Path(tempfile.mkdtemp()) / Path(LOGREG_S3_KEY).name
    s3.download_file(LOGREG_S3_BUCKET, LOGREG_S3_KEY, str(dst))
    return joblib.load(dst), f"s3://{LOGREG_S3_BUCKET}/{LOGREG_S3_KEY}"


def _logreg_from_mlflow():
    # The one that actually exists today. Notebook 02 used artifact_path="model", so the
    # run URI ends in /model. The sklearn flavour is the right loader; pyfunc is the
    # fallback for a differently-serialised run.
    import mlflow
    mlflow.set_tracking_uri(BOOKING_MLFLOW_TRACKING_URI)
    uri = f"runs:/{LOGREG_RUN_ID}/model"
    try:
        import mlflow.sklearn
        return mlflow.sklearn.load_model(uri), uri
    except Exception:
        return mlflow.pyfunc.load_model(uri), f"{uri} (pyfunc wrapper)"


for _strategy in (_logreg_from_local_file, _logreg_from_s3, _logreg_from_mlflow):
    _logreg_try(_strategy)

if LOGREG_MODEL is None:
    LOGREG_ERROR = ("no source yielded a model - see the attempts below, then either set "
                    "LOGREG_S3_KEY / LOGREG_LOCAL_CANDIDATES, or run "
                    "Section 12's publish cell, which mirrors it to S3 from this notebook")

print("Baseline LogisticRegression (target: instant_bookable)")
for _name, _outcome in LOGREG_ATTEMPTS:
    print(f"  {_name:<24} {_outcome}")

if LOGREG_MODEL is not None:
    LOGREG_TYPE = type(LOGREG_MODEL).__name__
    LOGREG_N_FEATURES = getattr(LOGREG_MODEL, "n_features_in_", None)
    LOGREG_FITTED_COLUMNS = list(getattr(LOGREG_MODEL, "feature_names_in_", []))
    LOGREG_N_ITER = getattr(LOGREG_MODEL, "n_iter_", None)
    print(f"\n  type        : {LOGREG_TYPE}")
    print(f"  C / penalty : {getattr(LOGREG_MODEL, 'C', '?')} / "
          f"{getattr(LOGREG_MODEL, 'penalty', '?')}   (Notebook 02: C=1.0, l2, lbfgs)")
    if LOGREG_N_FEATURES is not None:
        print(f"  n_features_ : {LOGREG_N_FEATURES}   "
              f"(the 100-column engineered matrix, NOT the 26 raw booking fields)")
    if LOGREG_N_ITER is not None:
        _iters = int(np.asarray(LOGREG_N_ITER).ravel()[0])
        _converged = _iters < int(getattr(LOGREG_MODEL, "max_iter", 1000))
        print(f"  n_iter_     : {_iters} of max_iter="
              f"{getattr(LOGREG_MODEL, 'max_iter', '?')}"
              f"{'' if _converged else '   <-- DID NOT CONVERGE (see the note below)'}")
        if not _converged:
            print("\n  Notebook 02's run emitted a ConvergenceWarning: lbfgs stopped at the")
            print("  iteration limit rather than at a solution. The coefficients are still")
            print("  a legitimate baseline - they are what Notebook 02 scored 0.7612 test")
            print("  ROC-AUC with - but they are not the L2 optimum, and the tab says so.")
else:
    LOGREG_TYPE = LOGREG_N_FEATURES = LOGREG_N_ITER = None
    LOGREG_FITTED_COLUMNS = []
    print(f"\n  UNAVAILABLE - {LOGREG_ERROR}")


Baseline LogisticRegression (target: instant_bookable)
  _logreg_from_local_file  not found
  _logreg_from_s3          not found
  _logreg_from_mlflow      loaded from runs:/5449338e75254622a4d50c70beec773b/model

  type        : LogisticRegression
  C / penalty : 1.0 / l2   (Notebook 02: C=1.0, l2, lbfgs)
  n_features_ : 100   (the 100-column engineered matrix, NOT the 26 raw booking fields)
  n_iter_     : 1000 of max_iter=1000   <-- DID NOT CONVERGE (see the note below)

  Notebook 02's run emitted a ConvergenceWarning: lbfgs stopped at the
  iteration limit rather than at a solution. The coefficients are still
  a legitimate baseline - they are what Notebook 02 scored 0.7612 test
  ROC-AUC with - but they are not the L2 optimum, and the tab says so.


#### 6.8b Notebook 01's transform, implemented here

This replaces the earlier approach of borrowing the booking endpoint's preprocessing.
That produced `P(instant) = 0.0%` — a saturated sigmoid, which is what a linear model
returns when the feature vector is nowhere near the data it was fitted on.

The cause is Notebook 01 section 6.6:

```python
scaler = StandardScaler()
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
```

**Nineteen continuous columns are standardised.** `maximum_nights` is not 1125 in the
training matrix, it's about +0.4. `host_tenure_days` is not 2,847, it's about −0.2. Feed
raw magnitudes into coefficients fitted on z-scores and the linear predictor goes to a
large negative number; `expit()` of that is 0.0 to four decimal places.

Every reconciliation rule in the previous version was operating *downstream* of that,
which is why none of them caught it — the columns were all present, correctly named,
correctly ordered, and every value was wrong by one standardisation. No amount of
patching turns a borrowed transform into Notebook 01's transform, so this cell implements
Notebook 01 sections 4.1–4.5, 5 and 6.2–6.6 directly.

The rules are transcriptions, not reinterpretations. A few worth flagging because earlier
guesses got them wrong:

| | Notebook 01 | earlier guess |
|---|---|---|
| `is_extended_stay` | `minimum_nights >= 14` | `>= 28` |
| `has_building_staff` | building staff / **front desk** / doorman | ...concierge |
| `amenity_count` | `len(list)` — counts duplicates | unique set |
| unmatched `property_type` | `'Other'`, then `drop_first` may drop it | always `'Other'` |
| coordinates | rounded to 2 dp **before** the distance calc | raw |

It needs Notebook 01's fitted state — scaler moments, train medians, amenity vocabulary,
per-city price moments. Without the moments no valid feature vector exists, and the cell
says so rather than producing a confident 0.0%.

In [20]:
# =====================================================================================
# 6.8b  Notebook 01's transform, implemented here
# -------------------------------------------------------------------------------------
# WHY THIS CELL WAS REWRITTEN. Earlier versions borrowed the booking endpoint's
# preprocessing and reconciled whatever it produced against the estimator's 100-column
# contract. That produced P(instant) = 0.0% -- a saturated sigmoid, which is what a
# linear model returns when the feature vector is nowhere near the data it was fitted on.
#
# The cause is in Notebook 01 section 6.6, and it is not subtle:
#
#     scaler = StandardScaler()
#     df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
#
# Nineteen continuous columns are STANDARDISED. `maximum_nights` is not 1125 in the
# training matrix, it is about +0.4. `host_tenure_days` is not 2,847, it is about -0.2.
# Feeding raw magnitudes into coefficients fitted on z-scores drives the linear predictor
# to a large negative number, and expit() of that is 0.0 to four decimal places. Every
# reconciliation rule in the previous version was operating downstream of that, which is
# why none of them caught it: the columns were all present, correctly named and correctly
# ordered, and every value was wrong by one standardisation.
#
# There is no way to patch a borrowed transform into being Notebook 01's transform. So
# this cell implements Notebook 01 sections 4.1-4.5, 5 and 6.2-6.6 directly, driven by
# the fitted state Notebook 01 computed (`scaler_means`, `scaler_scales`,
# `train_imputations`, `city_price_stats`, `top_amenities_vocab`, ...). That state is the
# whole ballgame: without the scaler moments this tab cannot produce a valid feature
# vector, and it says so rather than producing a confident 0.0%.
#
# The rules below are transcriptions of Notebook 01, not reinterpretations. Where its
# behaviour is surprising, the comment says so rather than tidying it up.
# =====================================================================================
import re
from datetime import datetime as _dt

# --- Notebook 01 policy constants, section 1 (verbatim) --------------------------------
NB01_SNAPSHOT_DATE = pd.Timestamp("2021-03-01")
NB01_COORD_DECIMALS = 2
NB01_MIN_NIGHTS_CAP = 365
NB01_MAX_NIGHTS_CAP = 1125
NB01_BEDROOMS_CAP = 16
NB01_TOP_K_AMENITIES = 30
NB01_CITY_CENTERS = {
    "Paris": (48.8530, 2.3499), "New York": (40.7580, -73.9855),
    "Sydney": (-33.8568, 151.2153), "Rome": (41.8986, 12.4769),
    "Rio de Janeiro": (-22.9711, -43.1822), "Istanbul": (41.0054, 28.9768),
    "Mexico City": (19.4326, -99.1332), "Bangkok": (13.7460, 100.5340),
    "Cape Town": (-33.9221, 18.4231), "Hong Kong": (22.2819, 114.1582),
}

# Section 6.5: get_dummies(..., drop_first=True) over these four columns. The dropped
# level is the alphabetically first category, and it encodes as an all-zero block. We do
# not hard-code which one that is -- it is whichever level is absent from the fitted
# contract, which the encoder below reads off `final_feature_columns` directly.
NB01_OHE_COLS = ["city", "room_type", "property_type_grouped", "host_response_time"]

# Section 4.5: missingness flags, captured BEFORE imputation.
NB01_ISNA_COLS = ["host_response_rate", "host_acceptance_rate", "host_response_time",
                  "review_scores_rating", "review_scores_accuracy",
                  "review_scores_cleanliness", "review_scores_checkin",
                  "review_scores_communication", "review_scores_location",
                  "review_scores_value", "bedrooms"]

NB01_STATE, NB01_STATE_SOURCE, NB01_STATE_ERROR = {}, None, None
BOOKING_PREPROC_ERROR = None      # kept for the handler's unavailability card
TRAIN_SAMPLE, TRAIN_SAMPLE_SOURCE, _TRAIN_RANGES = None, None, {}

_STATE_KEYS = ("scaler_features", "scaler_means", "scaler_scales", "train_imputations",
               "city_price_stats", "city_price_caps", "top_amenities_vocab",
               "host_listings_count_cap", "top_property_types", "final_feature_columns")
_ARTIFACT_NAMES = ["01_preprocessing_artifacts.json", "preprocessing_artifacts.json",
                   "preprocessor_artifacts.json", "artifacts.json"]


# --- Locating Notebook 01's fitted state ------------------------------------------------
def _load_nb01_state():
    """Notebook 01's `artifacts` dict. Local file first, then the team's S3 prefix.

    Note that Notebook 01 BUILDS this dict in section 6.1 and fills it throughout section
    6, but its section 7 export cell writes only the four CSVs -- the JSON is never
    persisted. So if this comes back empty, that is expected, and Section 6.8b-export
    below prints the two lines to add to Notebook 01 to fix it permanently.
    """
    for n in _ARTIFACT_NAMES:
        for d in (".", "artifacts", "processed", "processed_instant_booking",
                  os.path.join("processed_instant_booking", "processed")):
            p = os.path.join(d, n)
            if os.path.isfile(p):
                with open(p, encoding="utf-8") as fh:
                    return json.load(fh), os.path.abspath(p)
    for n in _ARTIFACT_NAMES:
        for key in (f"{BOOKING_PREFIX}/processed/{n}", f"{BOOKING_PREFIX}/artifacts/{n}"):
            try:
                body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
            except Exception:
                continue
            return json.loads(body.decode("utf-8")), f"s3://{BUCKET}/{key}"
    return {}, None


def _load_train_sample(max_bytes=6_000_000):
    """A byte range of train_features.csv -- the matrix the estimator was fitted on. Used
    to verify this transform rather than to build it."""
    key = f"{BOOKING_PREFIX}/processed/train_features.csv"
    obj = s3.get_object(Bucket=BUCKET, Key=key, Range=f"bytes=0-{max_bytes - 1}")
    blob = obj["Body"].read().decode("utf-8", errors="replace")
    return pd.read_csv(io.StringIO(blob[:blob.rfind("\n")]), low_memory=False), \
        f"s3://{BUCKET}/{key}"


NB01_STATE, NB01_STATE_SOURCE = _load_nb01_state()

try:
    TRAIN_SAMPLE, TRAIN_SAMPLE_SOURCE = _load_train_sample()
    _TRAIN_RANGES = {c: (float(TRAIN_SAMPLE[c].min()), float(TRAIN_SAMPLE[c].max()))
                     for c in TRAIN_SAMPLE.columns}
except Exception as _e:
    TRAIN_SAMPLE_SOURCE = f"unavailable ({type(_e).__name__}: {_e})"

# `final_feature_columns` is the one piece we can recover without the JSON: it is the
# header of train_features.csv, and the estimator's own feature_names_in_ agrees with it.
if not NB01_STATE.get("final_feature_columns"):
    NB01_STATE["final_feature_columns"] = list(
        LOGREG_FITTED_COLUMNS or LOGREG_FEATURE_CONTRACT)

_missing_state = [k for k in _STATE_KEYS if not NB01_STATE.get(k)]
NB01_READY = not _missing_state
if not NB01_READY:
    NB01_STATE_ERROR = (
        f"Notebook 01's fitted state is incomplete: missing {_missing_state}. "
        "The scaler moments in particular cannot be inferred from anything else -- "
        "train_features.csv holds the ALREADY-SCALED values, so it cannot tell you what "
        "was subtracted. Run the export cell in 6.8b-export from Notebook 01.")
    BOOKING_PREPROC_ERROR = NB01_STATE_ERROR


# --- Notebook 01, section 5.1: amenity parsing ------------------------------------------
def nb01_clean_amenities(raw_val):
    """Verbatim from Notebook 01 section 5.1. Note it returns a LIST, not a set, so
    `amenity_count` counts duplicates too -- preserved deliberately."""
    if raw_val is None or (isinstance(raw_val, float) and np.isnan(raw_val)):
        return []
    text = str(raw_val).strip()
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [re.sub(r"\s+", " ", str(i).strip().lower()) for i in parsed if i]
    except Exception:
        pass
    cleaned = re.sub(r"[\[\]\"'{}()]", "", text)
    return [re.sub(r"\s+", " ", i.strip().lower())
            for i in cleaned.split(",") if i.strip()]


def _amenity_col(term):
    """Notebook 01 section 6.2's column-naming rule, character for character."""
    return "amenity_" + re.sub(r"[^a-zA-Z0-9_]", "_", term.lower()).strip("_")


def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2.0) ** 2
    return R * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def _tf_to_int(v):
    """Section 4.3: 't'/'f' -> 1/0, and NaN is PRESERVED so the _isna flags can see it."""
    if v is None:
        return np.nan
    if isinstance(v, str):
        return {"t": 1, "f": 0}.get(v.strip().lower(), np.nan)
    if isinstance(v, bool):
        return int(v)
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return np.nan


def _rate_to_unit(v):
    """Section 4.3: percentages above 1.0 are divided by 100; values already in [0,1]
    are left alone. This notebook's `build_booking_payload` already sends 0-1, so this is
    idempotent for our payloads and correct for anything else."""
    if v is None:
        return np.nan
    v = float(v)
    return v / 100.0 if v > 1.0 else v


def nb01_raw_to_features(payloads):
    """One or more raw BOOKING_FIELDS dicts -> the 100-column matrix the LR was fitted on.

    Notebook 01 sections 4.1-4.5 (clean), 5 (engineer) and 6.2-6.6 (encode, impute,
    scale), applied with Notebook 01's own fitted state. The estimator was fitted on the
    output of exactly this, so nothing here is a reconstruction of somebody else's
    transform -- it IS the transform.
    """
    if not NB01_READY:
        raise RuntimeError(NB01_STATE_ERROR)

    st = NB01_STATE
    rows = payloads if isinstance(payloads, list) else [payloads]
    final_cols = list(st["final_feature_columns"])
    out_rows = []

    for p in rows:
        f = {}

        # ---- Section 4: cleaning -------------------------------------------------------
        lat = round(float(p["latitude"]), NB01_COORD_DECIMALS)
        lon = round(float(p["longitude"]), NB01_COORD_DECIMALS)

        superhost = _tf_to_int(p.get("host_is_superhost"))
        profile_pic = _tf_to_int(p.get("host_has_profile_pic"))
        verified = _tf_to_int(p.get("host_identity_verified"))

        resp_rate = _rate_to_unit(p.get("host_response_rate"))
        acc_rate = _rate_to_unit(p.get("host_acceptance_rate"))

        min_nights = min(max(int(p.get("minimum_nights") or 1), 1), NB01_MIN_NIGHTS_CAP)
        max_nights = min(max(int(p.get("maximum_nights") or 1), 1), NB01_MAX_NIGHTS_CAP)

        bedrooms = p.get("bedrooms")
        bedrooms = None if bedrooms is None else min(float(bedrooms), NB01_BEDROOMS_CAP)

        raw_vals = {"host_response_rate": resp_rate, "host_acceptance_rate": acc_rate,
                    "host_response_time": p.get("host_response_time"),
                    "bedrooms": bedrooms,
                    **{c: p.get(c) for c in REVIEW_COLS}}

        # Section 4.5: the _isna flags are taken BEFORE imputation. Order matters.
        for c in NB01_ISNA_COLS:
            v = raw_vals.get(c)
            f[f"{c}_isna"] = int(v is None or (isinstance(v, float) and np.isnan(v)))

        # ---- Section 5: feature engineering --------------------------------------------
        amen = nb01_clean_amenities(p.get("amenities"))
        f["amenity_count"] = len(amen)
        f["has_self_checkin"] = int(any("self check-in" in a or "self check in" in a
                                        for a in amen))
        f["has_keypad"] = int(any("keypad" in a for a in amen))
        f["has_smart_lock"] = int(any("smart lock" in a for a in amen))
        f["has_lockbox"] = int(any("lockbox" in a for a in amen))
        f["has_building_staff"] = int(any("building staff" in a or "front desk" in a
                                          or "doorman" in a for a in amen))
        f["has_keyless_entry_composite"] = int(
            f["has_self_checkin"] or f["has_keypad"] or f["has_smart_lock"]
            or f["has_lockbox"] or f["has_building_staff"])

        hs = p.get("host_since")
        tenure = np.nan
        if hs:
            try:
                tenure = (NB01_SNAPSHOT_DATE - pd.Timestamp(hs)).days
                if tenure < 0:
                    tenure = np.nan     # section 5.2 nulls anomalous tenure
            except Exception:
                tenure = np.nan
        f["host_tenure_days"] = tenure

        sh = 0 if (superhost is None or pd.isna(superhost)) else int(superhost)
        vf = 0 if (verified is None or pd.isna(verified)) else int(verified)
        f["is_superhost_verified"] = sh * vf

        accommodates = int(p.get("accommodates") or 1)
        denom = max(1.0, 1.0 if bedrooms is None else float(bedrooms))
        f["accommodates_per_bedroom"] = round(accommodates / denom, 2)

        # Section 5.3: the threshold is 14 nights, not 28.
        f["is_single_night"] = int(min_nights == 1)
        f["is_extended_stay"] = int(min_nights >= 14)
        f["superhost_x_single_night"] = sh * f["is_single_night"]

        city = p.get("city")
        if city in NB01_CITY_CENTERS:
            clat, clon = NB01_CITY_CENTERS[city]
            f["dist_to_center_km"] = round(float(_haversine_km(lat, lon, clat, clon)), 2)
        else:
            f["dist_to_center_km"] = np.nan

        # ---- Section 6.2: top-30 amenity vocabulary ------------------------------------
        for term in st["top_amenities_vocab"]:
            f[_amenity_col(term)] = int(term in amen)

        # ---- Section 6.3: caps and within-city price z-score ---------------------------
        listings = float(p.get("host_total_listings_count") or 0)
        f["host_total_listings_count"] = min(listings, float(st["host_listings_count_cap"]))

        price = float(p.get("price") or 0.0)
        cap = st["city_price_caps"].get(city)
        if cap is not None:
            price = min(price, float(cap))
        stats = st["city_price_stats"].get(city)
        # Section 6.3 initialises this to 0.0 and only overwrites it for known cities, so
        # an unseen city legitimately scores 0.0 here.
        f["log_price_city_zscore"] = (
            0.0 if stats is None
            else (np.log1p(price) - stats["mean"]) / (stats["std"] or 1.0))

        # ---- Passthrough columns -------------------------------------------------------
        f["latitude"], f["longitude"] = lat, lon
        f["accommodates"] = accommodates
        f["minimum_nights"], f["maximum_nights"] = min_nights, max_nights
        f["host_response_rate"], f["host_acceptance_rate"] = resp_rate, acc_rate
        f["host_is_superhost"] = superhost
        f["host_has_profile_pic"] = profile_pic
        f["host_identity_verified"] = verified
        f["bedrooms"] = bedrooms
        for c in REVIEW_COLS:
            v = p.get(c)
            f[c] = np.nan if v is None else float(v)

        # ---- Section 6.4: train-fitted imputation --------------------------------------
        imp = st["train_imputations"]
        for c, v in list(f.items()):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                if c in imp and not isinstance(imp[c], str):
                    f[c] = float(imp[c])
        resp_time = p.get("host_response_time") or imp.get("host_response_time")
        room_type = p.get("room_type") or imp.get("room_type")
        prop_type = p.get("property_type") or imp.get("property_type")
        for c in ("host_is_superhost", "host_has_profile_pic", "host_identity_verified"):
            if f[c] is None or (isinstance(f[c], float) and np.isnan(f[c])):
                f[c] = float(_tf_to_int(imp.get(c)) or 0)

        # ---- Section 6.5: one-hot, drop_first ------------------------------------------
        # `drop_first=True` removes the alphabetically first level, which then encodes as
        # an all-zero block. Rather than hard-coding which level that is, we set the
        # column if the fitted contract has one and leave the block at zero if it does
        # not -- which covers the dropped reference AND an unseen level, correctly, and
        # without assuming which city or room type happens to sort first.
        prop_grouped = (prop_type if prop_type in st["top_property_types"] else "Other")
        for col, val in (("city", city), ("room_type", room_type),
                         ("property_type_grouped", prop_grouped),
                         ("host_response_time", resp_time)):
            target = f"{col}_{val}"
            if target in final_cols:
                f[target] = 1

        out_rows.append(f)

    X = pd.DataFrame(out_rows).reindex(columns=final_cols, fill_value=0)
    X = X.fillna(0.0).astype(np.float64)

    # ---- Section 6.6: StandardScaler, using Notebook 01's fitted moments ---------------
    # This is the step whose absence produced 0.0%.
    for c in st["scaler_features"]:
        if c in X.columns:
            mean = float(st["scaler_means"][c])
            scale = float(st["scaler_scales"][c]) or 1.0
            X[c] = (X[c] - mean) / scale

    return X, []


# Keep the name the handler and the verify cell already use.
booking_raw_to_features = nb01_raw_to_features
BOOKING_PREPROC = nb01_raw_to_features if NB01_READY else None
BOOKING_PREPROC_KIND = "notebook 01 transform (implemented here)"
BOOKING_PREPROC_SOURCE = (f"Notebook 01 sections 4/5/6, fitted state from "
                          f"{NB01_STATE_SOURCE}") if NB01_READY else None

print("Baseline transform: Notebook 01, implemented in this notebook")
print(f"  fitted state : {NB01_STATE_SOURCE or 'NOT FOUND'}")
print(f"  train sample : {TRAIN_SAMPLE_SOURCE}")
if NB01_READY:
    print(f"  scaler       : {len(NB01_STATE['scaler_features'])} continuous columns "
          "standardised with Notebook 01's own means/scales")
    print(f"  amenities    : {len(NB01_STATE['top_amenities_vocab'])} vocabulary terms")
    print(f"  cities       : {len(NB01_STATE['city_price_stats'])} with fitted price "
          "moments")
    print(f"  contract     : {len(NB01_STATE['final_feature_columns'])} final columns")
    print("\n  The next cell checks the output against train_features.csv before the tab")
    print("  is allowed to quote a probability.")
else:
    print(f"\n  NOT READY - {NB01_STATE_ERROR}")
    print("  Two routes: 6.8b-export prints a cell to add to Notebook 01 (cheap,\n"
          "  exact), or 6.8b-refit rebuilds the state here from the raw S3 file and\n"
          "  verifies it against train_features.csv (slower, needs no other notebook).")


Baseline transform: Notebook 01, implemented in this notebook
  fitted state : /home/sagemaker-user/HSAssignment/01_preprocessing_artifacts.json
  train sample : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/processed/train_features.csv
  scaler       : 19 continuous columns standardised with Notebook 01's own means/scales
  amenities    : 30 vocabulary terms
  cities       : 10 with fitted price moments
  contract     : 100 final columns

  The next cell checks the output against train_features.csv before the tab
  is allowed to quote a probability.


#### 6.8b-refit Rebuilding Notebook 01's state without touching Notebook 01

Notebook 01 never persists its `artifacts` dict, and the scaler moments can't be read back
out of the CSVs. But they **can be recomputed**, because Notebook 01 uploads its raw input
to S3 in section 1.2:

```
s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/raw/Listings.csv
```

and everything downstream is deterministic. `train_test_split` is seeded
(`random_state=42`, `test_size=0.20`, stratified on city x target), `df_raw` isn't mutated
between load and cleaning, and every fitted quantity is a median, mode, quantile,
frequency count or mean/std over the training partition. Same input, same seed, same code
→ same numbers.

**"Deterministic in principle" is an argument, not evidence.** A replay that quietly
diverges would hand back plausible moments that are subtly wrong — worse than no moments,
because the tab would then produce a confident number instead of refusing. So the replay
doesn't get to assert its own correctness: it rebuilds the training matrix and compares it
against `train_features.csv` cell by cell. If every column of every sampled row matches,
the recovered moments provably *are* Notebook 01's. If one column disagrees, the state is
discarded and the tab stays disabled.

This is the more expensive route — it downloads the full raw file and reruns the pipeline,
tens of seconds rather than nothing — and it re-derives state that already existed once.
Notebook 01's section 7.1 export remains the better fix, since it writes the real dict at
the moment it was fitted, with nothing to reconstruct and nothing to verify. Use this when
re-running Notebook 01 is inconvenient, or to cross-check that the exported artifacts and
the raw data still agree.

In [21]:
# =====================================================================================
# 6.8b-refit  Rebuilding Notebook 01's fitted state without touching Notebook 01
# -------------------------------------------------------------------------------------
# Notebook 01 never persists its `artifacts` dict, and the scaler moments cannot be read
# back out of the CSVs. But they CAN be recomputed, because Notebook 01 uploads its raw
# input to S3 in section 1.2:
#
#     s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/raw/Listings.csv
#
# and everything downstream of it is deterministic. `train_test_split` is seeded
# (random_state=42, test_size=0.20, stratified on city x target), `df_raw` is not mutated
# between load and cleaning, and every fitted quantity is a median, a mode, a quantile, a
# frequency count or a mean/std over the training partition. Same input, same seed, same
# code -> same numbers.
#
# WHY THIS IS SAFE TO DO HERE, AND HOW WE KNOW. "Deterministic in principle" is an
# argument, not evidence, and a replay that quietly diverges would hand back plausible
# moments that are subtly wrong -- worse than no moments at all, because the tab would
# then produce a confident number instead of refusing. So the replay does not get to
# assert its own correctness: it rebuilds the training matrix and compares it against
# `train_features.csv` cell by cell. If every column of every sampled row matches to
# floating-point tolerance, the recovered moments provably ARE Notebook 01's. If a single
# column disagrees, the state is discarded and the tab stays disabled.
#
# This is the more expensive of the two routes -- it downloads the full raw file and
# reruns the pipeline, tens of seconds rather than nothing -- and it re-derives state that
# already existed once. Notebook 01 section 7.1 remains the better fix: it exports the
# real dict at the moment it was fitted, with no reconstruction and nothing to verify.
# Use this when re-running Notebook 01 is inconvenient, or to confirm that the exported
# artifacts and the raw data still agree.
# =====================================================================================
REFIT_FROM_RAW = True          # set False to skip this cell entirely
REFIT_VERIFY_ROWS = 400        # rows compared against train_features.csv
RAW_S3_KEY = f"{BOOKING_PREFIX}/raw/Listings.csv"

# Notebook 01 section 1 policy constants (verbatim).
NB01_RANDOM_STATE, NB01_TEST_SIZE = 42, 0.20
NB01_SPARSITY_THRESHOLD = 0.80
NB01_PRICE_CAP_Q, NB01_COUNT_CAP_Q = 0.995, 0.99
NB01_TARGET = "instant_bookable"

REFIT_STATE, REFIT_REPORT, REFIT_OK = None, [], False


def _refit_notebook01(raw_df):
    """Notebook 01 sections 4, 5 and 6.1-6.6, returning (artifacts, X_train, X_test).

    Transcribed, not reinterpreted. Where Notebook 01 does something surprising it is
    reproduced as-is -- a replay that 'improves' the original is not a replay.
    """
    from collections import Counter
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    d = raw_df.copy()

    # ---- Section 4: cleaning & governance ---------------------------------------------
    d.drop(columns=[c for c in ("listing_id", "host_id", "name") if c in d.columns],
           inplace=True)
    d["latitude"] = d["latitude"].round(NB01_COORD_DECIMALS)
    d["longitude"] = d["longitude"].round(NB01_COORD_DECIMALS)
    d.drop_duplicates(inplace=True)

    d = d[d[NB01_TARGET].notna()].copy()
    d[NB01_TARGET] = d[NB01_TARGET].map({"t": 1, "f": 0, True: 1, False: 0, 1: 1, 0: 0})
    d = d[d[NB01_TARGET].isin([0, 1])].copy()
    d[NB01_TARGET] = d[NB01_TARGET].astype(int)

    sparse = d.isna().mean()
    d.drop(columns=sparse[sparse > NB01_SPARSITY_THRESHOLD].index.tolist(), inplace=True)

    for c in ("host_is_superhost", "host_has_profile_pic", "host_identity_verified"):
        if c in d.columns:
            d[c] = d[c].map({"t": 1, "f": 0})
    for c in ("host_response_rate", "host_acceptance_rate"):
        if c in d.columns:
            m = d[c] > 1.0
            d.loc[m, c] = d.loc[m, c] / 100.0

    d = d[d["price"] > 0].copy()
    d.loc[d["minimum_nights"] > NB01_MIN_NIGHTS_CAP, "minimum_nights"] = NB01_MIN_NIGHTS_CAP
    d.loc[d["minimum_nights"] < 1, "minimum_nights"] = 1
    d.loc[d["maximum_nights"] > NB01_MAX_NIGHTS_CAP, "maximum_nights"] = NB01_MAX_NIGHTS_CAP
    d.loc[d["maximum_nights"] < 1, "maximum_nights"] = 1
    if "bedrooms" in d.columns:
        d.loc[d["bedrooms"] > NB01_BEDROOMS_CAP, "bedrooms"] = NB01_BEDROOMS_CAP

    for c in NB01_ISNA_COLS:
        if c in d.columns:
            d[f"{c}_isna"] = d[c].isna().astype(int)

    # ---- Section 5: deterministic feature engineering ---------------------------------
    parsed = d["amenities"].apply(nb01_clean_amenities)
    d["amenity_count"] = parsed.apply(len)
    d["has_self_checkin"] = parsed.apply(
        lambda i: int(any("self check-in" in x or "self check in" in x for x in i)))
    d["has_keypad"] = parsed.apply(lambda i: int(any("keypad" in x for x in i)))
    d["has_smart_lock"] = parsed.apply(lambda i: int(any("smart lock" in x for x in i)))
    d["has_lockbox"] = parsed.apply(lambda i: int(any("lockbox" in x for x in i)))
    d["has_building_staff"] = parsed.apply(
        lambda i: int(any("building staff" in x or "front desk" in x or "doorman" in x
                          for x in i)))
    d["has_keyless_entry_composite"] = (
        (d.has_self_checkin == 1) | (d.has_keypad == 1) | (d.has_smart_lock == 1)
        | (d.has_lockbox == 1) | (d.has_building_staff == 1)).astype(int)

    tenure = (NB01_SNAPSHOT_DATE - pd.to_datetime(d["host_since"], errors="coerce")).dt.days
    d["host_tenure_days"] = tenure
    d.loc[d["host_tenure_days"] < 0, "host_tenure_days"] = np.nan

    sh = d["host_is_superhost"].fillna(0).astype(int)
    vf = d["host_identity_verified"].fillna(0).astype(int)
    d["is_superhost_verified"] = (sh * vf).astype(int)

    d["accommodates_per_bedroom"] = (
        d["accommodates"] / d["bedrooms"].fillna(1.0).clip(lower=1.0)).round(2)
    d["is_single_night"] = (d["minimum_nights"] == 1).astype(int)
    d["is_extended_stay"] = (d["minimum_nights"] >= 14).astype(int)
    d["superhost_x_single_night"] = (sh * d["is_single_night"]).astype(int)

    clat = d["city"].map(lambda c: NB01_CITY_CENTERS.get(c, (np.nan, np.nan))[0])
    clon = d["city"].map(lambda c: NB01_CITY_CENTERS.get(c, (np.nan, np.nan))[1])
    d["dist_to_center_km"] = _haversine_km(
        d["latitude"].values, d["longitude"].values, clat.values, clon.values).round(2)

    # ---- Section 6.1: the seeded, stratified split ------------------------------------
    strat = d["city"].astype(str) + "_" + d[NB01_TARGET].astype(str)
    dtr, dte = train_test_split(d, test_size=NB01_TEST_SIZE,
                                random_state=NB01_RANDOM_STATE, stratify=strat)
    dtr, dte = dtr.copy(), dte.copy()

    art = {"train_imputations": {}, "city_price_stats": {}, "city_price_caps": {}}

    # ---- 6.2: top-K amenity vocabulary, fitted on train only --------------------------
    toks = [t for lst in dtr["amenities"].apply(nb01_clean_amenities) for t in lst]
    vocab = [t for t, _ in Counter(toks).most_common(NB01_TOP_K_AMENITIES * 2)
             if len(t) > 2][:NB01_TOP_K_AMENITIES]
    art["top_amenities_vocab"] = vocab
    for frame in (dtr, dte):
        pf = frame["amenities"].apply(nb01_clean_amenities)
        for term in vocab:
            frame[_amenity_col(term)] = pf.apply(lambda items: int(term in items))

    # ---- 6.3: caps and within-city price normalisation --------------------------------
    cap = float(dtr["host_total_listings_count"].quantile(NB01_COUNT_CAP_Q))
    art["host_listings_count_cap"] = cap
    for frame in (dtr, dte):
        frame["host_total_listings_count"] = frame["host_total_listings_count"].clip(upper=cap)
        frame["log_price_city_zscore"] = 0.0

    for city in dtr["city"].unique():
        mtr, mte = dtr["city"] == city, dte["city"] == city
        ccap = float(dtr.loc[mtr, "price"].quantile(NB01_PRICE_CAP_Q))
        art["city_price_caps"][city] = ccap
        dtr.loc[mtr, "price"] = dtr.loc[mtr, "price"].clip(upper=ccap)
        dte.loc[mte, "price"] = dte.loc[mte, "price"].clip(upper=ccap)
        lp = np.log1p(dtr.loc[mtr, "price"])
        mu = float(lp.mean())
        sd = float(lp.std()) if lp.std() > 0 else 1.0
        art["city_price_stats"][city] = {"mean": mu, "std": sd}
        dtr.loc[mtr, "log_price_city_zscore"] = (np.log1p(dtr.loc[mtr, "price"]) - mu) / sd
        dte.loc[mte, "log_price_city_zscore"] = (np.log1p(dte.loc[mte, "price"]) - mu) / sd

    # ---- 6.4: train-fitted imputation --------------------------------------------------
    for c in ("host_response_rate", "host_acceptance_rate", "host_total_listings_count",
              "bedrooms", *REVIEW_COLS, "host_tenure_days", "dist_to_center_km",
              "accommodates_per_bedroom"):
        if c in dtr.columns:
            med = float(dtr[c].median())
            art["train_imputations"][c] = med
            dtr[c] = dtr[c].fillna(med)
            dte[c] = dte[c].fillna(med)
    for c in ("host_response_time", "host_is_superhost", "host_has_profile_pic",
              "host_identity_verified", "property_type", "room_type"):
        if c in dtr.columns:
            mode = dtr[c].mode(dropna=True)[0]
            art["train_imputations"][c] = str(mode)
            dtr[c] = dtr[c].fillna(mode)
            dte[c] = dte[c].fillna(mode)

    # ---- 6.5: one-hot encoding & column alignment --------------------------------------
    topp = dtr["property_type"].value_counts().nlargest(10).index.tolist()
    art["top_property_types"] = topp
    for frame in (dtr, dte):
        frame["property_type_grouped"] = frame["property_type"].apply(
            lambda x: x if x in topp else "Other")

    dtr = pd.get_dummies(dtr, columns=NB01_OHE_COLS, drop_first=True, dtype=int)
    dte = pd.get_dummies(dte, columns=NB01_OHE_COLS, drop_first=True, dtype=int)

    y_tr, y_te = dtr[NB01_TARGET].copy(), dte[NB01_TARGET].copy()
    drop = [NB01_TARGET, "amenities", "host_since", "host_location", "neighbourhood",
            "property_type", "price", "log_price"]
    cols = [c for c in dtr.columns if c not in drop]
    dtr = dtr.reindex(columns=cols, fill_value=0)
    dte = dte.reindex(columns=cols, fill_value=0)

    # ---- 6.6: StandardScaler on the continuous block -----------------------------------
    scale_cols = [c for c in (
        "accommodates", "bedrooms", "minimum_nights", "maximum_nights", "amenity_count",
        "host_tenure_days", "accommodates_per_bedroom", "dist_to_center_km",
        "log_price_city_zscore", "host_total_listings_count", "host_response_rate",
        "host_acceptance_rate", *REVIEW_COLS) if c in dtr.columns]
    art["scaler_features"] = scale_cols

    scaler = StandardScaler()
    dtr[scale_cols] = scaler.fit_transform(dtr[scale_cols])
    dte[scale_cols] = scaler.transform(dte[scale_cols])
    art["scaler_means"] = {c: float(scaler.mean_[i]) for i, c in enumerate(scale_cols)}
    art["scaler_scales"] = {c: float(scaler.scale_[i]) for i, c in enumerate(scale_cols)}
    art["final_feature_columns"] = list(dtr.columns)

    return art, dtr, dte


if not REFIT_FROM_RAW:
    print("REFIT_FROM_RAW is False - skipped.")
elif NB01_READY:
    print(f"Notebook 01's exported state was already found at {NB01_STATE_SOURCE}")
    print("  Nothing to rebuild. Set REFIT_FROM_RAW = True and NB01_READY = False if you")
    print("  want to cross-check the export against a fresh replay of the raw data.")
else:
    try:
        _t0 = time.perf_counter()
        _obj = s3.get_object(Bucket=BUCKET, Key=RAW_S3_KEY)
        _raw = pd.read_csv(io.BytesIO(_obj["Body"].read()), encoding="utf-8",
                           encoding_errors="replace", low_memory=False)
        REFIT_REPORT.append(("raw data", "PASS",
                             f"{len(_raw):,} rows x {_raw.shape[1]} cols from "
                             f"s3://{BUCKET}/{RAW_S3_KEY}"))

        _art, _Xtr, _Xte = _refit_notebook01(_raw)
        REFIT_REPORT.append(("replay", "PASS",
                             f"{len(_Xtr):,} train rows x {_Xtr.shape[1]} cols in "
                             f"{time.perf_counter() - _t0:,.1f}s"))

        # --- The check that decides whether any of this is usable ----------------------
        # Recomputed moments are worth nothing on their own. They are worth something
        # only if the matrix they produce is the matrix the model was fitted on.
        _expected_cols = list(LOGREG_FITTED_COLUMNS or LOGREG_FEATURE_CONTRACT)
        _cols_ok = list(_Xtr.columns) == _expected_cols
        REFIT_REPORT.append(
            ("column contract", "PASS" if _cols_ok else "FAIL",
             f"{_Xtr.shape[1]} columns, "
             + ("identical to the fitted order" if _cols_ok else
                f"DIFFERS: only in replay {sorted(set(_Xtr.columns) - set(_expected_cols))[:4]}, "
                f"only in contract {sorted(set(_expected_cols) - set(_Xtr.columns))[:4]}")))

        if TRAIN_SAMPLE is None:
            REFIT_REPORT.append(("matrix match", "FAIL",
                                 "train_features.csv unavailable, so the replay cannot "
                                 "be checked - refusing to trust it"))
        elif _cols_ok:
            _n = min(REFIT_VERIFY_ROWS, len(TRAIN_SAMPLE), len(_Xtr))
            _a = _Xtr.iloc[:_n][_expected_cols].to_numpy(dtype=float)
            _b = TRAIN_SAMPLE.iloc[:_n][_expected_cols].to_numpy(dtype=float)
            _close = np.isclose(_a, _b, rtol=1e-6, atol=1e-8)
            _n_bad = int((~_close).sum())
            if _n_bad:
                _bad_cols = [(_expected_cols[j], int((~_close[:, j]).sum()))
                             for j in range(_close.shape[1]) if (~_close[:, j]).any()]
                REFIT_REPORT.append(
                    ("matrix match", "FAIL",
                     f"{_n_bad:,} of {_a.size:,} cells differ over {_n:,} rows; worst "
                     f"columns {sorted(_bad_cols, key=lambda t: -t[1])[:4]}"))
            else:
                REFIT_REPORT.append(
                    ("matrix match", "PASS",
                     f"all {_a.size:,} cells across {_n:,} rows x {len(_expected_cols)} "
                     "columns match train_features.csv - the recovered moments ARE "
                     "Notebook 01's"))

        REFIT_OK = all(s == "PASS" for _, s, _ in REFIT_REPORT)
        if REFIT_OK:
            REFIT_STATE = _art

    except Exception as _e:
        REFIT_REPORT.append(("refit", "FAIL", f"{type(_e).__name__}: {_e}"))

    print("Rebuilding Notebook 01's fitted state from the raw file")
    for _n, _s, _d in REFIT_REPORT:
        print(f"  [{_s:<4}] {_n:<17} {_d}")
    print()

    if REFIT_OK:
        # Adopt it, and cache it where 6.8b looks first so a kernel restart is cheap.
        NB01_STATE = REFIT_STATE
        NB01_STATE_SOURCE = f"refit from s3://{BUCKET}/{RAW_S3_KEY}, verified"
        NB01_STATE_ERROR, NB01_READY = None, True
        BOOKING_PREPROC, BOOKING_PREPROC_ERROR = nb01_raw_to_features, None
        BOOKING_PREPROC_SOURCE = NB01_STATE_SOURCE

        with open("01_preprocessing_artifacts.json", "w", encoding="utf-8") as _fh:
            json.dump(REFIT_STATE, _fh, indent=2, default=str)

        print("  VERIFIED and adopted. Cached locally as 01_preprocessing_artifacts.json,")
        print("  which 6.8b finds before it reaches S3 on a re-run.")
        print(f"  {len(REFIT_STATE['scaler_features'])} scaler moments, "
              f"{len(REFIT_STATE['top_amenities_vocab'])} amenity terms, "
              f"{len(REFIT_STATE['city_price_stats'])} cities.")
        print("\n  Re-run 6.8b-verify and 6.8c, then the launch cell.")
    else:
        print("  NOT ADOPTED. The replay did not reproduce train_features.csv, so its")
        print("  moments are not Notebook 01's and using them would put every listing in")
        print("  a slightly wrong space - a confident wrong answer rather than a refusal.")
        print("  A 'matrix match' failure usually means the raw file in S3 is not the one")
        print("  Notebook 01 actually ran on. Export the real state from Notebook 01")
        print("  section 7.1 instead; it needs no reconstruction.")


Notebook 01's exported state was already found at /home/sagemaker-user/HSAssignment/01_preprocessing_artifacts.json
  Nothing to rebuild. Set REFIT_FROM_RAW = True and NB01_READY = False if you
  want to cross-check the export against a fresh replay of the raw data.


#### 6.8b-export Getting Notebook 01's fitted state out of Notebook 01

Notebook 01 builds a complete `artifacts` dict in section 6 — scaler moments, train
imputations, the amenity vocabulary, per-city price statistics, the property-type
whitelist — and then its section 7 export cell writes **only the four CSVs**. The dict is
never persisted; it dies with the kernel.

That is the one thing that can stop 6.8b, and the missing piece is not recoverable from
anywhere else: `train_features.csv` holds the values *after* standardisation, so it can
tell you what the scaled data looks like but not what was subtracted to get there.

The cell below writes nothing. It prints the ~15 lines to add to Notebook 01, once.

In [22]:
# =====================================================================================
# 6.8b-export  Getting Notebook 01's fitted state out of Notebook 01
# -------------------------------------------------------------------------------------
# Notebook 01 builds a complete `artifacts` dict in section 6 -- scaler moments, train
# imputations, the amenity vocabulary, per-city price statistics, the property-type
# whitelist, the final column order -- and then its section 7 export cell writes only the
# four CSVs. The dict is never persisted, so it dies with the kernel.
#
# That is the single reason 6.8b can fail. And the missing piece cannot be recovered from
# anywhere else: `train_features.csv` holds the values AFTER standardisation, so it can
# tell you what the scaled data looks like but not what was subtracted to get there. Two
# lines in Notebook 01 fix it permanently.
#
# This cell prints those lines. It writes nothing itself.
# =====================================================================================
if NB01_READY:
    print("Nothing to do - Notebook 01's fitted state was found at:")
    print(f"  {NB01_STATE_SOURCE}")
    print(f"\n  {len(NB01_STATE['scaler_features'])} scaler moments, "
          f"{len(NB01_STATE['top_amenities_vocab'])} amenity terms, "
          f"{len(NB01_STATE['city_price_stats'])} city price stats.")
else:
    print("=" * 78)
    print("ADD THIS CELL TO NOTEBOOK 01, immediately after section 7's export cell")
    print("=" * 78)
    print(f'''
# --- Persist the fitted preprocessing state --------------------------------------
# Section 6 fills `artifacts` with every train-fitted quantity the pipeline needs at
# inference time. Section 7 exports the matrices but not this, so it is written here.
# Without it, nothing downstream can turn a raw listing into a model-ready row: the
# scaler moments are not recoverable from the already-scaled CSVs.
import json

artifacts["final_feature_columns"] = list(X_train.columns)

_key = f"{{PREFIX}}/processed/01_preprocessing_artifacts.json"
_body = json.dumps(artifacts, indent=2, default=str)

if not LOCAL_MODE and s3 is not None:
    s3.put_object(Bucket=BUCKET, Key=_key, Body=_body.encode("utf-8"),
                  ContentType="application/json")
    print(f"Wrote s3://{{BUCKET}}/{{_key}}")
else:
    import os
    os.makedirs(os.path.join(LOCAL_OUT, "processed"), exist_ok=True)
    _p = os.path.join(LOCAL_OUT, "processed", "01_preprocessing_artifacts.json")
    with open(_p, "w", encoding="utf-8") as fh:
        fh.write(_body)
    print(f"Wrote {{os.path.abspath(_p)}}")

print(f"  scaler moments : {{len(artifacts['scaler_features'])}} columns")
print(f"  amenity vocab  : {{len(artifacts['top_amenities_vocab'])}} terms")
print(f"  city price     : {{len(artifacts['city_price_stats'])}} cities")
print(f"  final columns  : {{len(artifacts['final_feature_columns'])}}")
''')
    print("=" * 78)
    print("Then re-run 6.8b here. It looks for the JSON at:")
    print(f"  s3://{BUCKET}/{BOOKING_PREFIX}/processed/01_preprocessing_artifacts.json")
    print("  ./01_preprocessing_artifacts.json")
    print("  ./processed_instant_booking/processed/01_preprocessing_artifacts.json")
    print()
    print("This is the only blocker. The estimator itself loaded fine; what is missing is")
    print("the arithmetic needed to put a raw listing into the space it was fitted in.")


Nothing to do - Notebook 01's fitted state was found at:
  /home/sagemaker-user/HSAssignment/01_preprocessing_artifacts.json

  19 scaler moments, 30 amenity terms, 10 city price stats.


#### 6.8b-verify Does the transform put the listing in the right space?

The previous version of this cell checked shape, column order, no-NaNs and one-hot
correctness — and passed a feature vector that produced 0.0%. Every check was true and the
answer was still meaningless, because none of them looked at **magnitude**. A
`maximum_nights` of 1125 sitting in a column whose training values run from −1.8 to +1.5
is correctly named, correctly ordered, not missing, and completely wrong.

So the checks are now built around distribution as well as structure:

- every standardised column must land inside the span `train_features.csv` covers — a
  value far outside it has not been scaled;
- the predicted probability must not be **saturated**;
- different listings must get different answers, because a transform that collapses
  everything onto one point also returns a stable, plausible-looking number;
- the estimator's mean prediction over real training rows must sit near the class balance.

Exact checks veto. The distributional ones veto only when unambiguous — an unusual listing
is allowed to be unusual, but a value hundreds of units out is an unscaled one.



In [23]:
# =====================================================================================
# 6.8b-verify  Does the transform put the listing in the right space?
# -------------------------------------------------------------------------------------
# The previous version checked shape, column order, no-NaNs and one-hot correctness --
# and passed a feature vector that produced P(instant) = 0.0%. Every one of those checks
# was true and the answer was still meaningless, because none of them looked at
# MAGNITUDE. A `maximum_nights` of 1125 sitting in a column whose training values run
# from about -1 to +2 is correctly named, correctly ordered, not missing, and completely
# wrong.
#
# So the checks below are built around distribution as well as structure:
#
#   - every standardised column must land inside the span the training matrix covers;
#   - the predicted probability must not be saturated;
#   - different listings must get different answers, because a transform that collapses
#     everything onto one point also returns a stable, plausible-looking number.
#
# Exact checks veto. The distributional ones veto only when unambiguous: a value hundreds
# of units outside the training span is not an unusual listing, it is an unscaled one.
# =====================================================================================
import io   # re-stated so 6.8 can be re-run without Section 6.5

LOGREG_VERIFIED, LOGREG_VERIFY_REPORT = False, []

_expected = list(LOGREG_FITTED_COLUMNS or LOGREG_FEATURE_CONTRACT)
_probe_ui = dict(BOOKING_PRESETS["Professional host"])
_probe = build_booking_payload(_probe_ui)

if LOGREG_MODEL is None:
    LOGREG_VERIFY_REPORT.append(("estimator", "SKIP", "not loaded"))
elif LOGREG_FITTED_COLUMNS:
    _same = LOGREG_FITTED_COLUMNS == LOGREG_FEATURE_CONTRACT
    LOGREG_VERIFY_REPORT.append(
        ("fitted columns", "PASS" if _same else "FAIL",
         "match Notebook 02's printed 100-feature list" if _same else
         "the loaded estimator was fitted on different columns than Notebook 02 printed"))

if TRAIN_SAMPLE is not None:
    _cols_match = list(TRAIN_SAMPLE.columns) == _expected
    LOGREG_VERIFY_REPORT.append(
        ("training matrix", "PASS" if _cols_match else "FAIL",
         f"{len(TRAIN_SAMPLE):,} sampled rows x {TRAIN_SAMPLE.shape[1]} columns, order "
         + ("identical to the fitted contract" if _cols_match else "DIFFERENT")))
else:
    LOGREG_VERIFY_REPORT.append(("training matrix", "SKIP", str(TRAIN_SAMPLE_SOURCE)))

if not NB01_READY:
    LOGREG_VERIFY_REPORT.append(("transform", "SKIP", NB01_STATE_ERROR))
else:
    try:
        _X, _ = nb01_raw_to_features(_probe)

        # --- structure -----------------------------------------------------------------
        LOGREG_VERIFY_REPORT.append(
            ("shape", "PASS" if _X.shape == (1, len(_expected)) else "FAIL",
             f"{_X.shape} vs expected (1, {len(_expected)})"))
        LOGREG_VERIFY_REPORT.append(
            ("column order", "PASS" if list(_X.columns) == _expected else "FAIL",
             "identical to the fitted order"))
        _n_bad = int((~np.isfinite(_X.to_numpy(dtype=float))).sum())
        LOGREG_VERIFY_REPORT.append(
            ("finite values", "PASS" if _n_bad == 0 else "FAIL", f"{_n_bad} NaN/inf"))

        # --- one-hot families, generalised ---------------------------------------------
        # Exactly one hot when the raw value is a fitted level, exactly zero when it is
        # the dropped reference. Reading the expected answer off the contract rather than
        # hard-coding which city sorts first is what lets this survive a re-fit.
        for _col, _raw in (("city", _probe["city"]), ("room_type", _probe["room_type"]),
                           ("host_response_time", _probe["host_response_time"])):
            _fam = [c for c in _expected if c.startswith(f"{_col}_")
                    and not c.endswith("_isna")]
            _hot = [c for c in _fam if float(_X.iloc[0][c]) == 1.0]
            _want = 1 if f"{_col}_{_raw}" in _fam else 0
            _ok = len(_hot) == _want and (not _want or _hot == [f"{_col}_{_raw}"])
            LOGREG_VERIFY_REPORT.append(
                (f"{_col} one-hot", "PASS" if _ok else "FAIL",
                 f"{_raw!r} -> {_hot or 'all zeros (dropped reference level)'}; "
                 f"expected {_want} hot"))

        # --- MAGNITUDE: the check that was missing -------------------------------------
        # Every standardised column has to land inside the span the training matrix
        # covers. A column sitting far outside it has not been standardised, and that is
        # exactly the failure that produced 0.0%.
        if TRAIN_SAMPLE is not None:
            _scaled = [c for c in NB01_STATE["scaler_features"] if c in _TRAIN_RANGES]
            _far = []
            for c in _scaled:
                lo, hi = _TRAIN_RANGES[c]
                v = float(_X.iloc[0][c])
                span = max(hi - lo, 1e-9)
                if v < lo - span or v > hi + span:
                    _far.append((c, round(v, 3), (round(lo, 3), round(hi, 3))))
            LOGREG_VERIFY_REPORT.append(
                ("scaled magnitudes", "PASS" if not _far else "FAIL",
                 f"all {len(_scaled)} standardised columns inside the training span"
                 if not _far else
                 f"{len(_far)} column(s) far outside it - these look UNSCALED: {_far[:4]}"))

            _oor = [c for c in _expected
                    if c in _TRAIN_RANGES
                    and not (_TRAIN_RANGES[c][0] - 1e-9 <= float(_X.iloc[0][c])
                             <= _TRAIN_RANGES[c][1] + 1e-9)]
            LOGREG_VERIFY_REPORT.append(
                ("training range", "PASS" if not _oor else "WARN",
                 "every value inside the training min/max" if not _oor else
                 f"{len(_oor)} column(s) mildly outside it (an unusual listing is allowed "
                 f"to be unusual): {_oor[:5]}"))

        # --- the probability itself ----------------------------------------------------
        if LOGREG_MODEL is not None:
            _p = float(LOGREG_MODEL.predict_proba(_X)[0, 1])
            _sat = _p < 1e-4 or _p > 1 - 1e-4
            LOGREG_VERIFY_REPORT.append(
                ("probability sane", "FAIL" if _sat else "PASS",
                 f"P(instant) = {_p:.4f}"
                 + (" - SATURATED. A linear model returns this when the feature vector "
                    "sits far outside the fitted space, which almost always means a "
                    "scaling step was skipped." if _sat else "")))

            # A transform that collapses every listing onto one point still returns a
            # plausible number, so check that different listings actually differ.
            _probs = []
            for _name, _preset in BOOKING_PRESETS.items():
                try:
                    _pp = build_booking_payload(dict(_preset))
                    _probs.append((_name, float(LOGREG_MODEL.predict_proba(
                        nb01_raw_to_features(_pp)[0])[0, 1])))
                except Exception:
                    pass
            if len(_probs) >= 2:
                _spread = max(p for _, p in _probs) - min(p for _, p in _probs)
                LOGREG_VERIFY_REPORT.append(
                    ("discriminates", "PASS" if _spread > 0.01 else "FAIL",
                     "; ".join(f"{n}={p:.3f}" for n, p in _probs)
                     + f"  (spread {_spread:.3f})"))

            # And the aggregate. If the estimator loaded here is the one Notebook 02
            # reported on, its mean prediction over real training rows should sit near
            # the class balance rather than at an extreme.
            if TRAIN_SAMPLE is not None:
                _n = min(2000, len(TRAIN_SAMPLE))
                _pred = LOGREG_MODEL.predict_proba(TRAIN_SAMPLE.head(_n))[:, 1]
                LOGREG_VERIFY_REPORT.append(
                    ("on real rows", "PASS" if 0.05 < _pred.mean() < 0.95 else "WARN",
                     f"mean P(instant) over {_n:,} real training rows = "
                     f"{_pred.mean():.3f}"))

    except Exception as _e:
        LOGREG_VERIFY_REPORT.append(("transform", "FAIL", f"{type(_e).__name__}: {_e}"))

_results = [s for _, s, _ in LOGREG_VERIFY_REPORT]
LOGREG_VERIFIED = bool(_results) and "FAIL" not in _results and "PASS" in _results

print("Baseline LR transform verification")
print(f"  ground truth : {TRAIN_SAMPLE_SOURCE or 'unavailable'}")
print(f"  fitted state : {NB01_STATE_SOURCE or 'NOT FOUND'}")
print()
for _name, _status, _detail in LOGREG_VERIFY_REPORT:
    print(f"  [{_status:<4}] {_name:<20} {_detail}")

print()
if LOGREG_VERIFIED:
    print("  VERIFIED. The tab may quote a probability.")
elif "FAIL" in _results:
    print("  NOT VERIFIED - the raw-listing mode is DISABLED rather than left quoting a")
    print("  number nobody can defend. Read the first FAIL above:")
    print("    'scaled magnitudes' / 'probability sane' -> the scaler moments are absent")
    print("      or wrong. Run 6.8b-export and check NB01_STATE_SOURCE.")
    print("    a one-hot line -> the contract and the encoder disagree about levels.")
    print("    'transform' -> read the exception; it names the field.")
else:
    print("  INCONCLUSIVE - nothing failed, but nothing could be proved either. The tab")
    print("  stays disabled: an unverified transform is not safer than no answer.")


Baseline LR transform verification
  ground truth : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/processed/train_features.csv
  fitted state : /home/sagemaker-user/HSAssignment/01_preprocessing_artifacts.json

  [PASS] fitted columns       match Notebook 02's printed 100-feature list
  [PASS] training matrix      10,887 sampled rows x 100 columns, order identical to the fitted contract
  [PASS] shape                (1, 100) vs expected (1, 100)
  [PASS] column order         identical to the fitted order
  [PASS] finite values        0 NaN/inf
  [PASS] city one-hot         'Paris' -> ['city_Paris']; expected 1 hot
  [PASS] room_type one-hot    'Entire home/apt' -> all zeros (dropped reference level); expected 0 hot
  [PASS] host_response_time one-hot 'within an hour' -> ['host_response_time_within an hour']; expected 1 hot
  [PASS] scaled magnitudes    all 19 standardised columns inside the training span
  [PASS] training range       every value inside the training min

#### 6.8c The handler

Same four returns as `predict_instant_booking`, same `build_booking_payload`, same `validate_booking`, same derived-features table. With the comparison box ticked it also posts the identical payload to the champion endpoint, so the two probabilities appear on one card — and because both were scored on the same test split, the card can put their ROC-AUCs side by side without a caveat.

In [24]:
# =====================================================================================
# 6.8c  The baseline LogisticRegression handler
# -------------------------------------------------------------------------------------
# Same four returns as `predict_instant_booking`, same `build_booking_payload`, same
# `validate_booking`, same derived-features table. Two things differ, and both are the
# point of the tab: there is no round trip (the estimator is in this kernel), and the
# result card can put the champion endpoint's answer next to its own.
#
# Both models are scored on the SAME test split here -- 55,629 rows, from the same
# Notebook 01 partition -- so unlike the price tabs, these numbers are directly
# comparable and the card says so.
# =====================================================================================

# Notebook 02's own numbers, quoted as it reported them. Baseline: "RUN 1: LOGISTIC
# REGRESSION BASELINE". Champion: the tuned RF parent run, test split.
LOGREG_SCORES = {
    "label": "LogisticRegression (C=1.0, L2)", "run": LOGREG_RUN_NAME,
    "accuracy": 0.7049, "roc_auc": 0.7612, "pr_auc": 0.6895, "f1": 0.6069,
    "precision": 0.6726, "recall": 0.5529, "log_loss": 0.5708, "brier": 0.1936,
    "split": "test, 55,629 rows", "trained_on": "222,513 rows",
}
RF_CHAMPION_SCORES = {
    "label": "RandomForest (tuned: 150 trees, depth 20)",
    "run": "team14_s1401_random_forest_tuning",
    "accuracy": 0.7657, "roc_auc": 0.8383, "pr_auc": 0.8111, "f1": 0.6841,
    "precision": 0.7698, "recall": 0.6155, "log_loss": 0.4873, "brier": 0.1605,
    "split": "test, 55,629 rows", "trained_on": "222,513 rows",
}

LOGREG_UI_FIELDS = [*BOOKING_UI_FIELDS, "compare_with_champion"]

# True only when every gate in 6.8a/6.8b/6.8b-verify passed.
LOGREG_TAB_READY = (LOGREG_MODEL is not None
                    and BOOKING_PREPROC is not None
                    and LOGREG_VERIFIED)


def logreg_predict_raw(payload):
    """One raw booking payload -> the record shape the classifier endpoint returns."""
    t0 = time.perf_counter()
    X, _ = booking_raw_to_features(payload)
    prob = float(LOGREG_MODEL.predict_proba(X)[0, 1])
    pred = int(LOGREG_MODEL.predict(X)[0])
    ms = (time.perf_counter() - t0) * 1000.0
    return {
        "prediction": pred,
        "probability_class_1": round(prob, 6),
        "class_1": round(prob, 6),
        "class_0": round(1.0 - prob, 6),
        "model": LOGREG_TYPE,
        "served_by": "in-process (no endpoint)",
    }, ms


def _logreg_unavailable_card():
    reasons = []
    if LOGREG_MODEL is None:
        reasons.append(f"- **Estimator**: {LOGREG_ERROR}")
    if BOOKING_PREPROC is None:
        reasons.append(f"- **Preprocessor**: {BOOKING_PREPROC_ERROR}")
    if LOGREG_MODEL is not None and BOOKING_PREPROC is not None and not LOGREG_VERIFIED:
        failed = [f"`{n}` — {d}" for n, s, d in LOGREG_VERIFY_REPORT if s == "FAIL"]
        reasons.append("- **Verification**: the transform loaded but could not be proved "
                       "correct, so the tab will not quote a probability.\n"
                       + "\n".join(f"    - {f}" for f in failed))
    attempts = "\n".join(f"  - `{n}` -> {o}" for n, o in LOGREG_ATTEMPTS)
    return ("### Baseline LogisticRegression unavailable\n\n" + "\n".join(reasons) +
            "\n\nEstimator load attempts:\n" + attempts +
            "\n\nFix it in Section 6.8a/6.8b: point `LOGREG_S3_KEY` at your mirror, add "
            "a path to `LOGREG_LOCAL_CANDIDATES`, set `BOOKING_PREPROC_LOCAL` to the "
            "classifier's `preprocess.py`, or run Section 12's publish cell.")


def _prob_bar(prob):
    return (f"<div class='bar'><div class='fill' style='width:{prob * 100:.1f}%'></div>"
            f"<div class='mid'></div></div>")


def _logreg_comparison_block(base_rec, champ_rec, champ_error):
    """The two probabilities side by side, plus the honest caveats."""
    if champ_error:
        return ("<div class='cmp'><div class='cmp-head'>Champion comparison</div>"
                f"<div class='cmp-note'>The champion endpoint did not answer: "
                f"{champ_error}<br>The baseline result above is unaffected - it never "
                "leaves this kernel.</div></div>")

    b_prob = base_rec["probability_class_1"]
    c_prob = champ_rec.get("probability_class_1")
    if c_prob is None:
        return ("<div class='cmp'><div class='cmp-head'>Champion comparison</div>"
                "<div class='cmp-note'>The endpoint returned a label but no probability, "
                "so there is nothing to compare against.</div></div>")

    b_lab = "Instant" if base_rec["prediction"] == 1 else "Manual"
    c_lab = "Instant" if champ_rec["prediction"] == 1 else "Manual"
    agree = base_rec["prediction"] == champ_rec["prediction"]

    rows = "".join(
        f"<tr><td class='id'>{name}</td><td>{lab}</td><td>{p:.1%}</td>"
        f"<td>{auc:.4f}</td></tr>"
        for name, lab, p, auc in (
            (LOGREG_SCORES["label"], b_lab, b_prob, LOGREG_SCORES["roc_auc"]),
            (RF_CHAMPION_SCORES["label"], c_lab, c_prob, RF_CHAMPION_SCORES["roc_auc"])))

    delta = c_prob - b_prob
    if agree:
        verdict = (f"Both models land on <strong>{b_lab.lower()}</strong>. They still "
                   f"disagree about how sure to be, by {abs(delta):.1%}.")
    else:
        verdict = ("<strong>The two models disagree on the label.</strong> On the test "
                   "split the champion is right more often, but a disagreement this "
                   "close to 0.5 is worth reading as \"the features do not settle it\" "
                   "rather than as one model being wrong.")

    return (
        "<div class='cmp'>"
        "<div class='cmp-head'>Baseline vs champion &middot; identical payload</div>"
        "<table class='rank'><thead><tr><th>Model</th><th>Label</th>"
        "<th>P(instant)</th><th>Test ROC-AUC</th></tr></thead>"
        f"<tbody>{rows}</tbody></table>"
        f"<div class='cmp-note'>{verdict}<br><br>"
        f"Both rows were scored on the <strong>same 55,629-row test split</strong> from "
        f"the same Notebook 01 partition, so the ROC-AUC column is a like-for-like "
        f"comparison - {RF_CHAMPION_SCORES['roc_auc']:.4f} against "
        f"{LOGREG_SCORES['roc_auc']:.4f}, a gain of "
        f"{RF_CHAMPION_SCORES['roc_auc'] - LOGREG_SCORES['roc_auc']:.4f}. What the "
        f"columns do not show is that the champion is also better calibrated "
        f"(Brier {RF_CHAMPION_SCORES['brier']:.4f} vs {LOGREG_SCORES['brier']:.4f}), "
        f"which is what makes its probability, not just its label, worth reading."
        "</div></div>")


def predict_instant_booking_baseline(*values):
    # Gradio click handler. Returns (result card, derived table, request JSON, records).
    ui = dict(zip(LOGREG_UI_FIELDS, values))

    try:
        payload = build_booking_payload(ui)
    except (TypeError, ValueError) as e:
        return (f"### Cannot build the request\n\n{e}\n\nCheck the numeric fields.",
                "", "", "")

    request_json = json.dumps(payload, indent=2)

    if not LOGREG_TAB_READY:
        return _logreg_unavailable_card(), "", request_json, ""

    problems = validate_booking(payload)
    if problems:
        bullets = "\n".join(f"- {p}" for p in problems)
        return (f"### Fix these before scoring\n\n{bullets}", "", request_json, "")

    try:
        rec, ms = logreg_predict_raw(payload)
    except Exception as e:
        return (f"### The baseline could not score this listing\n\n`{type(e).__name__}: "
                f"{e}`\n\nThis is a transform failure, not an endpoint failure - nothing "
                f"left this kernel.", booking_derived_preview(payload), request_json, "")

    prob = rec["probability_class_1"]
    label = "Instant bookable" if rec["prediction"] == 1 else "Not instant bookable"
    margin = abs(prob - 0.5)
    confidence = ("barely past the 0.5 boundary - treat this label as a coin flip"
                  if margin < 0.10 else
                  "clear of the boundary" if margin > 0.25 else
                  "moderately clear of the boundary")

    records = [rec]
    comparison = ""
    if ui.get("compare_with_champion"):
        try:
            champ_records, champ_ms = invoke_booking_endpoint(payload)
            comparison = _logreg_comparison_block(rec, champ_records[0], None)
            records.append(dict(champ_records[0], served_by=BOOKING_ENDPOINT,
                                round_trip_ms=round(champ_ms, 1)))
        except Exception as e:
            comparison = _logreg_comparison_block(rec, None, f"{type(e).__name__}: {e}")

    convergence = ""
    if LOGREG_N_ITER is not None:
        _it = int(np.asarray(LOGREG_N_ITER).ravel()[0])
        if _it >= int(getattr(LOGREG_MODEL, "max_iter", 1000)):
            convergence = (
                "<div class='cmp-note'><strong>Caveat.</strong> Notebook 02's lbfgs run "
                f"stopped at the iteration limit ({_it}) rather than at a solution, and "
                "warned that the data should be scaled. These coefficients are still the "
                "ones that scored 0.7612 test ROC-AUC, so the comparison is honest - but "
                "this is a baseline that was never given its best shot, which flatters "
                "the champion slightly.</div>")

    card = (
        "<div class='result'>"
        f"<div class='eyebrow'>Baseline &middot; instant-booking likelihood &middot; "
        f"{payload['city']}</div>"
        f"<div class='price'>{label}</div>"
        f"<div class='band'>P(instant bookable) = <strong>{prob:.1%}</strong>, "
        f"{confidence}.{_prob_bar(prob)}"
        f"<small>0% &middot; the marker is the 0.5 decision boundary &middot; 100%</small>"
        f"</div>"
        f"{comparison}{convergence}"
        f"<div class='meta'>class {rec['prediction']} &middot; {ms:,.1f} ms in-process "
        f"&middot; {LOGREG_TYPE} &middot; no endpoint</div>"
        "</div>")

    return (card, booking_derived_preview(payload), request_json,
            json.dumps(records, indent=2))


print(f"Baseline LR handler ready. {len(LOGREG_UI_FIELDS)} UI controls -> "
      f"{len(BOOKING_FIELDS)} raw fields -> "
      f"{len(LOGREG_FITTED_COLUMNS or LOGREG_FEATURE_CONTRACT)} columns -> in-process")
print(f"  tab enabled: {LOGREG_TAB_READY}")


Baseline LR handler ready. 28 UI controls -> 26 raw fields -> 100 columns -> in-process
  tab enabled: True


## 7. Launch the Gradio UI

Run the cell below to start the front-end UI inside the notebook.

The 27 fields are grouped into five tabs — Listing, Amenities, Booking rules, Host, Reviews — because
a flat wall of 27 inputs is unusable. Picking a city snaps the coordinates to that city's centre and
reloads the neighbourhood list, so the most common way to produce a `NaN` (a coordinate that belongs
to no trained market) cannot happen by accident.

> If the UI does not appear inline, check the printed local URL. In managed Studio environments,
> external sharing may be disabled, so `share=True` may need to become `share=False`.

In [25]:
CSS = '''
.result {padding: 20px 22px; border: 1px solid var(--border-color-primary);
         border-radius: 10px; background: var(--background-fill-secondary);}
.result .eyebrow {font-size: .78rem; letter-spacing: .09em; text-transform: uppercase;
         opacity: .65; margin-bottom: 6px;}
.result .price {font-size: 2.6rem; font-weight: 650; line-height: 1.1;
         font-variant-numeric: tabular-nums;}
.result .price .ccy {font-size: 1.1rem; font-weight: 500; opacity: .7; margin-left: 4px;}
.result .band {margin-top: 12px; padding: 10px 12px; border-radius: 8px;
         background: var(--background-fill-primary); font-size: .9rem; line-height: 1.5;}
.result .meta {margin-top: 12px; font-size: .78rem; opacity: .6;
         font-variant-numeric: tabular-nums;}
.result .bar {position: relative; height: 10px; border-radius: 5px; margin: 10px 0 6px;
         background: var(--background-fill-secondary); overflow: hidden;}
.result .bar .fill {height: 100%; background: var(--color-accent, #f59e0b);}
.result .bar .mid {position: absolute; left: 50%; top: -2px; width: 2px; height: 14px;
         background: var(--body-text-color); opacity: .55;}
.result table.rank {width: 100%; border-collapse: collapse; margin-top: 14px;
         font-size: .86rem; font-variant-numeric: tabular-nums;}
.result table.rank th {text-align: right; padding: 6px 8px; font-size: .7rem; font-weight: 600;
         letter-spacing: .05em; text-transform: uppercase; opacity: .6;
         border-bottom: 1px solid var(--border-color-primary);}
.result table.rank td {text-align: right; padding: 7px 8px;
         border-bottom: 1px solid var(--border-color-primary);}
.result table.rank th:first-child, .result table.rank td:first-child {text-align: left;
         opacity: .5; width: 2.2rem;}
.result table.rank td.id {text-align: left; font-family: var(--font-mono, monospace);
         font-size: .82rem;}
.result table.rank tbody tr:last-child td {border-bottom: none;}
.result table.rank td.deal {font-weight: 600;}
.result table.rank td.deal.good {color: #16a34a;}
.result table.rank td.deal.bad {opacity: .55;}
.result .cmp {margin-top: 14px; padding: 12px 14px; border-radius: 8px;
         background: var(--background-fill-primary);
         border: 1px solid var(--border-color-primary);}
.result .cmp .cmp-head {font-size: .7rem; font-weight: 600; letter-spacing: .05em;
         text-transform: uppercase; opacity: .6;}
.result .cmp .cmp-note {margin-top: 10px; font-size: .84rem; line-height: 1.5; opacity: .85;}
.result .cmp table.rank {margin-top: 8px;}
'''

rdef = review_defaults()

try:
    status = sm.describe_endpoint(EndpointName=PRICE_ENDPOINT)["EndpointStatus"]
except Exception:
    status = "NotFound"
status_note = {
    "InService": "endpoint is InService",
    "Creating": "endpoint is still provisioning - predictions will fail until it is InService",
    "NotFound": f"no endpoint named `{PRICE_ENDPOINT}` in `{REGION}`",
}.get(status, f"endpoint status: {status}")

fitted_note = (f"UI configured from `{ARTIFACTS_SOURCE}`" if ARTIFACTS
               else "`preprocessing_artifacts.json` not found - dropdowns use built-in defaults, "
                    "which may not match the frozen encoder levels")

try:
    booking_status = sm.describe_endpoint(EndpointName=BOOKING_ENDPOINT)["EndpointStatus"]
except Exception:
    booking_status = "NotFound"

if BASELINE_MODEL is None or PREPROCESSOR is None:
    BASELINE_STATUS_NOTE = "**unavailable** (see Section 6.7)"
else:
    BASELINE_STATUS_NOTE = f"**{BASELINE_TYPE}** loaded in-process, no endpoint"

if not LOGREG_TAB_READY:
    LOGREG_STATUS_NOTE = "**unavailable** (see Section 6.8)"
else:
    LOGREG_STATUS_NOTE = (f"**{LOGREG_TYPE}** loaded in-process, no endpoint, transform "
                          "verified")

if RECSYS_DF is None:
    RECSYS_STATUS_NOTE = f"**not loaded** ({RECSYS_ERROR})"
else:
    RECSYS_STATUS_NOTE = (f"**{len(RECSYS_DF):,}** listings scored offline by both models"
                          + ("" if RECSYS_LIVE_OK else ", live re-scoring unavailable"))

def build_price_panel(c):
    """Populate `c` with one control per UI_FIELDS entry.

    A copy of the champion tab's left-hand column, factored out so the baseline tab is a
    genuine mirror rather than a hand-maintained near-copy. Written as a function, not a
    loop over a spec, because the controls differ in kind: a slider whose maximum depends
    on the fitted median is not a dropdown with a different label.
    """
    with gr.Tab("Listing"):
        with gr.Row():
            c["city"] = gr.Dropdown(
                label="City", choices=CITIES, value="Paris",
                info="One of the 10 trained markets; sets the currency and the "
                     "city-centre reference point")
            c["neighbourhood"] = gr.Dropdown(
                label="Neighbourhood", choices=known_neighbourhoods("Paris") or [],
                value="Buttes-Montmartre", allow_custom_value=True,
                info="Type any value; unseen pairs get frequency 0.0")
        with gr.Row():
            c["latitude"] = gr.Number(label="Latitude", value=48.8867)
            c["longitude"] = gr.Number(label="Longitude", value=2.3431)
        gr.Markdown(
            "<small>Coordinates are used to derive distance from the city centre, then "
            "coarsened to 2 decimal places and discarded. Exact GPS never reaches a "
            "stored artifact.</small>")
        with gr.Row():
            c["property_type"] = gr.Dropdown(
                label="Property type", choices=FALLBACK_PROPERTY_TYPES,
                value="Private room in apartment", allow_custom_value=True,
                info="Free text; mapped to one of 7 governed groups")
            c["room_type"] = gr.Dropdown(
                label="Room type", choices=room_type_choices(), value="Private room")
        with gr.Row():
            c["accommodates"] = gr.Number(label="Accommodates", value=2, precision=0, minimum=1)
            c["bedrooms"] = gr.Number(label="Bedrooms", value=1, precision=0,
                                      minimum=0, maximum=BEDROOMS_CAP)
            c["bedrooms_unknown"] = gr.Checkbox(label="Bedrooms not stated", value=False)

    with gr.Tab("Amenities"):
        c["amenities_selected"] = gr.CheckboxGroup(
            label="Amenities in the model's vocabulary",
            choices=amenity_choices(), value=[],
            info="Each of these is a one-hot flag in the 88-column matrix")
        c["amenities_extra"] = gr.Textbox(
            label="Other amenities", lines=2, placeholder="balcony, sea view, piano",
            info="Comma separated. These do not get their own flag, but they raise "
                 "amenity_count and rare_amenity_count.")

    with gr.Tab("Booking rules"):
        with gr.Row():
            c["minimum_nights"] = gr.Number(
                label="Minimum nights", value=2, precision=0, minimum=1,
                info=f"Winsorised at {MIN_NIGHTS_CAP}; >= 28 sets is_monthly_min")
            c["maximum_nights"] = gr.Number(
                label="Maximum nights", value=90, precision=0, minimum=1,
                info=f"Winsorised at {MAX_NIGHTS_CAP}")
        c["instant_bookable"] = gr.Checkbox(label="Instant bookable", value=False)

    with gr.Tab("Host"):
        with gr.Row():
            c["host_since"] = gr.Textbox(
                label="Host since", value="2015-06-12", placeholder="YYYY-MM-DD",
                info=f"Tenure is measured against the {SNAPSHOT_DATE:%Y-%m-%d} data snapshot")
            c["host_since_unknown"] = gr.Checkbox(label="Not known", value=False)
        with gr.Row():
            c["host_is_superhost"] = gr.Checkbox(label="Superhost", value=False)
            c["host_has_profile_pic"] = gr.Checkbox(label="Has profile picture", value=True)
            c["host_identity_verified"] = gr.Checkbox(label="Identity verified", value=True)
        with gr.Row():
            c["host_total_listings_count"] = gr.Number(
                label="Total listings by this host", value=1, precision=0, minimum=0)
            c["host_response_time"] = gr.Dropdown(
                label="Response time", choices=response_time_choices(),
                value="within a few hours")
        with gr.Row():
            c["host_response_rate"] = gr.Slider(
                label="Response rate (%)", minimum=0, maximum=100, step=1, value=90)
            c["host_response_rate_known"] = gr.Checkbox(label="Known", value=True)
        with gr.Row():
            c["host_acceptance_rate"] = gr.Slider(
                label="Acceptance rate (%)", minimum=0, maximum=100, step=1, value=80)
            c["host_acceptance_rate_known"] = gr.Checkbox(label="Known", value=True)

    with gr.Tab("Reviews"):
        c["has_reviews"] = gr.Checkbox(
            label="This listing has reviews", value=True,
            info="Unchecked sends nulls for all seven scores, which sets "
                 "has_review_scores = 0 and imputes the fitted medians")
        for col in REVIEW_COLS:
            c[col] = gr.Slider(
                label=col.replace("review_scores_", "").replace("_", " ").title(),
                minimum=0, maximum=review_slider_max(col, rdef[col]),
                step=1 if col == "review_scores_rating" else 0.1,
                value=rdef[col])

    return c


def build_booking_panel(b):
    """Populate `b` with one control per BOOKING_UI_FIELDS entry.

    Factored out of the champion classifier tab so the baseline LR tab is a genuine
    mirror rather than a hand-maintained near-copy. Both classification tabs now read
    from one definition: if a control moves, it moves in both, and the two tabs cannot
    drift into sending different values for the same listing.

    Note what is NOT here: the comparison checkbox. That is the reader's setting, not
    the listing's, so it belongs to the tab that owns it.
    """
    with gr.Tab("Listing"):
        with gr.Row():
            b["city"] = gr.Dropdown(
                label="City", choices=CITIES, value="Paris",
                allow_custom_value=True,
                info="An unseen city gives log_price_city_zscore = 0.0")
            b["price"] = gr.Number(
                label="Advertised nightly price", value=150.0, minimum=1,
                info="An INPUT for this model, standardised within city")
        with gr.Row():
            b["latitude"] = gr.Number(label="Latitude", value=48.8867)
            b["longitude"] = gr.Number(label="Longitude", value=2.3431)
        with gr.Row():
            b["property_type"] = gr.Dropdown(
                label="Property type", choices=FALLBACK_PROPERTY_TYPES,
                value="Entire apartment", allow_custom_value=True,
                info="Kept if in the top-K list, else grouped to 'Other'")
            b["room_type"] = gr.Dropdown(
                label="Room type", choices=BOOKING_ROOM_TYPES,
                value="Entire home/apt")
        with gr.Row():
            b["accommodates"] = gr.Number(label="Accommodates", value=4,
                                          precision=0, minimum=1)
            b["bedrooms"] = gr.Number(label="Bedrooms", value=2, precision=0,
                                      minimum=1)

    with gr.Tab("Amenities"):
        b["amenities_selected"] = gr.CheckboxGroup(
            label="Amenities", choices=amenity_choices(), value=[],
            info="Matched against this model's own 80-term vocabulary")
        b["amenities_extra"] = gr.Textbox(
            label="Other amenities", lines=2, placeholder="balcony, sea view")

    with gr.Tab("Booking rules"):
        with gr.Row():
            b["minimum_nights"] = gr.Number(label="Minimum nights", value=2,
                                            precision=0, minimum=1)
            b["maximum_nights"] = gr.Number(label="Maximum nights", value=1125,
                                            precision=0, minimum=1)

    with gr.Tab("Host"):
        with gr.Row():
            b["host_since"] = gr.Textbox(
                label="Host since", value="2018-05-15", placeholder="YYYY-MM-DD",
                info=f"Tenure measured against {BOOKING_SNAPSHOT:%Y-%m-%d} here")
            b["host_total_listings_count"] = gr.Number(
                label="Total listings by this host", value=12, precision=0,
                minimum=0)
        b["host_response_time"] = gr.Dropdown(
            label="Response time", choices=response_time_choices(),
            value="within an hour")
        b["host_response_rate"] = gr.Slider(
            label="Response rate (%)", minimum=0, maximum=100, step=1, value=100,
            info="Shown 0-100, sent as 0-1 to match this model's training scale")
        b["host_acceptance_rate"] = gr.Slider(
            label="Acceptance rate (%)", minimum=0, maximum=100, step=1, value=95,
            info="Shown 0-100, sent as 0-1")
        with gr.Row():
            b["host_is_superhost"] = gr.Checkbox(label="Superhost", value=True)
            b["host_has_profile_pic"] = gr.Checkbox(label="Has profile picture",
                                                    value=True)
            b["host_identity_verified"] = gr.Checkbox(label="Identity verified",
                                                      value=True)

    with gr.Tab("Reviews"):
        gr.Markdown(
            "<small>This model never rescaled the review columns in training, so "
            "these use the raw 0-100 / 0-10 scale. Verify against "
            "<code>train_imputations</code> in its own "
            "preprocessing_artifacts.json.</small>")
        for col in REVIEW_COLS:
            b[col] = gr.Slider(
                label=col.replace("review_scores_", "").replace("_", " ").title(),
                minimum=0, maximum=review_slider_max(col, rdef[col]),
                step=1 if col == "review_scores_rating" else 0.1, value=rdef[col])

    return b



with gr.Blocks(title=f"{COURSE} {TEAM_ID} - model endpoints", css=CSS) as demo:
    gr.Markdown(
        f'''
        # {COURSE} {TEAM_ID} - deployed model endpoints

        **{STUDENT_ID}** &nbsp;&middot;&nbsp; region `{REGION}`
        &nbsp;&middot;&nbsp; price `{PRICE_ENDPOINT}` is **{status}**
        &nbsp;&middot;&nbsp; booking `{BOOKING_ENDPOINT}` is **{booking_status}**
        &nbsp;&middot;&nbsp; recommender corpus: {RECSYS_STATUS_NOTE}
        &nbsp;&middot;&nbsp; ridge baseline: {BASELINE_STATUS_NOTE}
        &nbsp;&middot;&nbsp; logreg baseline: {LOGREG_STATUS_NOTE}

        Five tabs, three of them deployed. The price and booking tabs post raw fields as JSON to
        their own endpoint and render what comes back; neither trains or preprocesses anything, and
        their two contracts are **not** interchangeable - see Section 5.3 for where their conventions
        disagree.

        The two **Baseline** tabs are the odd ones out: nothing is deployed, so each estimator is loaded
        from MLflow into this kernel and scored here, using the champion's own fitted preprocessor to
        get from 27 raw fields to 88 columns. **Baseline LogReg** does the same thing on the
        classification side: Notebook 02's LogisticRegression, scored here, with the booking
        endpoint's own transform turning 26 raw fields into 100 columns. Sections 6.7 and 6.8
        explain why borrowing those preprocessors - rather than reimplementing them - is what makes
        the comparisons mean anything at all. The LogReg pair is the cleaner of the two: both
        models were scored on the same test split, so their ROC-AUCs can be subtracted.

        The third tab is a different shape again. It ranks a whole city against a search rather than
        scoring one listing, and it uses the other two models' outputs as its features. By default
        it reads those from a batch-scored corpus rather than calling either endpoint - Section 6.5
        explains why that distinction is the whole design, and how to turn it off.

        {fitted_note}.
        '''
    )

    with gr.Tabs():
        with gr.Tab("Airbnb price (regression)"):
            c = {}

            with gr.Row():
                with gr.Column(scale=3):

                    with gr.Tab("Listing"):
                        with gr.Row():
                            c["city"] = gr.Dropdown(
                                label="City", choices=CITIES, value="Paris",
                                info="One of the 10 trained markets; sets the currency and the "
                                     "city-centre reference point")
                            c["neighbourhood"] = gr.Dropdown(
                                label="Neighbourhood", choices=known_neighbourhoods("Paris") or [],
                                value="Buttes-Montmartre", allow_custom_value=True,
                                info="Type any value; unseen pairs get frequency 0.0")
                        with gr.Row():
                            c["latitude"] = gr.Number(label="Latitude", value=48.8867)
                            c["longitude"] = gr.Number(label="Longitude", value=2.3431)
                        gr.Markdown(
                            "<small>Coordinates are used to derive distance from the city centre, then "
                            "coarsened to 2 decimal places and discarded. Exact GPS never reaches a "
                            "stored artifact.</small>")
                        with gr.Row():
                            c["property_type"] = gr.Dropdown(
                                label="Property type", choices=FALLBACK_PROPERTY_TYPES,
                                value="Private room in apartment", allow_custom_value=True,
                                info="Free text; mapped to one of 7 governed groups")
                            c["room_type"] = gr.Dropdown(
                                label="Room type", choices=room_type_choices(), value="Private room")
                        with gr.Row():
                            c["accommodates"] = gr.Number(label="Accommodates", value=2, precision=0, minimum=1)
                            c["bedrooms"] = gr.Number(label="Bedrooms", value=1, precision=0,
                                                      minimum=0, maximum=BEDROOMS_CAP)
                            c["bedrooms_unknown"] = gr.Checkbox(label="Bedrooms not stated", value=False)

                    with gr.Tab("Amenities"):
                        c["amenities_selected"] = gr.CheckboxGroup(
                            label="Amenities in the model's vocabulary",
                            choices=amenity_choices(), value=[],
                            info="Each of these is a one-hot flag in the 88-column matrix")
                        c["amenities_extra"] = gr.Textbox(
                            label="Other amenities", lines=2, placeholder="balcony, sea view, piano",
                            info="Comma separated. These do not get their own flag, but they raise "
                                 "amenity_count and rare_amenity_count.")

                    with gr.Tab("Booking rules"):
                        with gr.Row():
                            c["minimum_nights"] = gr.Number(
                                label="Minimum nights", value=2, precision=0, minimum=1,
                                info=f"Winsorised at {MIN_NIGHTS_CAP}; >= 28 sets is_monthly_min")
                            c["maximum_nights"] = gr.Number(
                                label="Maximum nights", value=90, precision=0, minimum=1,
                                info=f"Winsorised at {MAX_NIGHTS_CAP}")
                        c["instant_bookable"] = gr.Checkbox(label="Instant bookable", value=False)

                    with gr.Tab("Host"):
                        with gr.Row():
                            c["host_since"] = gr.Textbox(
                                label="Host since", value="2015-06-12", placeholder="YYYY-MM-DD",
                                info=f"Tenure is measured against the {SNAPSHOT_DATE:%Y-%m-%d} data snapshot")
                            c["host_since_unknown"] = gr.Checkbox(label="Not known", value=False)
                        with gr.Row():
                            c["host_is_superhost"] = gr.Checkbox(label="Superhost", value=False)
                            c["host_has_profile_pic"] = gr.Checkbox(label="Has profile picture", value=True)
                            c["host_identity_verified"] = gr.Checkbox(label="Identity verified", value=True)
                        with gr.Row():
                            c["host_total_listings_count"] = gr.Number(
                                label="Total listings by this host", value=1, precision=0, minimum=0)
                            c["host_response_time"] = gr.Dropdown(
                                label="Response time", choices=response_time_choices(),
                                value="within a few hours")
                        with gr.Row():
                            c["host_response_rate"] = gr.Slider(
                                label="Response rate (%)", minimum=0, maximum=100, step=1, value=90)
                            c["host_response_rate_known"] = gr.Checkbox(label="Known", value=True)
                        with gr.Row():
                            c["host_acceptance_rate"] = gr.Slider(
                                label="Acceptance rate (%)", minimum=0, maximum=100, step=1, value=80)
                            c["host_acceptance_rate_known"] = gr.Checkbox(label="Known", value=True)

                    with gr.Tab("Reviews"):
                        c["has_reviews"] = gr.Checkbox(
                            label="This listing has reviews", value=True,
                            info="Unchecked sends nulls for all seven scores, which sets "
                                 "has_review_scores = 0 and imputes the fitted medians")
                        for col in REVIEW_COLS:
                            c[col] = gr.Slider(
                                label=col.replace("review_scores_", "").replace("_", " ").title(),
                                minimum=0, maximum=review_slider_max(col, rdef[col]),
                                step=1 if col == "review_scores_rating" else 0.1,
                                value=rdef[col])

                with gr.Column(scale=2):
                    gr.Markdown("### Example listings")
                    preset_buttons = {name: gr.Button(name, size="sm") for name in PRESETS}
                    predict_button = gr.Button("Get suggested rate", variant="primary", size="lg")
                    result = gr.HTML(
                        "<div class='result'><div class='eyebrow'>No request sent yet</div>"
                        "<div class='meta'>Fill in the listing, or load an example, then get a suggested "
                        "rate. The first call after an idle period takes longer: the serverless endpoint "
                        "scales to zero, so it has to pull the container and deserialize the pipeline."
                        "</div></div>")
                    with gr.Accordion("What the endpoint derives from this request", open=False):
                        derived = gr.Markdown()
                    with gr.Accordion("Request JSON (the 27-field contract)", open=False):
                        request_view = gr.Code(language="json", label="POST body")
                    with gr.Accordion("Endpoint response", open=False):
                        response_view = gr.Code(language="json", label="Response")

            inputs = [c[k] for k in UI_FIELDS]
            outputs = [result, derived, request_view, response_view]

            predict_button.click(fn=predict_airbnb_price, inputs=inputs, outputs=outputs)

            # City drives the currency, the centre point and the neighbourhood list.
            def on_city(city):
                lat, lon = CITY_CENTERS.get(city, (0.0, 0.0))
                choices = known_neighbourhoods(city)
                return (gr.update(choices=choices, value=(choices[0] if choices else None)),
                        gr.update(value=lat),
                        gr.update(value=lon))

            c["city"].change(fn=on_city, inputs=c["city"],
                             outputs=[c["neighbourhood"], c["latitude"], c["longitude"]])

            def make_loader(name):
                def _load():
                    p = PRESETS[name]
                    return [gr.update(value=p[k]) for k in UI_FIELDS]
                return _load

            for _name, _button in preset_buttons.items():
                _button.click(fn=make_loader(_name), inputs=None, outputs=inputs)

        with gr.Tab("Baseline Ridge (regression)"):
            bl = {}
            gr.Markdown(
                f'''
                The **statistical floor** every other model had to beat: `RidgeCV`, fitted in
                Notebook 02 section 4 on the same 88-column matrix, alpha `0.01` chosen by
                3-fold CV. Same inputs as the champion tab, same contract, same derived
                features — a different estimator on the end, and no endpoint.

                It runs **in this kernel**. There is no `model.tar.gz` and nothing to invoke:
                the estimator was loaded from `{BASELINE_SOURCE or "— unavailable —"}`, and the
                27 raw fields are turned into 88 columns here, by the *champion's own fitted
                preprocessor*, pulled out of the deployed pipeline. That is deliberate. If this
                tab engineered its own features, the price gap between the two tabs would be
                measuring my transform code rather than the two models.
                '''
            )

            with gr.Row():
                with gr.Column(scale=3):
                    build_price_panel(bl)

                with gr.Column(scale=2):
                    gr.Markdown("### Example listings")
                    bl_preset_buttons = {name: gr.Button(name, size="sm") for name in PRESETS}
                    bl["compare_with_champion"] = gr.Checkbox(
                        label="Compare against the champion endpoint", value=True,
                        info="Sends the identical payload to "
                             f"{PRICE_ENDPOINT} as well, and shows both answers")
                    bl_predict = gr.Button("Get baseline rate", variant="primary", size="lg")
                    bl_result = gr.HTML(
                        "<div class='result'><div class='eyebrow'>No prediction yet</div>"
                        "<div class='meta'>Fill in the listing, or load an example. This tab does "
                        "not call an endpoint, so there is no cold start — but with the comparison "
                        "box ticked it waits on the champion endpoint, which does have one."
                        "</div></div>")
                    with gr.Accordion("What the preprocessor derives from this request", open=False):
                        bl_derived = gr.Markdown()
                    with gr.Accordion("Request JSON (the same 27-field contract)", open=False):
                        bl_request_view = gr.Code(language="json", label="Payload")
                    with gr.Accordion("Prediction record", open=False):
                        bl_response_view = gr.Code(language="json", label="Records")

            bl_preset_inputs = [bl[k] for k in UI_FIELDS]
            bl_inputs = [bl[k] for k in BASELINE_UI_FIELDS]
            bl_outputs = [bl_result, bl_derived, bl_request_view, bl_response_view]

            bl_predict.click(fn=predict_airbnb_price_baseline, inputs=bl_inputs,
                             outputs=bl_outputs)

            bl["city"].change(fn=on_city, inputs=bl["city"],
                              outputs=[bl["neighbourhood"], bl["latitude"], bl["longitude"]])

            def make_baseline_loader(name):
                def _load():
                    p = PRESETS[name]
                    return [gr.update(value=p[k]) for k in UI_FIELDS]
                return _load

            for _bln, _blb in bl_preset_buttons.items():
                # Presets fill the 27 contract controls only; the comparison checkbox is
                # the reader's setting, not the listing's, so a preset must not reset it.
                _blb.click(fn=make_baseline_loader(_bln), inputs=None, outputs=bl_preset_inputs)

        with gr.Tab("Instant booking (classification)"):
            b = {}
            gr.Markdown(
                f'''
                Predicts whether a host will enable **instant booking** for a listing.
                Endpoint `{BOOKING_ENDPOINT}` &middot; {booking_status}.

                Two conventions differ from the price tab and are handled for you: response and
                acceptance rates are sent as **0-1**, and `room_type` uses **"Entire home/apt"**.
                `price` is an *input* to this model, not the answer.
                '''
            )

            with gr.Row():
                with gr.Column(scale=3):
                    build_booking_panel(b)

                with gr.Column(scale=2):
                    gr.Markdown("### Example hosts")
                    b_preset_buttons = {n: gr.Button(n, size="sm") for n in BOOKING_PRESETS}
                    b_predict = gr.Button("Predict instant booking", variant="primary", size="lg")
                    b_result = gr.HTML(
                        "<div class='result'><div class='eyebrow'>No request sent yet</div>"
                        "<div class='meta'>This endpoint scales to zero too, so it has its own "
                        "cold start independent of the price tab.</div></div>")
                    with gr.Accordion("What the endpoint derives", open=False):
                        b_derived = gr.Markdown()
                    with gr.Accordion("Request JSON (26 fields)", open=False):
                        b_request = gr.Code(language="json", label="POST body")
                    with gr.Accordion("Endpoint response", open=False):
                        b_response = gr.Code(language="json", label="Response")

            b_inputs = [b[k] for k in BOOKING_UI_FIELDS]
            b_predict.click(fn=predict_instant_booking, inputs=b_inputs,
                            outputs=[b_result, b_derived, b_request, b_response])

            def make_booking_loader(name):
                def _load():
                    p = BOOKING_PRESETS[name]
                    return [gr.update(value=p[k]) for k in BOOKING_UI_FIELDS]
                return _load

            for _bn, _bb in b_preset_buttons.items():
                _bb.click(fn=make_booking_loader(_bn), inputs=None, outputs=b_inputs)

        with gr.Tab("Baseline LogReg (classification)"):
            lb = {}
            gr.Markdown(
                f'''
                The **statistical floor** the instant-booking champion had to beat:
                `LogisticRegression(C=1.0, penalty="l2", solver="lbfgs")`, fitted in
                Notebook 02 on the 100-column matrix and 222,513 training rows.

                Same 26-field contract as the champion classifier tab, same derived
                features, a different estimator on the end and no endpoint. The 26 raw
                fields are put through **Notebook 01's own transform**, implemented in
                6.8b — the transform this estimator was actually fitted behind, including
                the StandardScaler moments without which a linear model saturates to 0%.

                Estimator loaded from `{LOGREG_SOURCE or "- unavailable -"}`.
                Transform state: `{NB01_STATE_SOURCE or "NOT FOUND - run 6.8b-export"}`.

                Both models were scored on the **same test split**, so the ROC-AUC gap of
                {RF_CHAMPION_SCORES["roc_auc"] - LOGREG_SCORES["roc_auc"]:.4f}
                ({LOGREG_SCORES["roc_auc"]:.4f} -> {RF_CHAMPION_SCORES["roc_auc"]:.4f})
                is a like-for-like number. Tick the comparison box to send the identical
                payload to `{BOOKING_ENDPOINT}` and see both answers on one card.
                '''
            )

            with gr.Row():
                with gr.Column(scale=3):
                    build_booking_panel(lb)

                with gr.Column(scale=2):
                    gr.Markdown("### Example hosts")
                    lb_preset_buttons = {n: gr.Button(n, size="sm")
                                         for n in BOOKING_PRESETS}
                    lb["compare_with_champion"] = gr.Checkbox(
                        label="Compare against the champion endpoint", value=True,
                        info=f"Sends the identical payload to {BOOKING_ENDPOINT} "
                             "as well, and shows both probabilities")
                    lb_predict = gr.Button("Score with the baseline",
                                           variant="primary", size="lg")
                    lb_result = gr.HTML(
                        "<div class='result'><div class='eyebrow'>No prediction "
                        "yet</div><div class='meta'>Fill in the listing, or load "
                        "an example. This tab does not call an endpoint, so there "
                        "is no cold start - but with the comparison box ticked it "
                        "waits on the champion endpoint, which does have one."
                        "</div></div>"
                        if LOGREG_TAB_READY else
                        "<div class='result'><div class='eyebrow'>Raw-listing mode "
                        "disabled</div><div class='meta'>The transform could not "
                        "be loaded and verified, so this mode will not quote a "
                        "probability. Press the button for the diagnosis, or use "
                        "re-run 6.8b after exporting Notebook 01's fitted state."
                        "</div></div>")
                    with gr.Accordion("What the transform derives", open=False):
                        lb_derived = gr.Markdown()
                    with gr.Accordion("Request JSON (26-field contract)",
                                      open=False):
                        lb_request = gr.Code(language="json", label="Payload")
                    with gr.Accordion("Prediction records", open=False):
                        lb_response = gr.Code(language="json", label="Records")

            lb_preset_inputs = [lb[k] for k in BOOKING_UI_FIELDS]
            lb_inputs = [lb[k] for k in LOGREG_UI_FIELDS]
            lb_outputs = [lb_result, lb_derived, lb_request, lb_response]

            lb_predict.click(fn=predict_instant_booking_baseline, inputs=lb_inputs,
                             outputs=lb_outputs)

            def make_lb_loader(name):
                def _load():
                    p = BOOKING_PRESETS[name]
                    return [gr.update(value=p[k]) for k in BOOKING_UI_FIELDS]
                return _load

            for _lbn, _lbb in lb_preset_buttons.items():
                # Presets fill the 26 contract controls only; the comparison
                # checkbox is the reader's setting, not the listing's.
                _lbb.click(fn=make_lb_loader(_lbn), inputs=None,
                           outputs=lb_preset_inputs)


        with gr.Tab("Recommender (ranking)"):
            r = {}
            gr.Markdown(
                f'''
                Ranks a whole city against a search, instead of scoring one listing you already
                have. Two stages: hard filters throw away everything that does not fit, then the
                survivors are ordered by a weighted blend of the **other two models' outputs**.

                `pred_price` and `instant_book_prob` come from a batch scoring job, not from a call
                per request — {RECSYS_STATUS_NOTE}. Turn on **Live re-scoring** to POST the
                shortlist to `{PRICE_ENDPOINT}` and `{BOOKING_ENDPOINT}` and rank on fresh numbers.
                '''
            )

            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Tab("Search"):
                        with gr.Row():
                            r["city"] = gr.Dropdown(
                                label="City", choices=RECSYS_CITIES,
                                value=("Paris" if "Paris" in RECSYS_CITIES
                                       else (RECSYS_CITIES[0] if RECSYS_CITIES else None)),
                                info="Filters the corpus and sets the currency")
                            r["min_guests"] = gr.Number(
                                label="Guests", value=2, precision=0, minimum=1,
                                info="A hard filter on accommodates, not a preference")
                            r["max_price"] = gr.Number(
                                label="Max nightly price", value=150.0, minimum=1,
                                info="In the city's own currency")
                        gr.Markdown(
                            "<small>The two weights below are the guest's preference, not a fitted "
                            "parameter. They are normalised by their sum, so only their ratio "
                            "matters.</small>")
                        r["w_value"] = gr.Slider(
                            label="Weight: value for money", minimum=0.0, maximum=1.0,
                            step=0.05, value=0.5,
                            info="Rewards listings the price model reads as underpriced")
                        r["w_convenience"] = gr.Slider(
                            label="Weight: instant-book convenience", minimum=0.0, maximum=1.0,
                            step=0.05, value=0.5,
                            info="Rewards listings the classifier reads as instantly bookable")
                        with gr.Row():
                            r["top_k"] = gr.Slider(
                                label="Shortlist length", minimum=1, maximum=25, step=1, value=10)
                            r["scope"] = gr.Radio(
                                label="Min-max the value score over",
                                choices=[SCOPE_CANDIDATES, SCOPE_GLOBAL],
                                value=SCOPE_CANDIDATES,
                                info="Global scaling mixes ten currencies in one range, which "
                                     "flattens w_value for every city but the largest")

                    with gr.Tab("Live re-scoring"):
                        gr.Markdown(
                            "Off by default, and that is the production-correct setting: re-scoring "
                            "on every request is what batch scoring exists to avoid. Turn it on to "
                            "prove the recommender really does sit on top of both endpoints — and "
                            "to watch what it costs in latency."
                        )
                        r["live"] = gr.Checkbox(
                            label="Re-score the shortlist through both endpoints",
                            value=False,
                            interactive=RECSYS_LIVE_OK,
                            info=("Filters first, then POSTs the survivors to the price and booking "
                                  "endpoints" if RECSYS_LIVE_OK else
                                  "Unavailable: the corpus is missing fields the endpoints need"))
                        r["live_n"] = gr.Slider(
                            label="Shortlist size to re-score", minimum=5, maximum=100,
                            step=5, value=25,
                            info=f"Sent in chunks of {RECSYS_CHUNK}. Both endpoints are serverless "
                                 f"at max_concurrency=5, so this is a capacity limit, not a "
                                 f"performance tweak")
                        gr.Markdown(
                            "<small>Expect a slow first call: two independent cold starts, one per "
                            "endpoint. The round-trip figure on the result card is the real cost of "
                            "doing per-request what a batch job already did offline.</small>")

                    with gr.Tab("FastAPI service"):
                        gr.Markdown(
                            "The same ranking is also exposed as a REST service by the FastAPI cell "
                            "in `Test API Gradio.ipynb`. Tick the box to have this tab act as a "
                            "client of that service rather than ranking in this process — the same "
                            "`POST /recommend` call `API Test Client.ipynb` makes.\n\n"
                            "The service must already be running, in its own kernel: its `uvicorn` "
                            "cell blocks forever, so it cannot share this notebook's."
                        )
                        r["use_api"] = gr.Checkbox(
                            label="Call the FastAPI service instead of ranking in-process",
                            value=False)
                        r["api_url"] = gr.Textbox(
                            label="Service URL", value="http://localhost:8000/recommend",
                            info="Reachable from this kernel. On Studio that means the same "
                                 "instance, not a browser tab")

                with gr.Column(scale=2):
                    gr.Markdown("### Example searches")
                    r_preset_buttons = {n: gr.Button(n, size="sm") for n in RECSYS_PRESETS}
                    r_predict = gr.Button("Get shortlist", variant="primary", size="lg")
                    r_result = gr.HTML(
                        "<div class='result'><div class='eyebrow'>No search run yet</div>"
                        "<div class='meta'>Filtering and ranking are in-memory and near-instant. "
                        "Only Live re-scoring touches an endpoint, and only then does a cold start "
                        "apply.</div></div>")
                    with gr.Accordion("How this shortlist was built", open=False):
                        r_derived = gr.Markdown()
                    with gr.Accordion("Request JSON (the /recommend body)", open=False):
                        r_request = gr.Code(language="json", label="POST body")
                    with gr.Accordion("Response", open=False):
                        r_response = gr.Code(language="json", label="Response")

            r_inputs = [r[k] for k in RECSYS_UI_FIELDS]
            r_predict.click(fn=recommend_listings, inputs=r_inputs,
                            outputs=[r_result, r_derived, r_request, r_response])

            def make_search_loader(name):
                def _load():
                    p = RECSYS_PRESETS[name]
                    return [gr.update(value=p[k]) for k in RECSYS_UI_FIELDS]
                return _load

            for _rn, _rb in r_preset_buttons.items():
                _rb.click(fn=make_search_loader(_rn), inputs=None, outputs=r_inputs)

    gr.Markdown(
        f'''
        ---
        **Price contract** &middot; `{len(SERVING_COLUMNS)}` raw fields in, one record out:
        `suggested_nightly_price`, `currency`, `city`, `log_price`. Nullable fields: `bedrooms`,
        `host_since`, `host_response_rate`, `host_acceptance_rate`, `host_response_time`,
        `neighbourhood` and the seven review scores - each has an explicit missing-value flag or
        imputation inside the pipeline. Everything else must carry a value.

        **Baseline contract** &middot; the same `{len(SERVING_COLUMNS)}` raw fields in, the same record
        out - but assembled locally. The 27 fields are transformed to the 88-column matrix by the
        champion's fitted preprocessor and handed to a `RidgeCV` loaded from MLflow, so the two
        regression tabs differ in exactly one component. No endpoint, no cold start, no serving cost.

        **Booking contract** &middot; `{len(BOOKING_FIELDS)}` fields in; `prediction`,
        `probability_class_1` out. Only `city`, `amenities`, `host_total_listings_count` and `price`
        are enforced - every other field you omit is silently filled with 0 rather than rejected.

        **Recommender contract** &middot; a search in - `city`, `min_guests`, `max_price`,
        `w_value`, `w_convenience` - and an ordered shortlist out. No endpoint of its own and no
        learned parameters: the ranking is a weighted blend of `pred_price` from the price model and
        `instant_book_prob` from the classifier, read from a batch-scored corpus. The weights are
        the guest's preference, expressed at request time, not something fitted.

        **Cost** &middot; two serverless endpoints, 3072 MB and 2048 MB, `max_concurrency=5` each. Roughly $0 when idle, billed per
        invocation. Cold starts show up in the round-trip figure above and in the `ModelSetupTime`
        CloudWatch metric. The recommender adds no serving cost at all while Live re-scoring is off,
        which is the argument for batch scoring in one line.
        '''
    )

# In SageMaker Studio, inline=True is usually the most convenient for notebook demos.

# if below fails to show:
# demo.launch(
#     inline=True,
#     share=False,
#     debug=True,
#     server_name="0.0.0.0",
#     server_port=7860
# )
# run this temporarily
demo.launch(
    inline=True,
    share=True,
    debug=True
)

/tmp/ipykernel_1164/2637138679.py:269: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title=f"{COURSE} {TEAM_ID} - model endpoints", css=CSS) as demo:


* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://7841ea8682cb353e06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.close()  # to close gradio
# gradio does not close, press stop in jupyter notebook

In [ ]:
# run this to confirm gradio closed
import os
import signal
import subprocess

PORT = 7860

cmd = f"lsof -ti:{PORT}"
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

pids = result.stdout.strip().splitlines()

if not pids:
    print(f"No process found on port {PORT}")
else:
    for pid in pids:
        print(f"Killing process {pid} on port {PORT}")
        os.kill(int(pid), signal.SIGTERM)

    print(f"Port {PORT} should now be free.")

## 8. Example Inputs

The three buttons in the UI load these. Between them they exercise every branch of the contract.

### Paris studio — a fully populated listing

| Feature | Value |
|---|---:|
| city | Paris |
| neighbourhood | Buttes-Montmartre |
| latitude / longitude | 48.8867 / 2.3431 |
| property_type | Private room in apartment |
| room_type | Private room |
| accommodates | 2 |
| bedrooms | 1 |
| minimum_nights / maximum_nights | 2 / 90 |
| instant_bookable | f |
| host_since | 2015-06-12 |
| host_is_superhost | f |
| host_total_listings_count | 1 |
| host_response_time | within a few hours |
| host_response_rate / host_acceptance_rate | 90 / 80 |
| review_scores_rating | 95 |

### Bangkok condo — entire place, high amenity count, superhost

| Feature | Value |
|---|---:|
| city | Bangkok |
| neighbourhood | Khlong Toei |
| latitude / longitude | 13.7230 / 100.5680 |
| property_type | Entire condominium |
| room_type | Entire place |
| accommodates | 4 |
| bedrooms | 2 |
| minimum_nights / maximum_nights | 1 / 365 |
| instant_bookable | t |
| host_since | 2017-03-08 |
| host_is_superhost | t |
| host_total_listings_count | 6 |
| host_response_time | within an hour |
| host_response_rate / host_acceptance_rate | 100 / 95 |
| review_scores_rating | 95 |

### New York, brand new listing — every missing-value flag fires

This is the interesting one. It sends `null` for `bedrooms`, `host_since`, both host rates and all
seven review scores, and sets `minimum_nights = 30`.

| Feature | Value | What it exercises |
|---|---:|---|
| bedrooms | `null` | `bedrooms_missing = 1`, filled as `ceil(accommodates / 2)` |
| host_since | `null` | `host_tenure_days` imputed to the fitted median |
| host_response_rate | `null` | `host_response_rate_missing = 1` |
| host_acceptance_rate | `null` | `host_acceptance_rate_missing = 1` |
| host_response_time | no history | the fifth one-hot level |
| review_scores_* | `null` × 7 | `has_review_scores = 0` |
| minimum_nights | 30 | `is_monthly_min = 1` |

A brand new listing with no reviews and no host history is not an edge case to be defended against —
it is the single most valuable request this model can serve, because a host with no reviews has no
comparable listings of their own to price against. The pipeline was built to answer it.

### Things worth trying

- Move the coordinates far from the city centre and watch `dist_center_km` change the answer.
- Type a neighbourhood that does not exist. The prediction still works; the derived panel flags the
  pair as unseen, and `neighbourhood_freq` falls to 0.0. That is the drift signal, visible.
- Set `host_since` to a date after **2021-03-01**. Tenure goes negative, because the training data is
  a snapshot as of that date. The endpoint will answer anyway — models do not refuse
  out-of-distribution input, which is exactly why monitoring exists.

## 9. Troubleshooting

| Problem | Possible Cause | What to Check |
|---|---|---|
| `ValidationException: Could not find endpoint` | Wrong endpoint name or region | Re-run Section 2 and copy the exact endpoint name. |
| `AccessDeniedException` | Wrong team role, or endpoint outside team permission | Check that you are using your own SageMaker Studio profile and your own team endpoint. |
| `ModelError` | Payload does not match `inference.py` | Open the "Request JSON" panel and compare it against the 27 `SERVING_COLUMNS` fields. |
| `ModelError` mentioning NaNs | A non-nullable field was sent as `null` | See the nullability table in Section 5. Only 12 of the 27 fields accept `null`. |
| First prediction takes 30–60 s | Serverless cold start | Expected. The endpoint scales to zero; watch `ModelSetupTime` in CloudWatch. |
| `ThrottlingException` | `max_concurrency = 5` reached | Wait a moment and send again. |
| Predictions look wrong for a city | Coordinates do not match the city | Re-pick the city from the dropdown; it snaps the coordinates to that city's centre. |
| Dropdowns say "built-in defaults" | `preprocessing_artifacts.json` not readable | Run this notebook from the same directory as Notebook 03, or check S3 permissions on the `/processed` prefix. |
| Gradio UI does not appear | Studio/browser rendering issue | Check the printed local URL, or restart the notebook kernel and rerun. |
| Port 7860 already in use | An earlier demo is still running | Run the port-cleanup cell in Section 7. |
| Endpoint charges continue | Endpoint still exists | Serverless is ~$0 when idle, but delete it when the project is finished. |
```

## 10. Optional Cleanup Reminder

Do **not** run cleanup if the endpoint is still needed for demonstration.

A serverless endpoint scales to zero and bills per invocation, so an idle endpoint costs
approximately nothing — there is no urgency here. Delete it when the project is finished and it is
no longer being demonstrated.

In [ ]:
# Optional cleanup example. Uncomment only when you really want to delete the endpoint.

# endpoint_desc = sm.describe_endpoint(EndpointName=PRICE_ENDPOINT)
# endpoint_config_name = endpoint_desc["EndpointConfigName"]

# sm.delete_endpoint(EndpointName=PRICE_ENDPOINT)
# print("Deleted endpoint:", PRICE_ENDPOINT)

# sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
# print("Deleted endpoint config:", endpoint_config_name)

print(f"Cleanup cells ready for: {PRICE_ENDPOINT}, {BOOKING_ENDPOINT}")
print("(delete calls commented out)")

## 11. Appendix — making the baseline durable

Section 6.7a falls back to MLflow because that is the only place the baseline exists. If the
MLflow App is ever deleted, or a reviewer without MLflow access needs to run this notebook, the
Baseline tab stops working.

`nb02_publish_ridge_baseline.py` (shipped alongside this notebook) fixes that: run it as a cell
in **Notebook 02**, after section 4 has produced `ridge` and `RIDGE_RUN_ID`. It mirrors the
estimator to `s3://<bucket>/…/artifacts/baseline/ridge_baseline.joblib` and writes a
`baseline_model.json` hand-off next to Notebook 02's `best_model.json`. Once it has run,
`_from_s3()` in 6.7a wins before `_from_mlflow()` is ever reached, and the tab no longer depends
on the tracking server being up.

## 12. Publish the LogReg baseline to S3 — from this notebook

Section 6.8a loads the estimator out of MLflow, which works but makes the tracking server
a single point of failure for a tab that otherwise has nothing to do with MLflow. Notebook
02 persists the baseline with one line and its only export step writes `best_model.json`,
which points at the tuned Random Forest instead — so MLflow is the only durable copy.

The cell below fixes that **from here**. 6.8a already has the estimator in memory; it is
the same object Notebook 02 fitted, from the same run, so there is nothing to be gained by
going back there. This writes it to
`s3://nyp-26s1-iti113/iti113/team14/data/airbnb-instant-booking/artifacts/baseline/logreg_baseline.joblib`
plus a self-describing `logreg_baseline.json` beside it, and drops local copies too.

It verifies the round trip rather than trusting the upload: the object is downloaded back
and checked for identical coefficients, identical column order, and identical predictions
on five real training rows. "The upload succeeded" and "this loads into an equivalent
estimator" are different claims, and only the second one matters when 6.8a reaches for the
file in three weeks.

The hand-off JSON carries the 100-column order, which is the part worth having even if you
never lose MLflow access — a linear model without its column order is 100 anonymous
floats. It also records `converged: false`, so nothing downstream has to rediscover that
lbfgs stopped at its iteration limit.

Run it, then re-run 6.8a: `_logreg_from_s3()` will win before MLflow is reached.

In [ ]:
import tempfile
from datetime import datetime, timezone
from pathlib import Path

# =====================================================================================
# 12. Publish the LogReg baseline to S3 -- FROM THIS NOTEBOOK
# -------------------------------------------------------------------------------------
# 6.8a already loaded the estimator out of MLflow. That object in memory is the one
# Notebook 02 fitted -- same run, same coefficients -- so there is no reason to go back to
# Notebook 02 to mirror it. This cell writes what 6.8a has to the team's own S3 prefix,
# and after it runs `_logreg_from_s3()` wins before `_logreg_from_mlflow()` is reached.
#
# WHY BOTHER. Notebook 02 persists the baseline with one line --
# mlflow.sklearn.log_model(model_lr, artifact_path="model") -- and its only export step
# writes best_model.json, which points at the tuned Random Forest instead. So the MLflow
# App is currently a single point of failure for a tab that has nothing to do with
# MLflow. Delete the App, or hand this notebook to a reviewer without access, and the
# baseline disappears.
#
# WHAT IT WRITES. The estimator, plus a hand-off JSON carrying the 100-column order. A
# linear model without its column order is not a model, it is 100 anonymous floats --
# publishing them together is what makes the artifact self-describing.
#
# Idempotent, and safe to re-run: if 6.8a already loaded from S3 it declines rather than
# rewriting the object it just read.
# =====================================================================================
PUBLISH_BASELINE = True   # set False to skip

if not PUBLISH_BASELINE:
    print("PUBLISH_BASELINE is False - nothing written.")
elif LOGREG_MODEL is None:
    print("Nothing to publish: 6.8a did not load an estimator.")
    print(f"  {LOGREG_ERROR}")
    for _n, _o in LOGREG_ATTEMPTS:
        print(f"    {_n:<24} {_o}")
elif (LOGREG_SOURCE or "").startswith(f"s3://{LOGREG_S3_BUCKET}"):
    print(f"Already published - 6.8a loaded it from {LOGREG_SOURCE}")
    print("  Delete that object first if you want to overwrite it.")
else:
    import sklearn

    _feature_columns = list(LOGREG_FITTED_COLUMNS or LOGREG_FEATURE_CONTRACT)
    _n_iter = int(np.asarray(getattr(LOGREG_MODEL, "n_iter_", [0])).ravel()[0])
    _converged = _n_iter < int(getattr(LOGREG_MODEL, "max_iter", 1000))

    _handoff = {
        "team_id": TEAM_ID,
        "student_id": LOGREG_STUDENT,
        "project_name": BOOKING_PROJECT,
        "role": "baseline",
        "target": "instant_bookable",
        "task": "binary_classification",
        "model_type": LOGREG_TYPE,
        "params": LOGREG_MODEL.get_params(),
        "n_features": len(_feature_columns),
        "feature_columns": _feature_columns,
        "n_iter": _n_iter,
        "converged": _converged,
        "mlflow_run_id": LOGREG_RUN_ID,
        "mlflow_run_name": LOGREG_RUN_NAME,
        "mlflow_tracking_uri": BOOKING_MLFLOW_TRACKING_URI,
        "mlflow_experiment": BOOKING_MLFLOW_EXPERIMENT,
        "mlflow_model_uri": f"runs:/{LOGREG_RUN_ID}/model",
        "loaded_from": LOGREG_SOURCE,
        "metrics_test": {_k: _v for _k, _v in LOGREG_SCORES.items()
                         if isinstance(_v, (int, float))},
        "s3_model_uri": f"s3://{LOGREG_S3_BUCKET}/{LOGREG_S3_KEY}",
        "sklearn_version": sklearn.__version__,
        "joblib_version": joblib.__version__,
        "published_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "published_by": "Notebook 04 section 12",
        "note": ("Baseline for comparison against the champion RandomForest. Fitted on "
                 "the pre-engineered matrix from Notebook 01; it expects the columns in "
                 "feature_columns, in that exact order, and does NO preprocessing."),
    }
    _handoff_key = LOGREG_S3_KEY.rsplit("/", 1)[0] + "/logreg_baseline.json"

    _tmp = Path(tempfile.mkdtemp())
    joblib.dump(LOGREG_MODEL, _tmp / "logreg_baseline.joblib")

    s3.upload_file(str(_tmp / "logreg_baseline.joblib"), LOGREG_S3_BUCKET, LOGREG_S3_KEY)
    s3.put_object(Bucket=LOGREG_S3_BUCKET, Key=_handoff_key,
                  Body=json.dumps(_handoff, indent=2).encode("utf-8"),
                  ContentType="application/json")

    # A local copy too, so a kernel without S3 read permission still has source 1.
    joblib.dump(LOGREG_MODEL, "logreg_baseline.joblib")
    Path("logreg_baseline.json").write_text(json.dumps(_handoff, indent=2),
                                            encoding="utf-8")

    # --- Verify the round trip rather than trusting the upload -------------------------
    # "The upload succeeded" and "the object loads back into an equivalent estimator" are
    # different claims. Check the second, because it is the one that matters when 6.8a
    # reaches for this file in three weeks.
    _rt_path = _tmp / "roundtrip.joblib"
    s3.download_file(LOGREG_S3_BUCKET, LOGREG_S3_KEY, str(_rt_path))
    _rt = joblib.load(_rt_path)

    _same_coef = bool(np.allclose(_rt.coef_, LOGREG_MODEL.coef_))
    _same_cols = list(getattr(_rt, "feature_names_in_", [])) == _feature_columns
    if TRAIN_SAMPLE is not None:
        _probe_rows = TRAIN_SAMPLE[_feature_columns].head(5)
        _same_pred = bool(np.allclose(_rt.predict_proba(_probe_rows)[:, 1],
                                      LOGREG_MODEL.predict_proba(_probe_rows)[:, 1]))
        _pred_note = "on 5 real training rows"
    else:
        _same_pred, _pred_note = None, "skipped - no training sample loaded"

    print("=" * 74)
    print("PUBLISHED THE LOGISTIC REGRESSION BASELINE")
    print("=" * 74)
    print(f"  model    : s3://{LOGREG_S3_BUCKET}/{LOGREG_S3_KEY}")
    print(f"  hand-off : s3://{LOGREG_S3_BUCKET}/{_handoff_key}")
    print(f"  local    : {os.path.abspath('logreg_baseline.joblib')}")
    print(f"  features : {len(_feature_columns)} columns, order captured in the hand-off")
    print(f"  sklearn  : {sklearn.__version__}")
    print(f"  source   : {LOGREG_SOURCE}")
    print()
    print("  Round trip (downloaded the object back and compared):")
    print(f"    coefficients identical : {_same_coef}")
    print(f"    column order preserved : {_same_cols}")
    print(f"    predictions identical  : {_same_pred}   ({_pred_note})")

    if not _converged:
        print()
        print(f"  NOTE: this estimator did not converge - lbfgs stopped at n_iter="
              f"{_n_iter}, the max_iter={getattr(LOGREG_MODEL, 'max_iter', '?')} limit.")
        print("  Recorded as converged=false in the hand-off so nothing downstream has")
        print("  to guess. It is still the estimator the reported metrics belong to.")

    if not (_same_coef and _same_cols and (_same_pred is not False)):
        raise RuntimeError(
            "The published artifact does not round-trip to an equivalent estimator. Do "
            "NOT rely on it - check the joblib/scikit-learn versions first.")

    print()
    print("  Re-run 6.8a: _logreg_from_s3() will now win before MLflow is reached.")
    print("=" * 74)
